In [9]:
!pip install xlsxwriter
!pip install rdkit
!pip install xgboost

In [ ]:
import pandas as pd
import re

# ======================================
# 1. Load the Excel file
# ======================================
df = pd.read_excel("final_moa_cateogrization_Peptides+Smiles.xlsx")

# ======================================
# 2. Remove duplicate rows
# ======================================
df = df.drop_duplicates()

# ======================================
# 3. Fill missing MoA with "Unknown"
# ======================================
df["MoA"] = df["MoA"].fillna("Unknown")

# ======================================
# 4. Drop MoA with "Unknown"
# ======================================
df = df[df["MoA"] != "Unknown"]

# ======================================
# 4. Clean MoA text
# ======================================
def clean_moa(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", re.sub(r"[^a-z0-9\s]", " ", text))
    return text.strip()

df["MoA_clean"] = df["MoA"].apply(clean_moa)

# ======================================
# 5. Categorization
# ======================================
def categorize_moa(text):

    text = text.lower()

    if text == "unknown":
        return "Multi-target / Polypharmacology"

    # 1 Membrane / Cell Envelope Disruption
    if any(k in text for k in [
        "membrane",
        "cell membrane",
        "pore",
        "permeability",
        "depolar",
        "lipid",
        "bilayer",
        "detergent",
        "ferriprotoporphyrin",
        "reactive nitro radical",
        "reactive intermediates",
        "membrane depolarization",
        "membrane permeabilization",
        "membrane damage",
        "membrane disrupting",
        "breaks down membrane lipids",
        "destroys cell membranes"
    ]):
        return "Membrane / Cell Envelope Disruption"

    # 2 Cell Wall Biosynthesis Inhibition
    if any(k in text for k in [
        "penicillin-binding protein",
        "pbp",
        "peptidoglycan",
        "cell wall",
        "lipid ii",
        "alanine racemase",
        "beta lactamase",
        "beta-lactamase",
        "penicillin binding protein",
        "fab i",
        "enoyl acyl carrier protein reductase"
    ]):
        return "Cell Wall Biosynthesis Inhibition"

    # 3 Protein Synthesis Inhibition
    if any(k in text for k in [
        "ribosome",
        "70s",
        "30s",
        "50s",
        "translation",
        "elongation factor",
        "trna",
        "protein synthesis",
        "trna synthetase",
        "trna ligase"
    ]):
        return "Protein Synthesis Inhibition"

    # 4 Nucleic Acid Synthesis / Integrity Disruption
    if any(k in text for k in [
        "dna",
        "rna",
        "gyrase",
        "topoisomerase",
        "polymerase",
        "replication",
        "intercalat",
        "dna binding",
        "dna disrupting",
        "dna inhibitor",
        "ribonucleotide reductase"
    ]):
        return "Nucleic Acid Synthesis / Integrity Disruption"

    # 5 Metabolic Pathway Inhibition
    if any(k in text for k in [
        "dihydrofolate reductase",
        "dihydropteroate synthase",
        "folate",
        "biosynthesis",
        "metabolism",
        "amidophosphoribosyltransferase",
        "oxidoreductase",
        "thymidylate synthase",
        "inosine monophosphate dehydrogenase",
        "dxr",
        "reductase",
        "synthase",
        "dehydrogenase",
        "cyclooxygenase",
        "carbonic anhydrase",
        "monoamine oxidase",
        "hmg coa reductase",
        "xanthine dehydrogenase",
        "caspase",
        "matrix metalloproteinase",
        "reductoisomerase",
        "dioxp reductoisomerase",
        "dxr",
        "dipeptidase",
        "diacylglycerol o acyltransferase",
        "dgat",
        "glucosyltransferase",
        "ceramide glucosyltransferase"
    ]):
        return "Metabolic Pathway Inhibition"

    # 6 Redox / Oxidative Stress Induction
    if any(k in text for k in [
        "oxidative",
        "redox",
        "ros",
        "reactive",
        "nitro",
        "oxidase",
        "reducing agent",
        "antioxidant",
        "increases ros",
        "chelating",
        "copper chelating",
        "iron chelating",
        "aluminium chelating",
        "glutathione",
        "glutathione precursor",
        "redox regulator"
    ]):
        return "Redox / Oxidative Stress Induction"

    # 7 Regulatory / Signalling Interference
    if any(k in text for k in [
        "quorum",
        "two component",
        "regulation",
        "signalling",
        "stress response",
        "transcription",
        "toll like receptor",
        "nf kb",
        "mtor",
        "map kinase",
        "cyclin dependent kinase",
        "biofilm related genes",
        "guanylate cyclase",
        "cyclase activator",
        "cyclase inhibitor",
        "cap binding complex",
        "signal transduction modulator",
        "arca",
        "arcb",
        "arcc",
        "arcd",
        "arc gene",
        "arginine deiminase pathway"
    ]):
        return "Regulatory / Signalling Interference"

    # 8 Biofilm Matrix Disruption
    if any(k in text for k in [
        "biofilm",
        "edna",
        "nuclease"
    ]):
        return "Biofilm Matrix Disruption"

    # 9 Antifungal Sterol Biosynthesis
    if any(k in text for k in [
        "cytochrome p450",
        "cyp51",
        "lanosterol",
        "ergosterol",
        "sterol biosynthesis",
        "squalene monooxygenase"
    ]):
        return "Antifungal Sterol Biosynthesis Inhibitors"

    # 10 Antiviral agents
    if any(k in text for k in [
        "hiv",
        "herpesvirus",
        "influenza",
        "hepatitis c",
        "reverse transcriptase",
        "viral protease",
        "viral capsid",
        "neuraminidase",
        "viral polymerase",
        "viral", "hepatitis", "capsid", "viral kinase", "helicase", "primase",
        "hiv protease",
        "hiv 1 protease",
        "viral protease",
        "pul97",
        "viral phosphotransferase",
        "human immunodeficiency virus type 1 protease"
    ]):
        return "Antiviral agents"

    # 11 Affecting Human Physiology
    if any(k in text for k in [
        "adrenergic receptor",
        "dopamine receptor",
        "serotonin transporter",
        "histamine receptor",
        "muscarinic receptor",
        "angiotensin receptor",
        "opioid receptor",
        "cannabinoid receptor",
        "vitamin d receptor",
        "androgen receptor",
        "estrogen receptor",
        "progesterone receptor",
        "glucocorticoid receptor",
        "receptor",
        "agonist",
        "antagonist",
        "partial agonist",
        "inverse agonist",
        "allosteric modulator",
        "allosteric antagonist",
        "positive modulator",
        "negative modulator",
        "retinoid receptor",
        "retinoic acid receptor",
        "retinoid x receptor",
        "adenosine receptor",
        "dopamine receptor",
        "serotonin receptor",
        "histamine receptor",
        "angiotensin receptor",
        "muscarinic receptor",
        "nicotinic receptor",
        "cannabinoid receptor",
        "opioid receptor",
        "purinergic receptor",
        "vanilloid receptor",
        "bile acid receptor",
        "fxr receptor",
        "peroxisome proliferator activated receptor",
        "ppar",
        "glucocorticoid receptor",
        "mineralocorticoid receptor",
        "estrogen receptor",
        "androgen receptor",
        "progesterone receptor",
        "vitamin d receptor",
        "chemokine receptor",
        "cxcr",
        "s1p receptor",
        "sphingosine receptor",
        "nmda receptor",
        "gaba receptor",
        "acetylcholine receptor",

        "ion channel",
        "channel blocker",
        "channel opener",
        "channel modulator",
        "voltage gated",
        "voltage activated",
        "calcium channel",
        "sodium channel",
        "potassium channel",
        "chloride channel",
        "anion channel",
        "herg",
        "enac",
        "ryanodine receptor",
        "l type calcium channel",
        "inwardly rectifying potassium channel",
        "calcium activated potassium channel",

        "acetylcholinesterase",
        "cholinesterase",
        "dopamine transporter",
        "serotonin transporter",
        "norepinephrine transporter",
        "gaba transporter",
        "synaptic vesicle transporter",
        "vesicular amine transporter",
        "sv2a",
        "neurotransmitter reuptake",
        "bcr abl",
        "abl kinase",
        "fusion protein inhibitor"

        "angiotensin-converting enzyme",
        "angiotensin converting enzyme",
        "ace inhibitor",
        "lipoxygenase",
        "cyclooxygenase",
        "phosphodiesterase",
        "pde inhibitor",
        "neprilysin",
        "lipase",
        "glucosidase",
        "decarboxylase",
        "monoamine oxidase",
        "thrombin",
        "farnesyltransferase",
        "kinase",
        "phosphatase",
        "ubiquitin ligase",
        "aurora kinase",
        "pdgfr",
        "pdk1",
        "ikk",

        "transporter",
        "solute carrier",
        "slc",
        "cotransporter",
        "sodium chloride cotransporter",
        "sodium potassium chloride cotransporter",
        "npc1l1",
        "amine transporter",
        "renal transporter",

        "insulin receptor",
        "hormone receptor",
        "glucose metabolism",
        "lipid metabolism",
        "fatty acid metabolism",
        "ppar alpha",
        "ppar gamma",
        "ppar delta",
        "metabolic regulator",

        "growth factor receptor",
        "egfr",
        "map kinase",
        "mtor",
        "nf kb",
        "signal transduction",
        "transcription factor",
        "cell signaling",
        "cytokine receptor",

        "tubulin",
        "microtubule",
        "cytoskeleton",
        "spindle assembly",
        "microtubule stabilizer",
        "microtubule inhibitor"
        "potassium transporting atpase",
        "npc1l1",
        "niemmann-pick c1",
        "niemann pick c1",
        "synaptic vesicle glycoprotein",
        "sv2a",
        "sv2a modulator",
        "laxative",
        "potassium-transporting atpase inhibitor",
        "potassium transporting atpase",
        "potassium atpase",
        "synaptic vesicle glycoprotein 2a",
        "cholesterol absorption inhibitor"



    ]):
        return "Affecting Human Physiology"

    # 12 Multi-target
    if any(k in text for k in [
        "multi target",
        "multiple",
        "polypharmacology",
        "pleiotropic"
    ]):
        return "Multi-target / Polypharmacology"

    return "Others"


# ======================================
# 6. Apply categorization
# ======================================
df["MoA_category"] = df["MoA_clean"].apply(categorize_moa)

# ======================================
# 7. Keep only required columns
# ======================================
df_final = df[["Name", "SMILES", "MoA_category"]]

# ======================================
# 8. Rename column
# ======================================
df_final = df_final.rename(columns={"MoA_category": "MoA"})

# ======================================
# 9. Check distribution
# ======================================
print(df["MoA_category"].value_counts())

# ======================================
# 10. Save final file
# ======================================
df_final.to_excel("final_moa_categorized.xlsx", index=False)

print(df[df["MoA_category"]=="Others"]["MoA"].unique())

# ======================================
# 10. Save categorized data into multiple sheets
# ======================================

with pd.ExcelWriter("final_moa_categorized.xlsx", engine="xlsxwriter") as writer:

    # Main categorized dataset
    df_final.to_excel(writer, sheet_name="All_Data_PA", index=False)

    # Human physiology targets
    human_targets = df[df["MoA_category"] == "Affecting Human Physiology"]
    human_targets.to_excel(writer, sheet_name="Human_Physiology_Targets", index=False)

    # Antiviral agents
    antiviral_targets = df[df["MoA_category"] == "Antiviral agents"]
    antiviral_targets.to_excel(writer, sheet_name="Antiviral_Targets", index=False)

    # Antifungal agents
    antifungal_targets = df[df["MoA_category"] == "Antifungal Sterol Biosynthesis Inhibitors"]
    antifungal_targets.to_excel(writer, sheet_name="Antifungal_Targets", index=False)

    # Others
    Others = df[df["MoA_category"] == "Others"]
    Others.to_excel(writer, sheet_name="Others", index=False)

    # ======================================
    # Direct Antibacterial (remove unwanted categories)
    # ======================================
    direct_antibacterial = df[~df["MoA_category"].isin([
        "Affecting Human Physiology",
        "Antiviral agents",
        "Antifungal Sterol Biosynthesis Inhibitors",
        "Others"
    ])]

    direct_antibacterial.to_excel(writer, sheet_name="Direct_Antibacterial", index=False)

print("Excel file created with multiple sheets including Direct_Antibacterial.")

import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

# Load dataset from the correct file and sheet
df = pd.read_excel("final_moa_categorized.xlsx", sheet_name="Direct_Antibacterial")

# Get list of all RDKit descriptors
descriptor_names = [desc[0] for desc in Descriptors.descList]

def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return [None]*len(descriptor_names)

    descriptor_values = []

    for name, function in Descriptors.descList:
        try:
            value = function(mol)
        except:
            value = None
        descriptor_values.append(value)

    return descriptor_values


# Apply descriptor calculation
descriptor_data = df["SMILES"].apply(calculate_descriptors)

# Convert to dataframe
X = pd.DataFrame(descriptor_data.tolist(), columns=descriptor_names)

# Target variable
y = df["MoA_category"]

# Combine if needed
final_df = pd.concat([X, y], axis=1)

# Save descriptor dataset
final_df.to_excel("MoA_rdkit_all_descriptors_calculated.xlsx", index=False)

print("Total descriptors extracted:", len(descriptor_names))
print("Dataset shape:", final_df.shape)


# Load Descriptor data

import pandas as pd

df = pd.read_excel("MoA_rdkit_all_descriptors_calculated.xlsx")

print("Dataset shape:", df.shape)

df.head()

# seperate descriptors and label(moa)

X = df.drop(columns=["MoA_category"])
y = df["MoA_category"]

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

# -------------------------------
#  FILTER RARE CLASSES HERE
# -------------------------------

df_model = X.copy()
df_model["MoA_category"] = y

counts = df_model["MoA_category"].value_counts()
valid_classes = counts[counts >= 3].index

df_filtered = df_model[df_model["MoA_category"].isin(valid_classes)].copy()

# Update X and y AFTER filtering
X = df_filtered.drop(columns=["MoA_category"])
y = df_filtered["MoA_category"]

print("After filtering:", X.shape, y.shape)

import numpy as np

# Replace infinity values
X = X.replace([np.inf, -np.inf], np.nan)

# Convert to float32 to catch values too large for this dtype and re-handle potential new NaNs
X = X.astype(np.float32)

# Replace any new inf values that might have been created due to float32 conversion overflow
X = X.replace([np.inf, -np.inf], np.nan)

# Fill missing values

X = X.fillna(X.median())

# Remove Extremely Large Descriptors
# Some descriptors can reach 10¹² or more, causing overflow.

X.describe().T.sort_values("max", ascending=False).head(10)

# A safe solution is to clip values.

X = X.clip(-1e6, 1e6)

# A safe solution is to clip values.

X = X.astype(np.float64)

# check missing, infinity and large values

import numpy as np

print("NaN values:", np.isnan(X).sum().sum())
print("Infinity values:", np.isinf(X).sum().sum())
print("Max value in dataset:", np.nanmax(X.values))

# Train-Test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# label encoding

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_encoded = le.fit_transform(y_train)

y_test_encoded = le.transform(y_test)

# Feature stability

from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0)

# Fit ONLY on training data
X_train_var = selector.fit_transform(X_train)

# Apply SAME transformation on test data
X_test_var = selector.transform(X_test)

# Convert to DataFrame (optional but good)
selected_columns = X_train.columns[selector.get_support()]

X_train_var = pd.DataFrame(X_train_var, columns=selected_columns)
X_test_var = pd.DataFrame(X_test_var, columns=selected_columns)

print("After removing constant features:", X_train_var.shape[1])

import numpy as np

# Step 1: Compute correlation ONLY on training data
corr_matrix = X_train_var.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]

print("Descriptors to drop:", len(to_drop))

# Step 2: Drop from TRAIN
X_train_corr = X_train_var.drop(columns=to_drop)

# Step 3: Drop SAME columns from TEST
X_test_corr = X_test_var.drop(columns=to_drop)

print("After correlation removal (train):", X_train_corr.shape)
print("After correlation removal (test):", X_test_corr.shape)

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=400,
    random_state=42
)

# ✅ Train ONLY on training data
rf.fit(X_train_corr, y_train)

# Get importance
importance = pd.Series(
    rf.feature_importances_,
    index=X_train_corr.columns
)

# Sort
importance = importance.sort_values(ascending=False)

print("Top features:\n", importance.head(30))

# Select top 30 features
top_features = importance.head(30).index

# Apply SAME features to train and test
X_train_sel = X_train_corr[top_features]
X_test_sel = X_test_corr[top_features]

print("Final train shape:", X_train_sel.shape)
print("Final test shape:", X_test_sel.shape)

# Normalization

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit ONLY on training
X_train_scaled = scaler.fit_transform(X_train_sel)

# Apply to test
X_test_scaled = scaler.transform(X_test_sel)

# Find best K

from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt

bic_scores = []

k_values = [5, 8, 10, 12, 15, 18, 20, 22]

for k in k_values:

    gmm = GaussianMixture(
        n_components=k,
        covariance_type='full',
        random_state=42
    )

    gmm.fit(X_train_scaled)

    bic = gmm.bic(X_train_scaled)

    bic_scores.append(bic)

    print(f"K = {k}, BIC = {bic}")

# Silhouette Score for best K

from sklearn.metrics import silhouette_score

for k in k_values:

    gmm = GaussianMixture(n_components=k, random_state=42)

    labels = gmm.fit_predict(X_train_scaled)

    score = silhouette_score(X_train_scaled, labels)

    print(f"K = {k}, Silhouette Score = {score}")

# Apply GMM

from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=10, random_state=42)

gmm.fit(X_train_scaled)

train_gmm = gmm.predict_proba(X_train_scaled)
test_gmm = gmm.predict_proba(X_test_scaled)

# Combine
X_train_final = np.hstack([X_train_scaled, train_gmm])
X_test_final = np.hstack([X_test_scaled, test_gmm])

# GMM probabilities for TRAIN
gmm_train_probs = gmm.predict_proba(X_train_scaled)

gmm_train_df = pd.DataFrame(
    gmm_train_probs,
    columns=[f"GMM_Cluster_{i}" for i in range(gmm_train_probs.shape[1])]
)

# Convert scaled train to DataFrame
X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=X_train_sel.columns
)

# Concatenate
X_train_enhanced = pd.concat([X_train_scaled_df, gmm_train_df], axis=1)

print("Train shape:", X_train_enhanced.shape)
print(X_train_enhanced.columns[-10:])

# GMM probabilities for TEST
gmm_test_probs = gmm.predict_proba(X_test_scaled)

gmm_test_df = pd.DataFrame(
    gmm_test_probs,
    columns=[f"GMM_Cluster_{i}" for i in range(gmm_test_probs.shape[1])]
)

# Convert scaled test to DataFrame
X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=X_train_sel.columns   # IMPORTANT: use train columns
)

# Concatenate
X_test_enhanced = pd.concat([X_test_scaled_df, gmm_test_df], axis=1)

print("Test shape:", X_test_enhanced.shape)
print(X_test_enhanced.columns[-10:])

cluster_labels = gmm.predict(X_train_scaled)

df_analysis = pd.DataFrame({
    "Cluster": cluster_labels,
    "MoA": y_train.values
})

print(df_analysis.groupby("Cluster")["MoA"].value_counts())

# ---Function for evaluating metrices---

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    classification_report
)

def evaluate_model_full(model_name, y_test, y_pred, y_prob):

    print("\n==============================")
    print("MODEL:", model_name)
    print("==============================")

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    print("\nOverall Confusion Matrix:\n")
    print(cm)

    # Plot confusion matrix
    plt.figure(figsize=(8,6))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=le.classes_,
        yticklabels=le.classes_
    )

    plt.xlabel("Predicted MoA")
    plt.ylabel("Actual MoA")
    plt.title(model_name + " Confusion Matrix")

    plt.show()

    # ------------------------------
    # TP FP FN TN PER CLASS - CALCULATE HERE
    # ------------------------------

    TP = np.diag(cm)

    FP = np.sum(cm, axis=0) - TP

    FN = np.sum(cm, axis=1) - TP

    TN = np.sum(cm) - (TP + FP + FN)

    # --------------------------------
    # PER-MoA CONFUSION MATRICES
    # --------------------------------

    per_moa_cm_list = []

    for i, moa in enumerate(le.classes_):

        tp = TP[i]
        fp = FP[i]
        fn = FN[i]
        tn = TN[i]

        cm_moa = np.array([
            [tp, fn],
            [fp, tn]
        ])

        print("\nConfusion Matrix for MoA:", moa)
        print(cm_moa)

        plt.figure(figsize=(4,4))

        sns.heatmap(
            cm_moa,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=["Predicted "+moa, "Predicted Other"],
            yticklabels=["Actual "+moa, "Actual Other"]
        )

        plt.title(model_name + " - " + moa)

        plt.show()

        per_moa_cm_list.append({
            "MoA": moa,
            "TP": tp,
            "FN": fn,
            "FP": fp,
            "TN": tn
        })

    per_moa_cm_df = pd.DataFrame(per_moa_cm_list)

    per_moa_cm_df.to_excel(model_name+"_per_moa_confusion_matrix.xlsx", index=False)

    # Sensitivity (Recall)
    sensitivity = TP / (TP + FN)

    # Specificity
    specificity = TN / (TN + FP)

    per_class_df = pd.DataFrame({
        "MoA": le.classes_,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "Sensitivity": sensitivity,
        "Specificity": specificity
    })

    print("\nPer-MoA Metrics\n")
    print(per_class_df)

    # Save results
    per_class_df.to_excel(model_name+"_per_MoA_metrics.xlsx", index=False)

    # ------------------------------
    # OVERALL METRICS
    # ------------------------------

    overall_TP = TP.sum()

    overall_FP = FP.sum()

    overall_FN = FN.sum()

    overall_TN = TN.sum()

    overall_sensitivity = overall_TP / (overall_TP + overall_FN)

    overall_specificity = overall_TN / (overall_TN + overall_FP)

    overall_df = pd.DataFrame({
        "Metric": [
            "TP",
            "FP",
            "FN",
            "TN",
            "Sensitivity",
            "Specificity"
        ],
        "Value": [
            overall_TP,
            overall_FP,
            overall_FN,
            overall_TN,
            overall_sensitivity,
            overall_specificity
        ]
    })

    print("\nOverall Metrics\n")
    print(overall_df)

    overall_df.to_excel(model_name+"_overall_metrics.xlsx", index=False)

    # ------------------------------
    # Accuracy
    # ------------------------------

    acc = accuracy_score(y_test, y_pred)

    print("\nAccuracy:", acc)

    # ------------------------------
    # MCC
    # ------------------------------

    mcc = matthews_corrcoef(y_test, y_pred)

    print("MCC:", mcc)

    # ------------------------------
    # ROC AUC (Multiclass)
    # ------------------------------

    from sklearn.preprocessing import label_binarize

    y_test_bin = label_binarize(y_test, classes=np.unique(y_test))

    auc = roc_auc_score(y_test_bin, y_prob, multi_class="ovr")

    print("ROC-AUC:", auc)

    # Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features='sqrt'
)

# ✅ Train on FINAL enhanced features (IMPORTANT)
rf_model.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_rf = rf_model.predict(X_test_enhanced)

# Accuracy
print("Random Forest Accuracy:",
      accuracy_score(y_test_encoded, y_pred_rf))

# Classification Report
print(classification_report(y_test_encoded, y_pred_rf))

# Probabilities (for ROC-AUC)
y_prob_rf = rf_model.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "Random Forest",
    y_test_encoded,
    y_pred_rf,
    y_prob_rf
)

# Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features='sqrt'
)

# ✅ Train on FINAL enhanced features (IMPORTANT)
rf_model.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_rf = rf_model.predict(X_test_enhanced)

# Accuracy
print("Random Forest Accuracy:",
      accuracy_score(y_test_encoded, y_pred_rf))

# Classification Report
print(classification_report(y_test_encoded, y_pred_rf))

# Probabilities (for ROC-AUC)
y_prob_rf = rf_model.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "Random Forest",
    y_test_encoded,
    y_pred_rf,
    y_prob_rf
)


# XGBoost

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    eval_metric="mlogloss"
)

# ✅ Train on FINAL enhanced features
xgb.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_xgb = xgb.predict(X_test_enhanced)

# Accuracy
print("XGBoost Accuracy:",
      accuracy_score(y_test_encoded, y_pred_xgb))

# Classification Report
print(classification_report(y_test_encoded, y_pred_xgb))

# Probabilities (for ROC-AUC)
y_prob_xgb = xgb.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "XGBoost",
    y_test_encoded,
    y_pred_xgb,
    y_prob_xgb
)




MoA_category
Affecting Human Physiology                       308
Metabolic Pathway Inhibition                      69
Nucleic Acid Synthesis / Integrity Disruption     69
Cell Wall Biosynthesis Inhibition                 48
Protein Synthesis Inhibition                      33
Antifungal Sterol Biosynthesis Inhibitors         21
Membrane / Cell Envelope Disruption               18
Antiviral agents                                  18
Redox / Oxidative Stress Induction                15
Regulatory / Signalling Interference              13
Biofilm Matrix Disruption                          1
Name: count, dtype: int64
[]
Excel file created with multiple sheets including Direct_Antibacterial.


k = 10 f = 50

In [ ]:
import pandas as pd
import re

# ======================================
# 1. Load the Excel file
# ======================================
df = pd.read_excel("final_moa_cateogrization_Peptides+Smiles.xlsx")

# ======================================
# 2. Remove duplicate rows
# ======================================
df = df.drop_duplicates()

# ======================================
# 3. Fill missing MoA with "Unknown"
# ======================================
df["MoA"] = df["MoA"].fillna("Unknown")

# ======================================
# 4. Drop MoA with "Unknown"
# ======================================
df = df[df["MoA"] != "Unknown"]

# ======================================
# 4. Clean MoA text
# ======================================
def clean_moa(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", re.sub(r"[^a-z0-9\s]", " ", text))
    return text.strip()

df["MoA_clean"] = df["MoA"].apply(clean_moa)

# ======================================
# 5. Categorization
# ======================================
def categorize_moa(text):

    text = text.lower()

    if text == "unknown":
        return "Multi-target / Polypharmacology"

    # 1 Membrane / Cell Envelope Disruption
    if any(k in text for k in [
        "membrane",
        "cell membrane",
        "pore",
        "permeability",
        "depolar",
        "lipid",
        "bilayer",
        "detergent",
        "ferriprotoporphyrin",
        "reactive nitro radical",
        "reactive intermediates",
        "membrane depolarization",
        "membrane permeabilization",
        "membrane damage",
        "membrane disrupting",
        "breaks down membrane lipids",
        "destroys cell membranes"
    ]):
        return "Membrane / Cell Envelope Disruption"

    # 2 Cell Wall Biosynthesis Inhibition
    if any(k in text for k in [
        "penicillin-binding protein",
        "pbp",
        "peptidoglycan",
        "cell wall",
        "lipid ii",
        "alanine racemase",
        "beta lactamase",
        "beta-lactamase",
        "penicillin binding protein",
        "fab i",
        "enoyl acyl carrier protein reductase"
    ]):
        return "Cell Wall Biosynthesis Inhibition"

    # 3 Protein Synthesis Inhibition
    if any(k in text for k in [
        "ribosome",
        "70s",
        "30s",
        "50s",
        "translation",
        "elongation factor",
        "trna",
        "protein synthesis",
        "trna synthetase",
        "trna ligase"
    ]):
        return "Protein Synthesis Inhibition"

    # 4 Nucleic Acid Synthesis / Integrity Disruption
    if any(k in text for k in [
        "dna",
        "rna",
        "gyrase",
        "topoisomerase",
        "polymerase",
        "replication",
        "intercalat",
        "dna binding",
        "dna disrupting",
        "dna inhibitor",
        "ribonucleotide reductase"
    ]):
        return "Nucleic Acid Synthesis / Integrity Disruption"

    # 5 Metabolic Pathway Inhibition
    if any(k in text for k in [
        "dihydrofolate reductase",
        "dihydropteroate synthase",
        "folate",
        "biosynthesis",
        "metabolism",
        "amidophosphoribosyltransferase",
        "oxidoreductase",
        "thymidylate synthase",
        "inosine monophosphate dehydrogenase",
        "dxr",
        "reductase",
        "synthase",
        "dehydrogenase",
        "cyclooxygenase",
        "carbonic anhydrase",
        "monoamine oxidase",
        "hmg coa reductase",
        "xanthine dehydrogenase",
        "caspase",
        "matrix metalloproteinase",
        "reductoisomerase",
        "dioxp reductoisomerase",
        "dxr",
        "dipeptidase",
        "diacylglycerol o acyltransferase",
        "dgat",
        "glucosyltransferase",
        "ceramide glucosyltransferase"
    ]):
        return "Metabolic Pathway Inhibition"

    # 6 Redox / Oxidative Stress Induction
    if any(k in text for k in [
        "oxidative",
        "redox",
        "ros",
        "reactive",
        "nitro",
        "oxidase",
        "reducing agent",
        "antioxidant",
        "increases ros",
        "chelating",
        "copper chelating",
        "iron chelating",
        "aluminium chelating",
        "glutathione",
        "glutathione precursor",
        "redox regulator"
    ]):
        return "Redox / Oxidative Stress Induction"

    # 7 Regulatory / Signalling Interference
    if any(k in text for k in [
        "quorum",
        "two component",
        "regulation",
        "signalling",
        "stress response",
        "transcription",
        "toll like receptor",
        "nf kb",
        "mtor",
        "map kinase",
        "cyclin dependent kinase",
        "biofilm related genes",
        "guanylate cyclase",
        "cyclase activator",
        "cyclase inhibitor",
        "cap binding complex",
        "signal transduction modulator",
        "arca",
        "arcb",
        "arcc",
        "arcd",
        "arc gene",
        "arginine deiminase pathway"
    ]):
        return "Regulatory / Signalling Interference"

    # 8 Biofilm Matrix Disruption
    if any(k in text for k in [
        "biofilm",
        "edna",
        "nuclease"
    ]):
        return "Biofilm Matrix Disruption"

    # 9 Antifungal Sterol Biosynthesis
    if any(k in text for k in [
        "cytochrome p450",
        "cyp51",
        "lanosterol",
        "ergosterol",
        "sterol biosynthesis",
        "squalene monooxygenase"
    ]):
        return "Antifungal Sterol Biosynthesis Inhibitors"

    # 10 Antiviral agents
    if any(k in text for k in [
        "hiv",
        "herpesvirus",
        "influenza",
        "hepatitis c",
        "reverse transcriptase",
        "viral protease",
        "viral capsid",
        "neuraminidase",
        "viral polymerase",
        "viral", "hepatitis", "capsid", "viral kinase", "helicase", "primase",
        "hiv protease",
        "hiv 1 protease",
        "viral protease",
        "pul97",
        "viral phosphotransferase",
        "human immunodeficiency virus type 1 protease"
    ]):
        return "Antiviral agents"

    # 11 Affecting Human Physiology
    if any(k in text for k in [
        "adrenergic receptor",
        "dopamine receptor",
        "serotonin transporter",
        "histamine receptor",
        "muscarinic receptor",
        "angiotensin receptor",
        "opioid receptor",
        "cannabinoid receptor",
        "vitamin d receptor",
        "androgen receptor",
        "estrogen receptor",
        "progesterone receptor",
        "glucocorticoid receptor",
        "receptor",
        "agonist",
        "antagonist",
        "partial agonist",
        "inverse agonist",
        "allosteric modulator",
        "allosteric antagonist",
        "positive modulator",
        "negative modulator",
        "retinoid receptor",
        "retinoic acid receptor",
        "retinoid x receptor",
        "adenosine receptor",
        "dopamine receptor",
        "serotonin receptor",
        "histamine receptor",
        "angiotensin receptor",
        "muscarinic receptor",
        "nicotinic receptor",
        "cannabinoid receptor",
        "opioid receptor",
        "purinergic receptor",
        "vanilloid receptor",
        "bile acid receptor",
        "fxr receptor",
        "peroxisome proliferator activated receptor",
        "ppar",
        "glucocorticoid receptor",
        "mineralocorticoid receptor",
        "estrogen receptor",
        "androgen receptor",
        "progesterone receptor",
        "vitamin d receptor",
        "chemokine receptor",
        "cxcr",
        "s1p receptor",
        "sphingosine receptor",
        "nmda receptor",
        "gaba receptor",
        "acetylcholine receptor",

        "ion channel",
        "channel blocker",
        "channel opener",
        "channel modulator",
        "voltage gated",
        "voltage activated",
        "calcium channel",
        "sodium channel",
        "potassium channel",
        "chloride channel",
        "anion channel",
        "herg",
        "enac",
        "ryanodine receptor",
        "l type calcium channel",
        "inwardly rectifying potassium channel",
        "calcium activated potassium channel",

        "acetylcholinesterase",
        "cholinesterase",
        "dopamine transporter",
        "serotonin transporter",
        "norepinephrine transporter",
        "gaba transporter",
        "synaptic vesicle transporter",
        "vesicular amine transporter",
        "sv2a",
        "neurotransmitter reuptake",
        "bcr abl",
        "abl kinase",
        "fusion protein inhibitor"

        "angiotensin-converting enzyme",
        "angiotensin converting enzyme",
        "ace inhibitor",
        "lipoxygenase",
        "cyclooxygenase",
        "phosphodiesterase",
        "pde inhibitor",
        "neprilysin",
        "lipase",
        "glucosidase",
        "decarboxylase",
        "monoamine oxidase",
        "thrombin",
        "farnesyltransferase",
        "kinase",
        "phosphatase",
        "ubiquitin ligase",
        "aurora kinase",
        "pdgfr",
        "pdk1",
        "ikk",

        "transporter",
        "solute carrier",
        "slc",
        "cotransporter",
        "sodium chloride cotransporter",
        "sodium potassium chloride cotransporter",
        "npc1l1",
        "amine transporter",
        "renal transporter",

        "insulin receptor",
        "hormone receptor",
        "glucose metabolism",
        "lipid metabolism",
        "fatty acid metabolism",
        "ppar alpha",
        "ppar gamma",
        "ppar delta",
        "metabolic regulator",

        "growth factor receptor",
        "egfr",
        "map kinase",
        "mtor",
        "nf kb",
        "signal transduction",
        "transcription factor",
        "cell signaling",
        "cytokine receptor",

        "tubulin",
        "microtubule",
        "cytoskeleton",
        "spindle assembly",
        "microtubule stabilizer",
        "microtubule inhibitor"
        "potassium transporting atpase",
        "npc1l1",
        "niemmann-pick c1",
        "niemann pick c1",
        "synaptic vesicle glycoprotein",
        "sv2a",
        "sv2a modulator",
        "laxative",
        "potassium-transporting atpase inhibitor",
        "potassium transporting atpase",
        "potassium atpase",
        "synaptic vesicle glycoprotein 2a",
        "cholesterol absorption inhibitor"



    ]):
        return "Affecting Human Physiology"

    # 12 Multi-target
    if any(k in text for k in [
        "multi target",
        "multiple",
        "polypharmacology",
        "pleiotropic"
    ]):
        return "Multi-target / Polypharmacology"

    return "Others"


# ======================================
# 6. Apply categorization
# ======================================
df["MoA_category"] = df["MoA_clean"].apply(categorize_moa)

# ======================================
# 7. Keep only required columns
# ======================================
df_final = df[["Name", "SMILES", "MoA_category"]]

# ======================================
# 8. Rename column
# ======================================
df_final = df_final.rename(columns={"MoA_category": "MoA"})

# ======================================
# 9. Check distribution
# ======================================
print(df["MoA_category"].value_counts())

# ======================================
# 10. Save final file
# ======================================
df_final.to_excel("final_moa_categorized.xlsx", index=False)

print(df[df["MoA_category"]=="Others"]["MoA"].unique())

# ======================================
# 10. Save categorized data into multiple sheets
# ======================================

with pd.ExcelWriter("final_moa_categorized.xlsx", engine="xlsxwriter") as writer:

    # Main categorized dataset
    df_final.to_excel(writer, sheet_name="All_Data_PA", index=False)

    # Human physiology targets
    human_targets = df[df["MoA_category"] == "Affecting Human Physiology"]
    human_targets.to_excel(writer, sheet_name="Human_Physiology_Targets", index=False)

    # Antiviral agents
    antiviral_targets = df[df["MoA_category"] == "Antiviral agents"]
    antiviral_targets.to_excel(writer, sheet_name="Antiviral_Targets", index=False)

    # Antifungal agents
    antifungal_targets = df[df["MoA_category"] == "Antifungal Sterol Biosynthesis Inhibitors"]
    antifungal_targets.to_excel(writer, sheet_name="Antifungal_Targets", index=False)

    # Others
    Others = df[df["MoA_category"] == "Others"]
    Others.to_excel(writer, sheet_name="Others", index=False)

    # ======================================
    # Direct Antibacterial (remove unwanted categories)
    # ======================================
    direct_antibacterial = df[~df["MoA_category"].isin([
        "Affecting Human Physiology",
        "Antiviral agents",
        "Antifungal Sterol Biosynthesis Inhibitors",
        "Others"
    ])]

    direct_antibacterial.to_excel(writer, sheet_name="Direct_Antibacterial", index=False)

print("Excel file created with multiple sheets including Direct_Antibacterial.")

import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

# Load dataset from the correct file and sheet
df = pd.read_excel("final_moa_categorized.xlsx", sheet_name="Direct_Antibacterial")

# Get list of all RDKit descriptors
descriptor_names = [desc[0] for desc in Descriptors.descList]

def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return [None]*len(descriptor_names)

    descriptor_values = []

    for name, function in Descriptors.descList:
        try:
            value = function(mol)
        except:
            value = None
        descriptor_values.append(value)

    return descriptor_values


# Apply descriptor calculation
descriptor_data = df["SMILES"].apply(calculate_descriptors)

# Convert to dataframe
X = pd.DataFrame(descriptor_data.tolist(), columns=descriptor_names)

# Target variable
y = df["MoA_category"]

# Combine if needed
final_df = pd.concat([X, y], axis=1)

# Save descriptor dataset
final_df.to_excel("MoA_rdkit_all_descriptors_calculated.xlsx", index=False)

print("Total descriptors extracted:", len(descriptor_names))
print("Dataset shape:", final_df.shape)


# Load Descriptor data

import pandas as pd

df = pd.read_excel("MoA_rdkit_all_descriptors_calculated.xlsx")

print("Dataset shape:", df.shape)

df.head()

# seperate descriptors and label(moa)

X = df.drop(columns=["MoA_category"])
y = df["MoA_category"]

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

# -------------------------------
#  FILTER RARE CLASSES HERE
# -------------------------------

df_model = X.copy()
df_model["MoA_category"] = y

counts = df_model["MoA_category"].value_counts()
valid_classes = counts[counts >= 3].index

df_filtered = df_model[df_model["MoA_category"].isin(valid_classes)].copy()

# Update X and y AFTER filtering
X = df_filtered.drop(columns=["MoA_category"])
y = df_filtered["MoA_category"]

print("After filtering:", X.shape, y.shape)

import numpy as np

# Replace infinity values
X = X.replace([np.inf, -np.inf], np.nan)

# Convert to float32 to catch values too large for this dtype and re-handle potential new NaNs
X = X.astype(np.float32)

# Replace any new inf values that might have been created due to float32 conversion overflow
X = X.replace([np.inf, -np.inf], np.nan)

# Fill missing values

X = X.fillna(X.median())

# Remove Extremely Large Descriptors
# Some descriptors can reach 10¹² or more, causing overflow.

X.describe().T.sort_values("max", ascending=False).head(10)

# A safe solution is to clip values.

X = X.clip(-1e6, 1e6)

# A safe solution is to clip values.

X = X.astype(np.float64)

# check missing, infinity and large values

import numpy as np

print("NaN values:", np.isnan(X).sum().sum())
print("Infinity values:", np.isinf(X).sum().sum())
print("Max value in dataset:", np.nanmax(X.values))

# Train-Test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# label encoding

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_encoded = le.fit_transform(y_train)

y_test_encoded = le.transform(y_test)

# Feature stability

from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0)

# Fit ONLY on training data
X_train_var = selector.fit_transform(X_train)

# Apply SAME transformation on test data
X_test_var = selector.transform(X_test)

# Convert to DataFrame (optional but good)
selected_columns = X_train.columns[selector.get_support()]

X_train_var = pd.DataFrame(X_train_var, columns=selected_columns)
X_test_var = pd.DataFrame(X_test_var, columns=selected_columns)

print("After removing constant features:", X_train_var.shape[1])

import numpy as np

# Step 1: Compute correlation ONLY on training data
corr_matrix = X_train_var.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]

print("Descriptors to drop:", len(to_drop))

# Step 2: Drop from TRAIN
X_train_corr = X_train_var.drop(columns=to_drop)

# Step 3: Drop SAME columns from TEST
X_test_corr = X_test_var.drop(columns=to_drop)

print("After correlation removal (train):", X_train_corr.shape)
print("After correlation removal (test):", X_test_corr.shape)

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=400,
    random_state=42
)

# ✅ Train ONLY on training data
rf.fit(X_train_corr, y_train)

# Get importance
importance = pd.Series(
    rf.feature_importances_,
    index=X_train_corr.columns
)

# Sort
importance = importance.sort_values(ascending=False)

print("Top features:\n", importance.head(30))

# Select top 50 features
top_features = importance.head(50).index

# Apply SAME features to train and test
X_train_sel = X_train_corr[top_features]
X_test_sel = X_test_corr[top_features]

print("Final train shape:", X_train_sel.shape)
print("Final test shape:", X_test_sel.shape)

# Normalization

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit ONLY on training
X_train_scaled = scaler.fit_transform(X_train_sel)

# Apply to test
X_test_scaled = scaler.transform(X_test_sel)

# Find best K

from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt

bic_scores = []

k_values = [5, 8, 10, 12, 15, 18, 20, 22]

for k in k_values:

    gmm = GaussianMixture(
        n_components=k,
        covariance_type='full',
        random_state=42
    )

    gmm.fit(X_train_scaled)

    bic = gmm.bic(X_train_scaled)

    bic_scores.append(bic)

    print(f"K = {k}, BIC = {bic}")

# Silhouette Score for best K

from sklearn.metrics import silhouette_score

for k in k_values:

    gmm = GaussianMixture(n_components=k, random_state=42)

    labels = gmm.fit_predict(X_train_scaled)

    score = silhouette_score(X_train_scaled, labels)

    print(f"K = {k}, Silhouette Score = {score}")

# Apply GMM

from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=10, random_state=42)

gmm.fit(X_train_scaled)

train_gmm = gmm.predict_proba(X_train_scaled)
test_gmm = gmm.predict_proba(X_test_scaled)

# Combine
X_train_final = np.hstack([X_train_scaled, train_gmm])
X_test_final = np.hstack([X_test_scaled, test_gmm])

# GMM probabilities for TRAIN
gmm_train_probs = gmm.predict_proba(X_train_scaled)

gmm_train_df = pd.DataFrame(
    gmm_train_probs,
    columns=[f"GMM_Cluster_{i}" for i in range(gmm_train_probs.shape[1])]
)

# Convert scaled train to DataFrame
X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=X_train_sel.columns
)

# Concatenate
X_train_enhanced = pd.concat([X_train_scaled_df, gmm_train_df], axis=1)

print("Train shape:", X_train_enhanced.shape)
print(X_train_enhanced.columns[-10:])

# GMM probabilities for TEST
gmm_test_probs = gmm.predict_proba(X_test_scaled)

gmm_test_df = pd.DataFrame(
    gmm_test_probs,
    columns=[f"GMM_Cluster_{i}" for i in range(gmm_test_probs.shape[1])]
)

# Convert scaled test to DataFrame
X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=X_train_sel.columns   # IMPORTANT: use train columns
)

# Concatenate
X_test_enhanced = pd.concat([X_test_scaled_df, gmm_test_df], axis=1)

print("Test shape:", X_test_enhanced.shape)
print(X_test_enhanced.columns[-10:])

cluster_labels = gmm.predict(X_train_scaled)

df_analysis = pd.DataFrame({
    "Cluster": cluster_labels,
    "MoA": y_train.values
})

print(df_analysis.groupby("Cluster")["MoA"].value_counts())

# ---Function for evaluating metrices---

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    classification_report
)

def evaluate_model_full(model_name, y_test, y_pred, y_prob):

    print("\n==============================")
    print("MODEL:", model_name)
    print("==============================")

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    print("\nOverall Confusion Matrix:\n")
    print(cm)

    # Plot confusion matrix
    plt.figure(figsize=(8,6))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=le.classes_,
        yticklabels=le.classes_
    )

    plt.xlabel("Predicted MoA")
    plt.ylabel("Actual MoA")
    plt.title(model_name + " Confusion Matrix")

    plt.show()

    # ------------------------------
    # TP FP FN TN PER CLASS - CALCULATE HERE
    # ------------------------------

    TP = np.diag(cm)

    FP = np.sum(cm, axis=0) - TP

    FN = np.sum(cm, axis=1) - TP

    TN = np.sum(cm) - (TP + FP + FN)

    # --------------------------------
    # PER-MoA CONFUSION MATRICES
    # --------------------------------

    per_moa_cm_list = []

    for i, moa in enumerate(le.classes_):

        tp = TP[i]
        fp = FP[i]
        fn = FN[i]
        tn = TN[i]

        cm_moa = np.array([
            [tp, fn],
            [fp, tn]
        ])

        print("\nConfusion Matrix for MoA:", moa)
        print(cm_moa)

        plt.figure(figsize=(4,4))

        sns.heatmap(
            cm_moa,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=["Predicted "+moa, "Predicted Other"],
            yticklabels=["Actual "+moa, "Actual Other"]
        )

        plt.title(model_name + " - " + moa)

        plt.show()

        per_moa_cm_list.append({
            "MoA": moa,
            "TP": tp,
            "FN": fn,
            "FP": fp,
            "TN": tn
        })

    per_moa_cm_df = pd.DataFrame(per_moa_cm_list)

    per_moa_cm_df.to_excel(model_name+"_per_moa_confusion_matrix.xlsx", index=False)

    # Sensitivity (Recall)
    sensitivity = TP / (TP + FN)

    # Specificity
    specificity = TN / (TN + FP)

    per_class_df = pd.DataFrame({
        "MoA": le.classes_,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "Sensitivity": sensitivity,
        "Specificity": specificity
    })

    print("\nPer-MoA Metrics\n")
    print(per_class_df)

    # Save results
    per_class_df.to_excel(model_name+"_per_MoA_metrics.xlsx", index=False)

    # ------------------------------
    # OVERALL METRICS
    # ------------------------------

    overall_TP = TP.sum()

    overall_FP = FP.sum()

    overall_FN = FN.sum()

    overall_TN = TN.sum()

    overall_sensitivity = overall_TP / (overall_TP + overall_FN)

    overall_specificity = overall_TN / (overall_TN + overall_FP)

    overall_df = pd.DataFrame({
        "Metric": [
            "TP",
            "FP",
            "FN",
            "TN",
            "Sensitivity",
            "Specificity"
        ],
        "Value": [
            overall_TP,
            overall_FP,
            overall_FN,
            overall_TN,
            overall_sensitivity,
            overall_specificity
        ]
    })

    print("\nOverall Metrics\n")
    print(overall_df)

    overall_df.to_excel(model_name+"_overall_metrics.xlsx", index=False)

    # ------------------------------
    # Accuracy
    # ------------------------------

    acc = accuracy_score(y_test, y_pred)

    print("\nAccuracy:", acc)

    # ------------------------------
    # MCC
    # ------------------------------

    mcc = matthews_corrcoef(y_test, y_pred)

    print("MCC:", mcc)

    # ------------------------------
    # ROC AUC (Multiclass)
    # ------------------------------

    from sklearn.preprocessing import label_binarize

    y_test_bin = label_binarize(y_test, classes=np.unique(y_test))

    auc = roc_auc_score(y_test_bin, y_prob, multi_class="ovr")

    print("ROC-AUC:", auc)

    # Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features='sqrt'
)

# ✅ Train on FINAL enhanced features (IMPORTANT)
rf_model.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_rf = rf_model.predict(X_test_enhanced)

# Accuracy
print("Random Forest Accuracy:",
      accuracy_score(y_test_encoded, y_pred_rf))

# Classification Report
print(classification_report(y_test_encoded, y_pred_rf))

# Probabilities (for ROC-AUC)
y_prob_rf = rf_model.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "Random Forest",
    y_test_encoded,
    y_pred_rf,
    y_prob_rf
)

# Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features='sqrt'
)

# ✅ Train on FINAL enhanced features (IMPORTANT)
rf_model.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_rf = rf_model.predict(X_test_enhanced)

# Accuracy
print("Random Forest Accuracy:",
      accuracy_score(y_test_encoded, y_pred_rf))

# Classification Report
print(classification_report(y_test_encoded, y_pred_rf))

# Probabilities (for ROC-AUC)
y_prob_rf = rf_model.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "Random Forest",
    y_test_encoded,
    y_pred_rf,
    y_prob_rf
)


# XGBoost

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    eval_metric="mlogloss"
)

# ✅ Train on FINAL enhanced features
xgb.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_xgb = xgb.predict(X_test_enhanced)

# Accuracy
print("XGBoost Accuracy:",
      accuracy_score(y_test_encoded, y_pred_xgb))

# Classification Report
print(classification_report(y_test_encoded, y_pred_xgb))

# Probabilities (for ROC-AUC)
y_prob_xgb = xgb.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "XGBoost",
    y_test_encoded,
    y_pred_xgb,
    y_prob_xgb
)




k = 12

In [ ]:
import pandas as pd
import re

# ======================================
# 1. Load the Excel file
# ======================================
df = pd.read_excel("final_moa_cateogrization_Peptides+Smiles.xlsx")

# ======================================
# 2. Remove duplicate rows
# ======================================
df = df.drop_duplicates()

# ======================================
# 3. Fill missing MoA with "Unknown"
# ======================================
df["MoA"] = df["MoA"].fillna("Unknown")

# ======================================
# 4. Drop MoA with "Unknown"
# ======================================
df = df[df["MoA"] != "Unknown"]

# ======================================
# 4. Clean MoA text
# ======================================
def clean_moa(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", re.sub(r"[^a-z0-9\s]", " ", text))
    return text.strip()

df["MoA_clean"] = df["MoA"].apply(clean_moa)

# ======================================
# 5. Categorization
# ======================================
def categorize_moa(text):

    text = text.lower()

    if text == "unknown":
        return "Multi-target / Polypharmacology"

    # 1 Membrane / Cell Envelope Disruption
    if any(k in text for k in [
        "membrane",
        "cell membrane",
        "pore",
        "permeability",
        "depolar",
        "lipid",
        "bilayer",
        "detergent",
        "ferriprotoporphyrin",
        "reactive nitro radical",
        "reactive intermediates",
        "membrane depolarization",
        "membrane permeabilization",
        "membrane damage",
        "membrane disrupting",
        "breaks down membrane lipids",
        "destroys cell membranes"
    ]):
        return "Membrane / Cell Envelope Disruption"

    # 2 Cell Wall Biosynthesis Inhibition
    if any(k in text for k in [
        "penicillin-binding protein",
        "pbp",
        "peptidoglycan",
        "cell wall",
        "lipid ii",
        "alanine racemase",
        "beta lactamase",
        "beta-lactamase",
        "penicillin binding protein",
        "fab i",
        "enoyl acyl carrier protein reductase"
    ]):
        return "Cell Wall Biosynthesis Inhibition"

    # 3 Protein Synthesis Inhibition
    if any(k in text for k in [
        "ribosome",
        "70s",
        "30s",
        "50s",
        "translation",
        "elongation factor",
        "trna",
        "protein synthesis",
        "trna synthetase",
        "trna ligase"
    ]):
        return "Protein Synthesis Inhibition"

    # 4 Nucleic Acid Synthesis / Integrity Disruption
    if any(k in text for k in [
        "dna",
        "rna",
        "gyrase",
        "topoisomerase",
        "polymerase",
        "replication",
        "intercalat",
        "dna binding",
        "dna disrupting",
        "dna inhibitor",
        "ribonucleotide reductase"
    ]):
        return "Nucleic Acid Synthesis / Integrity Disruption"

    # 5 Metabolic Pathway Inhibition
    if any(k in text for k in [
        "dihydrofolate reductase",
        "dihydropteroate synthase",
        "folate",
        "biosynthesis",
        "metabolism",
        "amidophosphoribosyltransferase",
        "oxidoreductase",
        "thymidylate synthase",
        "inosine monophosphate dehydrogenase",
        "dxr",
        "reductase",
        "synthase",
        "dehydrogenase",
        "cyclooxygenase",
        "carbonic anhydrase",
        "monoamine oxidase",
        "hmg coa reductase",
        "xanthine dehydrogenase",
        "caspase",
        "matrix metalloproteinase",
        "reductoisomerase",
        "dioxp reductoisomerase",
        "dxr",
        "dipeptidase",
        "diacylglycerol o acyltransferase",
        "dgat",
        "glucosyltransferase",
        "ceramide glucosyltransferase"
    ]):
        return "Metabolic Pathway Inhibition"

    # 6 Redox / Oxidative Stress Induction
    if any(k in text for k in [
        "oxidative",
        "redox",
        "ros",
        "reactive",
        "nitro",
        "oxidase",
        "reducing agent",
        "antioxidant",
        "increases ros",
        "chelating",
        "copper chelating",
        "iron chelating",
        "aluminium chelating",
        "glutathione",
        "glutathione precursor",
        "redox regulator"
    ]):
        return "Redox / Oxidative Stress Induction"

    # 7 Regulatory / Signalling Interference
    if any(k in text for k in [
        "quorum",
        "two component",
        "regulation",
        "signalling",
        "stress response",
        "transcription",
        "toll like receptor",
        "nf kb",
        "mtor",
        "map kinase",
        "cyclin dependent kinase",
        "biofilm related genes",
        "guanylate cyclase",
        "cyclase activator",
        "cyclase inhibitor",
        "cap binding complex",
        "signal transduction modulator",
        "arca",
        "arcb",
        "arcc",
        "arcd",
        "arc gene",
        "arginine deiminase pathway"
    ]):
        return "Regulatory / Signalling Interference"

    # 8 Biofilm Matrix Disruption
    if any(k in text for k in [
        "biofilm",
        "edna",
        "nuclease"
    ]):
        return "Biofilm Matrix Disruption"

    # 9 Antifungal Sterol Biosynthesis
    if any(k in text for k in [
        "cytochrome p450",
        "cyp51",
        "lanosterol",
        "ergosterol",
        "sterol biosynthesis",
        "squalene monooxygenase"
    ]):
        return "Antifungal Sterol Biosynthesis Inhibitors"

    # 10 Antiviral agents
    if any(k in text for k in [
        "hiv",
        "herpesvirus",
        "influenza",
        "hepatitis c",
        "reverse transcriptase",
        "viral protease",
        "viral capsid",
        "neuraminidase",
        "viral polymerase",
        "viral", "hepatitis", "capsid", "viral kinase", "helicase", "primase",
        "hiv protease",
        "hiv 1 protease",
        "viral protease",
        "pul97",
        "viral phosphotransferase",
        "human immunodeficiency virus type 1 protease"
    ]):
        return "Antiviral agents"

    # 11 Affecting Human Physiology
    if any(k in text for k in [
        "adrenergic receptor",
        "dopamine receptor",
        "serotonin transporter",
        "histamine receptor",
        "muscarinic receptor",
        "angiotensin receptor",
        "opioid receptor",
        "cannabinoid receptor",
        "vitamin d receptor",
        "androgen receptor",
        "estrogen receptor",
        "progesterone receptor",
        "glucocorticoid receptor",
        "receptor",
        "agonist",
        "antagonist",
        "partial agonist",
        "inverse agonist",
        "allosteric modulator",
        "allosteric antagonist",
        "positive modulator",
        "negative modulator",
        "retinoid receptor",
        "retinoic acid receptor",
        "retinoid x receptor",
        "adenosine receptor",
        "dopamine receptor",
        "serotonin receptor",
        "histamine receptor",
        "angiotensin receptor",
        "muscarinic receptor",
        "nicotinic receptor",
        "cannabinoid receptor",
        "opioid receptor",
        "purinergic receptor",
        "vanilloid receptor",
        "bile acid receptor",
        "fxr receptor",
        "peroxisome proliferator activated receptor",
        "ppar",
        "glucocorticoid receptor",
        "mineralocorticoid receptor",
        "estrogen receptor",
        "androgen receptor",
        "progesterone receptor",
        "vitamin d receptor",
        "chemokine receptor",
        "cxcr",
        "s1p receptor",
        "sphingosine receptor",
        "nmda receptor",
        "gaba receptor",
        "acetylcholine receptor",

        "ion channel",
        "channel blocker",
        "channel opener",
        "channel modulator",
        "voltage gated",
        "voltage activated",
        "calcium channel",
        "sodium channel",
        "potassium channel",
        "chloride channel",
        "anion channel",
        "herg",
        "enac",
        "ryanodine receptor",
        "l type calcium channel",
        "inwardly rectifying potassium channel",
        "calcium activated potassium channel",

        "acetylcholinesterase",
        "cholinesterase",
        "dopamine transporter",
        "serotonin transporter",
        "norepinephrine transporter",
        "gaba transporter",
        "synaptic vesicle transporter",
        "vesicular amine transporter",
        "sv2a",
        "neurotransmitter reuptake",
        "bcr abl",
        "abl kinase",
        "fusion protein inhibitor"

        "angiotensin-converting enzyme",
        "angiotensin converting enzyme",
        "ace inhibitor",
        "lipoxygenase",
        "cyclooxygenase",
        "phosphodiesterase",
        "pde inhibitor",
        "neprilysin",
        "lipase",
        "glucosidase",
        "decarboxylase",
        "monoamine oxidase",
        "thrombin",
        "farnesyltransferase",
        "kinase",
        "phosphatase",
        "ubiquitin ligase",
        "aurora kinase",
        "pdgfr",
        "pdk1",
        "ikk",

        "transporter",
        "solute carrier",
        "slc",
        "cotransporter",
        "sodium chloride cotransporter",
        "sodium potassium chloride cotransporter",
        "npc1l1",
        "amine transporter",
        "renal transporter",

        "insulin receptor",
        "hormone receptor",
        "glucose metabolism",
        "lipid metabolism",
        "fatty acid metabolism",
        "ppar alpha",
        "ppar gamma",
        "ppar delta",
        "metabolic regulator",

        "growth factor receptor",
        "egfr",
        "map kinase",
        "mtor",
        "nf kb",
        "signal transduction",
        "transcription factor",
        "cell signaling",
        "cytokine receptor",

        "tubulin",
        "microtubule",
        "cytoskeleton",
        "spindle assembly",
        "microtubule stabilizer",
        "microtubule inhibitor"
        "potassium transporting atpase",
        "npc1l1",
        "niemmann-pick c1",
        "niemann pick c1",
        "synaptic vesicle glycoprotein",
        "sv2a",
        "sv2a modulator",
        "laxative",
        "potassium-transporting atpase inhibitor",
        "potassium transporting atpase",
        "potassium atpase",
        "synaptic vesicle glycoprotein 2a",
        "cholesterol absorption inhibitor"



    ]):
        return "Affecting Human Physiology"

    # 12 Multi-target
    if any(k in text for k in [
        "multi target",
        "multiple",
        "polypharmacology",
        "pleiotropic"
    ]):
        return "Multi-target / Polypharmacology"

    return "Others"


# ======================================
# 6. Apply categorization
# ======================================
df["MoA_category"] = df["MoA_clean"].apply(categorize_moa)

# ======================================
# 7. Keep only required columns
# ======================================
df_final = df[["Name", "SMILES", "MoA_category"]]

# ======================================
# 8. Rename column
# ======================================
df_final = df_final.rename(columns={"MoA_category": "MoA"})

# ======================================
# 9. Check distribution
# ======================================
print(df["MoA_category"].value_counts())

# ======================================
# 10. Save final file
# ======================================
df_final.to_excel("final_moa_categorized.xlsx", index=False)

print(df[df["MoA_category"]=="Others"]["MoA"].unique())

# ======================================
# 10. Save categorized data into multiple sheets
# ======================================

with pd.ExcelWriter("final_moa_categorized.xlsx", engine="xlsxwriter") as writer:

    # Main categorized dataset
    df_final.to_excel(writer, sheet_name="All_Data_PA", index=False)

    # Human physiology targets
    human_targets = df[df["MoA_category"] == "Affecting Human Physiology"]
    human_targets.to_excel(writer, sheet_name="Human_Physiology_Targets", index=False)

    # Antiviral agents
    antiviral_targets = df[df["MoA_category"] == "Antiviral agents"]
    antiviral_targets.to_excel(writer, sheet_name="Antiviral_Targets", index=False)

    # Antifungal agents
    antifungal_targets = df[df["MoA_category"] == "Antifungal Sterol Biosynthesis Inhibitors"]
    antifungal_targets.to_excel(writer, sheet_name="Antifungal_Targets", index=False)

    # Others
    Others = df[df["MoA_category"] == "Others"]
    Others.to_excel(writer, sheet_name="Others", index=False)

    # ======================================
    # Direct Antibacterial (remove unwanted categories)
    # ======================================
    direct_antibacterial = df[~df["MoA_category"].isin([
        "Affecting Human Physiology",
        "Antiviral agents",
        "Antifungal Sterol Biosynthesis Inhibitors",
        "Others"
    ])]

    direct_antibacterial.to_excel(writer, sheet_name="Direct_Antibacterial", index=False)

print("Excel file created with multiple sheets including Direct_Antibacterial.")

import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

# Load dataset from the correct file and sheet
df = pd.read_excel("final_moa_categorized.xlsx", sheet_name="Direct_Antibacterial")

# Get list of all RDKit descriptors
descriptor_names = [desc[0] for desc in Descriptors.descList]

def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return [None]*len(descriptor_names)

    descriptor_values = []

    for name, function in Descriptors.descList:
        try:
            value = function(mol)
        except:
            value = None
        descriptor_values.append(value)

    return descriptor_values


# Apply descriptor calculation
descriptor_data = df["SMILES"].apply(calculate_descriptors)

# Convert to dataframe
X = pd.DataFrame(descriptor_data.tolist(), columns=descriptor_names)

# Target variable
y = df["MoA_category"]

# Combine if needed
final_df = pd.concat([X, y], axis=1)

# Save descriptor dataset
final_df.to_excel("MoA_rdkit_all_descriptors_calculated.xlsx", index=False)

print("Total descriptors extracted:", len(descriptor_names))
print("Dataset shape:", final_df.shape)


# Load Descriptor data

import pandas as pd

df = pd.read_excel("MoA_rdkit_all_descriptors_calculated.xlsx")

print("Dataset shape:", df.shape)

df.head()

# seperate descriptors and label(moa)

X = df.drop(columns=["MoA_category"])
y = df["MoA_category"]

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

# -------------------------------
#  FILTER RARE CLASSES HERE
# -------------------------------

df_model = X.copy()
df_model["MoA_category"] = y

counts = df_model["MoA_category"].value_counts()
valid_classes = counts[counts >= 3].index

df_filtered = df_model[df_model["MoA_category"].isin(valid_classes)].copy()

# Update X and y AFTER filtering
X = df_filtered.drop(columns=["MoA_category"])
y = df_filtered["MoA_category"]

print("After filtering:", X.shape, y.shape)

import numpy as np

# Replace infinity values
X = X.replace([np.inf, -np.inf], np.nan)

# Convert to float32 to catch values too large for this dtype and re-handle potential new NaNs
X = X.astype(np.float32)

# Replace any new inf values that might have been created due to float32 conversion overflow
X = X.replace([np.inf, -np.inf], np.nan)

# Fill missing values

X = X.fillna(X.median())

# Remove Extremely Large Descriptors
# Some descriptors can reach 10¹² or more, causing overflow.

X.describe().T.sort_values("max", ascending=False).head(10)

# A safe solution is to clip values.

X = X.clip(-1e6, 1e6)

# A safe solution is to clip values.

X = X.astype(np.float64)

# check missing, infinity and large values

import numpy as np

print("NaN values:", np.isnan(X).sum().sum())
print("Infinity values:", np.isinf(X).sum().sum())
print("Max value in dataset:", np.nanmax(X.values))

# Train-Test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# label encoding

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_encoded = le.fit_transform(y_train)

y_test_encoded = le.transform(y_test)

# Feature stability

from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0)

# Fit ONLY on training data
X_train_var = selector.fit_transform(X_train)

# Apply SAME transformation on test data
X_test_var = selector.transform(X_test)

# Convert to DataFrame (optional but good)
selected_columns = X_train.columns[selector.get_support()]

X_train_var = pd.DataFrame(X_train_var, columns=selected_columns)
X_test_var = pd.DataFrame(X_test_var, columns=selected_columns)

print("After removing constant features:", X_train_var.shape[1])

import numpy as np

# Step 1: Compute correlation ONLY on training data
corr_matrix = X_train_var.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]

print("Descriptors to drop:", len(to_drop))

# Step 2: Drop from TRAIN
X_train_corr = X_train_var.drop(columns=to_drop)

# Step 3: Drop SAME columns from TEST
X_test_corr = X_test_var.drop(columns=to_drop)

print("After correlation removal (train):", X_train_corr.shape)
print("After correlation removal (test):", X_test_corr.shape)

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=400,
    random_state=42
)

# ✅ Train ONLY on training data
rf.fit(X_train_corr, y_train)

# Get importance
importance = pd.Series(
    rf.feature_importances_,
    index=X_train_corr.columns
)

# Sort
importance = importance.sort_values(ascending=False)

print("Top features:\n", importance.head(30))

# Select top 50 features
top_features = importance.head(50).index

# Apply SAME features to train and test
X_train_sel = X_train_corr[top_features]
X_test_sel = X_test_corr[top_features]

print("Final train shape:", X_train_sel.shape)
print("Final test shape:", X_test_sel.shape)

# Normalization

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit ONLY on training
X_train_scaled = scaler.fit_transform(X_train_sel)

# Apply to test
X_test_scaled = scaler.transform(X_test_sel)

# Find best K

from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt

bic_scores = []

k_values = [5, 8, 10, 12, 15, 18, 20, 22]

for k in k_values:

    gmm = GaussianMixture(
        n_components=k,
        covariance_type='full',
        random_state=42
    )

    gmm.fit(X_train_scaled)

    bic = gmm.bic(X_train_scaled)

    bic_scores.append(bic)

    print(f"K = {k}, BIC = {bic}")

# Silhouette Score for best K

from sklearn.metrics import silhouette_score

for k in k_values:

    gmm = GaussianMixture(n_components=k, random_state=42)

    labels = gmm.fit_predict(X_train_scaled)

    score = silhouette_score(X_train_scaled, labels)

    print(f"K = {k}, Silhouette Score = {score}")

# Apply GMM

from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=12, random_state=42)

gmm.fit(X_train_scaled)

train_gmm = gmm.predict_proba(X_train_scaled)
test_gmm = gmm.predict_proba(X_test_scaled)

# Combine
X_train_final = np.hstack([X_train_scaled, train_gmm])
X_test_final = np.hstack([X_test_scaled, test_gmm])

# GMM probabilities for TRAIN
gmm_train_probs = gmm.predict_proba(X_train_scaled)

gmm_train_df = pd.DataFrame(
    gmm_train_probs,
    columns=[f"GMM_Cluster_{i}" for i in range(gmm_train_probs.shape[1])]
)

# Convert scaled train to DataFrame
X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=X_train_sel.columns
)

# Concatenate
X_train_enhanced = pd.concat([X_train_scaled_df, gmm_train_df], axis=1)

print("Train shape:", X_train_enhanced.shape)
print(X_train_enhanced.columns[-10:])

# GMM probabilities for TEST
gmm_test_probs = gmm.predict_proba(X_test_scaled)

gmm_test_df = pd.DataFrame(
    gmm_test_probs,
    columns=[f"GMM_Cluster_{i}" for i in range(gmm_test_probs.shape[1])]
)

# Convert scaled test to DataFrame
X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=X_train_sel.columns   # IMPORTANT: use train columns
)

# Concatenate
X_test_enhanced = pd.concat([X_test_scaled_df, gmm_test_df], axis=1)

print("Test shape:", X_test_enhanced.shape)
print(X_test_enhanced.columns[-10:])

cluster_labels = gmm.predict(X_train_scaled)

df_analysis = pd.DataFrame({
    "Cluster": cluster_labels,
    "MoA": y_train.values
})

print(df_analysis.groupby("Cluster")["MoA"].value_counts())

# ---Function for evaluating metrices---

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    classification_report
)

def evaluate_model_full(model_name, y_test, y_pred, y_prob):

    print("\n==============================")
    print("MODEL:", model_name)
    print("==============================")

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    print("\nOverall Confusion Matrix:\n")
    print(cm)

    # Plot confusion matrix
    plt.figure(figsize=(8,6))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=le.classes_,
        yticklabels=le.classes_
    )

    plt.xlabel("Predicted MoA")
    plt.ylabel("Actual MoA")
    plt.title(model_name + " Confusion Matrix")

    plt.show()

    # ------------------------------
    # TP FP FN TN PER CLASS - CALCULATE HERE
    # ------------------------------

    TP = np.diag(cm)

    FP = np.sum(cm, axis=0) - TP

    FN = np.sum(cm, axis=1) - TP

    TN = np.sum(cm) - (TP + FP + FN)

    # --------------------------------
    # PER-MoA CONFUSION MATRICES
    # --------------------------------

    per_moa_cm_list = []

    for i, moa in enumerate(le.classes_):

        tp = TP[i]
        fp = FP[i]
        fn = FN[i]
        tn = TN[i]

        cm_moa = np.array([
            [tp, fn],
            [fp, tn]
        ])

        print("\nConfusion Matrix for MoA:", moa)
        print(cm_moa)

        plt.figure(figsize=(4,4))

        sns.heatmap(
            cm_moa,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=["Predicted "+moa, "Predicted Other"],
            yticklabels=["Actual "+moa, "Actual Other"]
        )

        plt.title(model_name + " - " + moa)

        plt.show()

        per_moa_cm_list.append({
            "MoA": moa,
            "TP": tp,
            "FN": fn,
            "FP": fp,
            "TN": tn
        })

    per_moa_cm_df = pd.DataFrame(per_moa_cm_list)

    per_moa_cm_df.to_excel(model_name+"_per_moa_confusion_matrix.xlsx", index=False)

    # Sensitivity (Recall)
    sensitivity = TP / (TP + FN)

    # Specificity
    specificity = TN / (TN + FP)

    per_class_df = pd.DataFrame({
        "MoA": le.classes_,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "Sensitivity": sensitivity,
        "Specificity": specificity
    })

    print("\nPer-MoA Metrics\n")
    print(per_class_df)

    # Save results
    per_class_df.to_excel(model_name+"_per_MoA_metrics.xlsx", index=False)

    # ------------------------------
    # OVERALL METRICS
    # ------------------------------

    overall_TP = TP.sum()

    overall_FP = FP.sum()

    overall_FN = FN.sum()

    overall_TN = TN.sum()

    overall_sensitivity = overall_TP / (overall_TP + overall_FN)

    overall_specificity = overall_TN / (overall_TN + overall_FP)

    overall_df = pd.DataFrame({
        "Metric": [
            "TP",
            "FP",
            "FN",
            "TN",
            "Sensitivity",
            "Specificity"
        ],
        "Value": [
            overall_TP,
            overall_FP,
            overall_FN,
            overall_TN,
            overall_sensitivity,
            overall_specificity
        ]
    })

    print("\nOverall Metrics\n")
    print(overall_df)

    overall_df.to_excel(model_name+"_overall_metrics.xlsx", index=False)

    # ------------------------------
    # Accuracy
    # ------------------------------

    acc = accuracy_score(y_test, y_pred)

    print("\nAccuracy:", acc)

    # ------------------------------
    # MCC
    # ------------------------------

    mcc = matthews_corrcoef(y_test, y_pred)

    print("MCC:", mcc)

    # ------------------------------
    # ROC AUC (Multiclass)
    # ------------------------------

    from sklearn.preprocessing import label_binarize

    y_test_bin = label_binarize(y_test, classes=np.unique(y_test))

    auc = roc_auc_score(y_test_bin, y_prob, multi_class="ovr")

    print("ROC-AUC:", auc)

    # Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features='sqrt'
)

# ✅ Train on FINAL enhanced features (IMPORTANT)
rf_model.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_rf = rf_model.predict(X_test_enhanced)

# Accuracy
print("Random Forest Accuracy:",
      accuracy_score(y_test_encoded, y_pred_rf))

# Classification Report
print(classification_report(y_test_encoded, y_pred_rf))

# Probabilities (for ROC-AUC)
y_prob_rf = rf_model.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "Random Forest",
    y_test_encoded,
    y_pred_rf,
    y_prob_rf
)

# Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features='sqrt'
)

# ✅ Train on FINAL enhanced features (IMPORTANT)
rf_model.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_rf = rf_model.predict(X_test_enhanced)

# Accuracy
print("Random Forest Accuracy:",
      accuracy_score(y_test_encoded, y_pred_rf))

# Classification Report
print(classification_report(y_test_encoded, y_pred_rf))

# Probabilities (for ROC-AUC)
y_prob_rf = rf_model.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "Random Forest",
    y_test_encoded,
    y_pred_rf,
    y_prob_rf
)


# XGBoost

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    eval_metric="mlogloss"
)

# ✅ Train on FINAL enhanced features
xgb.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_xgb = xgb.predict(X_test_enhanced)

# Accuracy
print("XGBoost Accuracy:",
      accuracy_score(y_test_encoded, y_pred_xgb))

# Classification Report
print(classification_report(y_test_encoded, y_pred_xgb))

# Probabilities (for ROC-AUC)
y_prob_xgb = xgb.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "XGBoost",
    y_test_encoded,
    y_pred_xgb,
    y_prob_xgb
)




k = 15

In [ ]:
import pandas as pd
import re

# ======================================
# 1. Load the Excel file
# ======================================
df = pd.read_excel("final_moa_cateogrization_Peptides+Smiles.xlsx")

# ======================================
# 2. Remove duplicate rows
# ======================================
df = df.drop_duplicates()

# ======================================
# 3. Fill missing MoA with "Unknown"
# ======================================
df["MoA"] = df["MoA"].fillna("Unknown")

# ======================================
# 4. Drop MoA with "Unknown"
# ======================================
df = df[df["MoA"] != "Unknown"]

# ======================================
# 4. Clean MoA text
# ======================================
def clean_moa(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", re.sub(r"[^a-z0-9\s]", " ", text))
    return text.strip()

df["MoA_clean"] = df["MoA"].apply(clean_moa)

# ======================================
# 5. Categorization
# ======================================
def categorize_moa(text):

    text = text.lower()

    if text == "unknown":
        return "Multi-target / Polypharmacology"

    # 1 Membrane / Cell Envelope Disruption
    if any(k in text for k in [
        "membrane",
        "cell membrane",
        "pore",
        "permeability",
        "depolar",
        "lipid",
        "bilayer",
        "detergent",
        "ferriprotoporphyrin",
        "reactive nitro radical",
        "reactive intermediates",
        "membrane depolarization",
        "membrane permeabilization",
        "membrane damage",
        "membrane disrupting",
        "breaks down membrane lipids",
        "destroys cell membranes"
    ]):
        return "Membrane / Cell Envelope Disruption"

    # 2 Cell Wall Biosynthesis Inhibition
    if any(k in text for k in [
        "penicillin-binding protein",
        "pbp",
        "peptidoglycan",
        "cell wall",
        "lipid ii",
        "alanine racemase",
        "beta lactamase",
        "beta-lactamase",
        "penicillin binding protein",
        "fab i",
        "enoyl acyl carrier protein reductase"
    ]):
        return "Cell Wall Biosynthesis Inhibition"

    # 3 Protein Synthesis Inhibition
    if any(k in text for k in [
        "ribosome",
        "70s",
        "30s",
        "50s",
        "translation",
        "elongation factor",
        "trna",
        "protein synthesis",
        "trna synthetase",
        "trna ligase"
    ]):
        return "Protein Synthesis Inhibition"

    # 4 Nucleic Acid Synthesis / Integrity Disruption
    if any(k in text for k in [
        "dna",
        "rna",
        "gyrase",
        "topoisomerase",
        "polymerase",
        "replication",
        "intercalat",
        "dna binding",
        "dna disrupting",
        "dna inhibitor",
        "ribonucleotide reductase"
    ]):
        return "Nucleic Acid Synthesis / Integrity Disruption"

    # 5 Metabolic Pathway Inhibition
    if any(k in text for k in [
        "dihydrofolate reductase",
        "dihydropteroate synthase",
        "folate",
        "biosynthesis",
        "metabolism",
        "amidophosphoribosyltransferase",
        "oxidoreductase",
        "thymidylate synthase",
        "inosine monophosphate dehydrogenase",
        "dxr",
        "reductase",
        "synthase",
        "dehydrogenase",
        "cyclooxygenase",
        "carbonic anhydrase",
        "monoamine oxidase",
        "hmg coa reductase",
        "xanthine dehydrogenase",
        "caspase",
        "matrix metalloproteinase",
        "reductoisomerase",
        "dioxp reductoisomerase",
        "dxr",
        "dipeptidase",
        "diacylglycerol o acyltransferase",
        "dgat",
        "glucosyltransferase",
        "ceramide glucosyltransferase"
    ]):
        return "Metabolic Pathway Inhibition"

    # 6 Redox / Oxidative Stress Induction
    if any(k in text for k in [
        "oxidative",
        "redox",
        "ros",
        "reactive",
        "nitro",
        "oxidase",
        "reducing agent",
        "antioxidant",
        "increases ros",
        "chelating",
        "copper chelating",
        "iron chelating",
        "aluminium chelating",
        "glutathione",
        "glutathione precursor",
        "redox regulator"
    ]):
        return "Redox / Oxidative Stress Induction"

    # 7 Regulatory / Signalling Interference
    if any(k in text for k in [
        "quorum",
        "two component",
        "regulation",
        "signalling",
        "stress response",
        "transcription",
        "toll like receptor",
        "nf kb",
        "mtor",
        "map kinase",
        "cyclin dependent kinase",
        "biofilm related genes",
        "guanylate cyclase",
        "cyclase activator",
        "cyclase inhibitor",
        "cap binding complex",
        "signal transduction modulator",
        "arca",
        "arcb",
        "arcc",
        "arcd",
        "arc gene",
        "arginine deiminase pathway"
    ]):
        return "Regulatory / Signalling Interference"

    # 8 Biofilm Matrix Disruption
    if any(k in text for k in [
        "biofilm",
        "edna",
        "nuclease"
    ]):
        return "Biofilm Matrix Disruption"

    # 9 Antifungal Sterol Biosynthesis
    if any(k in text for k in [
        "cytochrome p450",
        "cyp51",
        "lanosterol",
        "ergosterol",
        "sterol biosynthesis",
        "squalene monooxygenase"
    ]):
        return "Antifungal Sterol Biosynthesis Inhibitors"

    # 10 Antiviral agents
    if any(k in text for k in [
        "hiv",
        "herpesvirus",
        "influenza",
        "hepatitis c",
        "reverse transcriptase",
        "viral protease",
        "viral capsid",
        "neuraminidase",
        "viral polymerase",
        "viral", "hepatitis", "capsid", "viral kinase", "helicase", "primase",
        "hiv protease",
        "hiv 1 protease",
        "viral protease",
        "pul97",
        "viral phosphotransferase",
        "human immunodeficiency virus type 1 protease"
    ]):
        return "Antiviral agents"

    # 11 Affecting Human Physiology
    if any(k in text for k in [
        "adrenergic receptor",
        "dopamine receptor",
        "serotonin transporter",
        "histamine receptor",
        "muscarinic receptor",
        "angiotensin receptor",
        "opioid receptor",
        "cannabinoid receptor",
        "vitamin d receptor",
        "androgen receptor",
        "estrogen receptor",
        "progesterone receptor",
        "glucocorticoid receptor",
        "receptor",
        "agonist",
        "antagonist",
        "partial agonist",
        "inverse agonist",
        "allosteric modulator",
        "allosteric antagonist",
        "positive modulator",
        "negative modulator",
        "retinoid receptor",
        "retinoic acid receptor",
        "retinoid x receptor",
        "adenosine receptor",
        "dopamine receptor",
        "serotonin receptor",
        "histamine receptor",
        "angiotensin receptor",
        "muscarinic receptor",
        "nicotinic receptor",
        "cannabinoid receptor",
        "opioid receptor",
        "purinergic receptor",
        "vanilloid receptor",
        "bile acid receptor",
        "fxr receptor",
        "peroxisome proliferator activated receptor",
        "ppar",
        "glucocorticoid receptor",
        "mineralocorticoid receptor",
        "estrogen receptor",
        "androgen receptor",
        "progesterone receptor",
        "vitamin d receptor",
        "chemokine receptor",
        "cxcr",
        "s1p receptor",
        "sphingosine receptor",
        "nmda receptor",
        "gaba receptor",
        "acetylcholine receptor",

        "ion channel",
        "channel blocker",
        "channel opener",
        "channel modulator",
        "voltage gated",
        "voltage activated",
        "calcium channel",
        "sodium channel",
        "potassium channel",
        "chloride channel",
        "anion channel",
        "herg",
        "enac",
        "ryanodine receptor",
        "l type calcium channel",
        "inwardly rectifying potassium channel",
        "calcium activated potassium channel",

        "acetylcholinesterase",
        "cholinesterase",
        "dopamine transporter",
        "serotonin transporter",
        "norepinephrine transporter",
        "gaba transporter",
        "synaptic vesicle transporter",
        "vesicular amine transporter",
        "sv2a",
        "neurotransmitter reuptake",
        "bcr abl",
        "abl kinase",
        "fusion protein inhibitor"

        "angiotensin-converting enzyme",
        "angiotensin converting enzyme",
        "ace inhibitor",
        "lipoxygenase",
        "cyclooxygenase",
        "phosphodiesterase",
        "pde inhibitor",
        "neprilysin",
        "lipase",
        "glucosidase",
        "decarboxylase",
        "monoamine oxidase",
        "thrombin",
        "farnesyltransferase",
        "kinase",
        "phosphatase",
        "ubiquitin ligase",
        "aurora kinase",
        "pdgfr",
        "pdk1",
        "ikk",

        "transporter",
        "solute carrier",
        "slc",
        "cotransporter",
        "sodium chloride cotransporter",
        "sodium potassium chloride cotransporter",
        "npc1l1",
        "amine transporter",
        "renal transporter",

        "insulin receptor",
        "hormone receptor",
        "glucose metabolism",
        "lipid metabolism",
        "fatty acid metabolism",
        "ppar alpha",
        "ppar gamma",
        "ppar delta",
        "metabolic regulator",

        "growth factor receptor",
        "egfr",
        "map kinase",
        "mtor",
        "nf kb",
        "signal transduction",
        "transcription factor",
        "cell signaling",
        "cytokine receptor",

        "tubulin",
        "microtubule",
        "cytoskeleton",
        "spindle assembly",
        "microtubule stabilizer",
        "microtubule inhibitor"
        "potassium transporting atpase",
        "npc1l1",
        "niemmann-pick c1",
        "niemann pick c1",
        "synaptic vesicle glycoprotein",
        "sv2a",
        "sv2a modulator",
        "laxative",
        "potassium-transporting atpase inhibitor",
        "potassium transporting atpase",
        "potassium atpase",
        "synaptic vesicle glycoprotein 2a",
        "cholesterol absorption inhibitor"



    ]):
        return "Affecting Human Physiology"

    # 12 Multi-target
    if any(k in text for k in [
        "multi target",
        "multiple",
        "polypharmacology",
        "pleiotropic"
    ]):
        return "Multi-target / Polypharmacology"

    return "Others"


# ======================================
# 6. Apply categorization
# ======================================
df["MoA_category"] = df["MoA_clean"].apply(categorize_moa)

# ======================================
# 7. Keep only required columns
# ======================================
df_final = df[["Name", "SMILES", "MoA_category"]]

# ======================================
# 8. Rename column
# ======================================
df_final = df_final.rename(columns={"MoA_category": "MoA"})

# ======================================
# 9. Check distribution
# ======================================
print(df["MoA_category"].value_counts())

# ======================================
# 10. Save final file
# ======================================
df_final.to_excel("final_moa_categorized.xlsx", index=False)

print(df[df["MoA_category"]=="Others"]["MoA"].unique())

# ======================================
# 10. Save categorized data into multiple sheets
# ======================================

with pd.ExcelWriter("final_moa_categorized.xlsx", engine="xlsxwriter") as writer:

    # Main categorized dataset
    df_final.to_excel(writer, sheet_name="All_Data_PA", index=False)

    # Human physiology targets
    human_targets = df[df["MoA_category"] == "Affecting Human Physiology"]
    human_targets.to_excel(writer, sheet_name="Human_Physiology_Targets", index=False)

    # Antiviral agents
    antiviral_targets = df[df["MoA_category"] == "Antiviral agents"]
    antiviral_targets.to_excel(writer, sheet_name="Antiviral_Targets", index=False)

    # Antifungal agents
    antifungal_targets = df[df["MoA_category"] == "Antifungal Sterol Biosynthesis Inhibitors"]
    antifungal_targets.to_excel(writer, sheet_name="Antifungal_Targets", index=False)

    # Others
    Others = df[df["MoA_category"] == "Others"]
    Others.to_excel(writer, sheet_name="Others", index=False)

    # ======================================
    # Direct Antibacterial (remove unwanted categories)
    # ======================================
    direct_antibacterial = df[~df["MoA_category"].isin([
        "Affecting Human Physiology",
        "Antiviral agents",
        "Antifungal Sterol Biosynthesis Inhibitors",
        "Others"
    ])]

    direct_antibacterial.to_excel(writer, sheet_name="Direct_Antibacterial", index=False)

print("Excel file created with multiple sheets including Direct_Antibacterial.")

import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

# Load dataset from the correct file and sheet
df = pd.read_excel("final_moa_categorized.xlsx", sheet_name="Direct_Antibacterial")

# Get list of all RDKit descriptors
descriptor_names = [desc[0] for desc in Descriptors.descList]

def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return [None]*len(descriptor_names)

    descriptor_values = []

    for name, function in Descriptors.descList:
        try:
            value = function(mol)
        except:
            value = None
        descriptor_values.append(value)

    return descriptor_values


# Apply descriptor calculation
descriptor_data = df["SMILES"].apply(calculate_descriptors)

# Convert to dataframe
X = pd.DataFrame(descriptor_data.tolist(), columns=descriptor_names)

# Target variable
y = df["MoA_category"]

# Combine if needed
final_df = pd.concat([X, y], axis=1)

# Save descriptor dataset
final_df.to_excel("MoA_rdkit_all_descriptors_calculated.xlsx", index=False)

print("Total descriptors extracted:", len(descriptor_names))
print("Dataset shape:", final_df.shape)


# Load Descriptor data

import pandas as pd

df = pd.read_excel("MoA_rdkit_all_descriptors_calculated.xlsx")

print("Dataset shape:", df.shape)

df.head()

# seperate descriptors and label(moa)

X = df.drop(columns=["MoA_category"])
y = df["MoA_category"]

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

# -------------------------------
#  FILTER RARE CLASSES HERE
# -------------------------------

df_model = X.copy()
df_model["MoA_category"] = y

counts = df_model["MoA_category"].value_counts()
valid_classes = counts[counts >= 3].index

df_filtered = df_model[df_model["MoA_category"].isin(valid_classes)].copy()

# Update X and y AFTER filtering
X = df_filtered.drop(columns=["MoA_category"])
y = df_filtered["MoA_category"]

print("After filtering:", X.shape, y.shape)

import numpy as np

# Replace infinity values
X = X.replace([np.inf, -np.inf], np.nan)

# Convert to float32 to catch values too large for this dtype and re-handle potential new NaNs
X = X.astype(np.float32)

# Replace any new inf values that might have been created due to float32 conversion overflow
X = X.replace([np.inf, -np.inf], np.nan)

# Fill missing values

X = X.fillna(X.median())

# Remove Extremely Large Descriptors
# Some descriptors can reach 10¹² or more, causing overflow.

X.describe().T.sort_values("max", ascending=False).head(10)

# A safe solution is to clip values.

X = X.clip(-1e6, 1e6)

# A safe solution is to clip values.

X = X.astype(np.float64)

# check missing, infinity and large values

import numpy as np

print("NaN values:", np.isnan(X).sum().sum())
print("Infinity values:", np.isinf(X).sum().sum())
print("Max value in dataset:", np.nanmax(X.values))

# Train-Test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# label encoding

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_encoded = le.fit_transform(y_train)

y_test_encoded = le.transform(y_test)

# Feature stability

from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0)

# Fit ONLY on training data
X_train_var = selector.fit_transform(X_train)

# Apply SAME transformation on test data
X_test_var = selector.transform(X_test)

# Convert to DataFrame (optional but good)
selected_columns = X_train.columns[selector.get_support()]

X_train_var = pd.DataFrame(X_train_var, columns=selected_columns)
X_test_var = pd.DataFrame(X_test_var, columns=selected_columns)

print("After removing constant features:", X_train_var.shape[1])

import numpy as np

# Step 1: Compute correlation ONLY on training data
corr_matrix = X_train_var.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]

print("Descriptors to drop:", len(to_drop))

# Step 2: Drop from TRAIN
X_train_corr = X_train_var.drop(columns=to_drop)

# Step 3: Drop SAME columns from TEST
X_test_corr = X_test_var.drop(columns=to_drop)

print("After correlation removal (train):", X_train_corr.shape)
print("After correlation removal (test):", X_test_corr.shape)

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=400,
    random_state=42
)

# ✅ Train ONLY on training data
rf.fit(X_train_corr, y_train)

# Get importance
importance = pd.Series(
    rf.feature_importances_,
    index=X_train_corr.columns
)

# Sort
importance = importance.sort_values(ascending=False)

print("Top features:\n", importance.head(30))

# Select top 50 features
top_features = importance.head(50).index

# Apply SAME features to train and test
X_train_sel = X_train_corr[top_features]
X_test_sel = X_test_corr[top_features]

print("Final train shape:", X_train_sel.shape)
print("Final test shape:", X_test_sel.shape)

# Normalization

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit ONLY on training
X_train_scaled = scaler.fit_transform(X_train_sel)

# Apply to test
X_test_scaled = scaler.transform(X_test_sel)

# Find best K

from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt

bic_scores = []

k_values = [5, 8, 10, 12, 15, 18, 20, 22]

for k in k_values:

    gmm = GaussianMixture(
        n_components=k,
        covariance_type='full',
        random_state=42
    )

    gmm.fit(X_train_scaled)

    bic = gmm.bic(X_train_scaled)

    bic_scores.append(bic)

    print(f"K = {k}, BIC = {bic}")

# Silhouette Score for best K

from sklearn.metrics import silhouette_score

for k in k_values:

    gmm = GaussianMixture(n_components=k, random_state=42)

    labels = gmm.fit_predict(X_train_scaled)

    score = silhouette_score(X_train_scaled, labels)

    print(f"K = {k}, Silhouette Score = {score}")

# Apply GMM

from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=15, random_state=42)

gmm.fit(X_train_scaled)

train_gmm = gmm.predict_proba(X_train_scaled)
test_gmm = gmm.predict_proba(X_test_scaled)

# Combine
X_train_final = np.hstack([X_train_scaled, train_gmm])
X_test_final = np.hstack([X_test_scaled, test_gmm])

# GMM probabilities for TRAIN
gmm_train_probs = gmm.predict_proba(X_train_scaled)

gmm_train_df = pd.DataFrame(
    gmm_train_probs,
    columns=[f"GMM_Cluster_{i}" for i in range(gmm_train_probs.shape[1])]
)

# Convert scaled train to DataFrame
X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=X_train_sel.columns
)

# Concatenate
X_train_enhanced = pd.concat([X_train_scaled_df, gmm_train_df], axis=1)

print("Train shape:", X_train_enhanced.shape)
print(X_train_enhanced.columns[-10:])

# GMM probabilities for TEST
gmm_test_probs = gmm.predict_proba(X_test_scaled)

gmm_test_df = pd.DataFrame(
    gmm_test_probs,
    columns=[f"GMM_Cluster_{i}" for i in range(gmm_test_probs.shape[1])]
)

# Convert scaled test to DataFrame
X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=X_train_sel.columns   # IMPORTANT: use train columns
)

# Concatenate
X_test_enhanced = pd.concat([X_test_scaled_df, gmm_test_df], axis=1)

print("Test shape:", X_test_enhanced.shape)
print(X_test_enhanced.columns[-10:])

cluster_labels = gmm.predict(X_train_scaled)

df_analysis = pd.DataFrame({
    "Cluster": cluster_labels,
    "MoA": y_train.values
})

print(df_analysis.groupby("Cluster")["MoA"].value_counts())

# ---Function for evaluating metrices---

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    classification_report
)

def evaluate_model_full(model_name, y_test, y_pred, y_prob):

    print("\n==============================")
    print("MODEL:", model_name)
    print("==============================")

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    print("\nOverall Confusion Matrix:\n")
    print(cm)

    # Plot confusion matrix
    plt.figure(figsize=(8,6))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=le.classes_,
        yticklabels=le.classes_
    )

    plt.xlabel("Predicted MoA")
    plt.ylabel("Actual MoA")
    plt.title(model_name + " Confusion Matrix")

    plt.show()

    # ------------------------------
    # TP FP FN TN PER CLASS - CALCULATE HERE
    # ------------------------------

    TP = np.diag(cm)

    FP = np.sum(cm, axis=0) - TP

    FN = np.sum(cm, axis=1) - TP

    TN = np.sum(cm) - (TP + FP + FN)

    # --------------------------------
    # PER-MoA CONFUSION MATRICES
    # --------------------------------

    per_moa_cm_list = []

    for i, moa in enumerate(le.classes_):

        tp = TP[i]
        fp = FP[i]
        fn = FN[i]
        tn = TN[i]

        cm_moa = np.array([
            [tp, fn],
            [fp, tn]
        ])

        print("\nConfusion Matrix for MoA:", moa)
        print(cm_moa)

        plt.figure(figsize=(4,4))

        sns.heatmap(
            cm_moa,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=["Predicted "+moa, "Predicted Other"],
            yticklabels=["Actual "+moa, "Actual Other"]
        )

        plt.title(model_name + " - " + moa)

        plt.show()

        per_moa_cm_list.append({
            "MoA": moa,
            "TP": tp,
            "FN": fn,
            "FP": fp,
            "TN": tn
        })

    per_moa_cm_df = pd.DataFrame(per_moa_cm_list)

    per_moa_cm_df.to_excel(model_name+"_per_moa_confusion_matrix.xlsx", index=False)

    # Sensitivity (Recall)
    sensitivity = TP / (TP + FN)

    # Specificity
    specificity = TN / (TN + FP)

    per_class_df = pd.DataFrame({
        "MoA": le.classes_,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "Sensitivity": sensitivity,
        "Specificity": specificity
    })

    print("\nPer-MoA Metrics\n")
    print(per_class_df)

    # Save results
    per_class_df.to_excel(model_name+"_per_MoA_metrics.xlsx", index=False)

    # ------------------------------
    # OVERALL METRICS
    # ------------------------------

    overall_TP = TP.sum()

    overall_FP = FP.sum()

    overall_FN = FN.sum()

    overall_TN = TN.sum()

    overall_sensitivity = overall_TP / (overall_TP + overall_FN)

    overall_specificity = overall_TN / (overall_TN + overall_FP)

    overall_df = pd.DataFrame({
        "Metric": [
            "TP",
            "FP",
            "FN",
            "TN",
            "Sensitivity",
            "Specificity"
        ],
        "Value": [
            overall_TP,
            overall_FP,
            overall_FN,
            overall_TN,
            overall_sensitivity,
            overall_specificity
        ]
    })

    print("\nOverall Metrics\n")
    print(overall_df)

    overall_df.to_excel(model_name+"_overall_metrics.xlsx", index=False)

    # ------------------------------
    # Accuracy
    # ------------------------------

    acc = accuracy_score(y_test, y_pred)

    print("\nAccuracy:", acc)

    # ------------------------------
    # MCC
    # ------------------------------

    mcc = matthews_corrcoef(y_test, y_pred)

    print("MCC:", mcc)

    # ------------------------------
    # ROC AUC (Multiclass)
    # ------------------------------

    from sklearn.preprocessing import label_binarize

    y_test_bin = label_binarize(y_test, classes=np.unique(y_test))

    auc = roc_auc_score(y_test_bin, y_prob, multi_class="ovr")

    print("ROC-AUC:", auc)

    # Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features='sqrt'
)

# ✅ Train on FINAL enhanced features (IMPORTANT)
rf_model.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_rf = rf_model.predict(X_test_enhanced)

# Accuracy
print("Random Forest Accuracy:",
      accuracy_score(y_test_encoded, y_pred_rf))

# Classification Report
print(classification_report(y_test_encoded, y_pred_rf))

# Probabilities (for ROC-AUC)
y_prob_rf = rf_model.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "Random Forest",
    y_test_encoded,
    y_pred_rf,
    y_prob_rf
)

# Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features='sqrt'
)

# ✅ Train on FINAL enhanced features (IMPORTANT)
rf_model.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_rf = rf_model.predict(X_test_enhanced)

# Accuracy
print("Random Forest Accuracy:",
      accuracy_score(y_test_encoded, y_pred_rf))

# Classification Report
print(classification_report(y_test_encoded, y_pred_rf))

# Probabilities (for ROC-AUC)
y_prob_rf = rf_model.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "Random Forest",
    y_test_encoded,
    y_pred_rf,
    y_prob_rf
)


# XGBoost

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    eval_metric="mlogloss"
)

# ✅ Train on FINAL enhanced features
xgb.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_xgb = xgb.predict(X_test_enhanced)

# Accuracy
print("XGBoost Accuracy:",
      accuracy_score(y_test_encoded, y_pred_xgb))

# Classification Report
print(classification_report(y_test_encoded, y_pred_xgb))

# Probabilities (for ROC-AUC)
y_prob_xgb = xgb.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "XGBoost",
    y_test_encoded,
    y_pred_xgb,
    y_prob_xgb
)




k = 18

In [ ]:
import pandas as pd
import re

# ======================================
# 1. Load the Excel file
# ======================================
df = pd.read_excel("final_moa_cateogrization_Peptides+Smiles.xlsx")

# ======================================
# 2. Remove duplicate rows
# ======================================
df = df.drop_duplicates()

# ======================================
# 3. Fill missing MoA with "Unknown"
# ======================================
df["MoA"] = df["MoA"].fillna("Unknown")

# ======================================
# 4. Drop MoA with "Unknown"
# ======================================
df = df[df["MoA"] != "Unknown"]

# ======================================
# 4. Clean MoA text
# ======================================
def clean_moa(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", re.sub(r"[^a-z0-9\s]", " ", text))
    return text.strip()

df["MoA_clean"] = df["MoA"].apply(clean_moa)

# ======================================
# 5. Categorization
# ======================================
def categorize_moa(text):

    text = text.lower()

    if text == "unknown":
        return "Multi-target / Polypharmacology"

    # 1 Membrane / Cell Envelope Disruption
    if any(k in text for k in [
        "membrane",
        "cell membrane",
        "pore",
        "permeability",
        "depolar",
        "lipid",
        "bilayer",
        "detergent",
        "ferriprotoporphyrin",
        "reactive nitro radical",
        "reactive intermediates",
        "membrane depolarization",
        "membrane permeabilization",
        "membrane damage",
        "membrane disrupting",
        "breaks down membrane lipids",
        "destroys cell membranes"
    ]):
        return "Membrane / Cell Envelope Disruption"

    # 2 Cell Wall Biosynthesis Inhibition
    if any(k in text for k in [
        "penicillin-binding protein",
        "pbp",
        "peptidoglycan",
        "cell wall",
        "lipid ii",
        "alanine racemase",
        "beta lactamase",
        "beta-lactamase",
        "penicillin binding protein",
        "fab i",
        "enoyl acyl carrier protein reductase"
    ]):
        return "Cell Wall Biosynthesis Inhibition"

    # 3 Protein Synthesis Inhibition
    if any(k in text for k in [
        "ribosome",
        "70s",
        "30s",
        "50s",
        "translation",
        "elongation factor",
        "trna",
        "protein synthesis",
        "trna synthetase",
        "trna ligase"
    ]):
        return "Protein Synthesis Inhibition"

    # 4 Nucleic Acid Synthesis / Integrity Disruption
    if any(k in text for k in [
        "dna",
        "rna",
        "gyrase",
        "topoisomerase",
        "polymerase",
        "replication",
        "intercalat",
        "dna binding",
        "dna disrupting",
        "dna inhibitor",
        "ribonucleotide reductase"
    ]):
        return "Nucleic Acid Synthesis / Integrity Disruption"

    # 5 Metabolic Pathway Inhibition
    if any(k in text for k in [
        "dihydrofolate reductase",
        "dihydropteroate synthase",
        "folate",
        "biosynthesis",
        "metabolism",
        "amidophosphoribosyltransferase",
        "oxidoreductase",
        "thymidylate synthase",
        "inosine monophosphate dehydrogenase",
        "dxr",
        "reductase",
        "synthase",
        "dehydrogenase",
        "cyclooxygenase",
        "carbonic anhydrase",
        "monoamine oxidase",
        "hmg coa reductase",
        "xanthine dehydrogenase",
        "caspase",
        "matrix metalloproteinase",
        "reductoisomerase",
        "dioxp reductoisomerase",
        "dxr",
        "dipeptidase",
        "diacylglycerol o acyltransferase",
        "dgat",
        "glucosyltransferase",
        "ceramide glucosyltransferase"
    ]):
        return "Metabolic Pathway Inhibition"

    # 6 Redox / Oxidative Stress Induction
    if any(k in text for k in [
        "oxidative",
        "redox",
        "ros",
        "reactive",
        "nitro",
        "oxidase",
        "reducing agent",
        "antioxidant",
        "increases ros",
        "chelating",
        "copper chelating",
        "iron chelating",
        "aluminium chelating",
        "glutathione",
        "glutathione precursor",
        "redox regulator"
    ]):
        return "Redox / Oxidative Stress Induction"

    # 7 Regulatory / Signalling Interference
    if any(k in text for k in [
        "quorum",
        "two component",
        "regulation",
        "signalling",
        "stress response",
        "transcription",
        "toll like receptor",
        "nf kb",
        "mtor",
        "map kinase",
        "cyclin dependent kinase",
        "biofilm related genes",
        "guanylate cyclase",
        "cyclase activator",
        "cyclase inhibitor",
        "cap binding complex",
        "signal transduction modulator",
        "arca",
        "arcb",
        "arcc",
        "arcd",
        "arc gene",
        "arginine deiminase pathway"
    ]):
        return "Regulatory / Signalling Interference"

    # 8 Biofilm Matrix Disruption
    if any(k in text for k in [
        "biofilm",
        "edna",
        "nuclease"
    ]):
        return "Biofilm Matrix Disruption"

    # 9 Antifungal Sterol Biosynthesis
    if any(k in text for k in [
        "cytochrome p450",
        "cyp51",
        "lanosterol",
        "ergosterol",
        "sterol biosynthesis",
        "squalene monooxygenase"
    ]):
        return "Antifungal Sterol Biosynthesis Inhibitors"

    # 10 Antiviral agents
    if any(k in text for k in [
        "hiv",
        "herpesvirus",
        "influenza",
        "hepatitis c",
        "reverse transcriptase",
        "viral protease",
        "viral capsid",
        "neuraminidase",
        "viral polymerase",
        "viral", "hepatitis", "capsid", "viral kinase", "helicase", "primase",
        "hiv protease",
        "hiv 1 protease",
        "viral protease",
        "pul97",
        "viral phosphotransferase",
        "human immunodeficiency virus type 1 protease"
    ]):
        return "Antiviral agents"

    # 11 Affecting Human Physiology
    if any(k in text for k in [
        "adrenergic receptor",
        "dopamine receptor",
        "serotonin transporter",
        "histamine receptor",
        "muscarinic receptor",
        "angiotensin receptor",
        "opioid receptor",
        "cannabinoid receptor",
        "vitamin d receptor",
        "androgen receptor",
        "estrogen receptor",
        "progesterone receptor",
        "glucocorticoid receptor",
        "receptor",
        "agonist",
        "antagonist",
        "partial agonist",
        "inverse agonist",
        "allosteric modulator",
        "allosteric antagonist",
        "positive modulator",
        "negative modulator",
        "retinoid receptor",
        "retinoic acid receptor",
        "retinoid x receptor",
        "adenosine receptor",
        "dopamine receptor",
        "serotonin receptor",
        "histamine receptor",
        "angiotensin receptor",
        "muscarinic receptor",
        "nicotinic receptor",
        "cannabinoid receptor",
        "opioid receptor",
        "purinergic receptor",
        "vanilloid receptor",
        "bile acid receptor",
        "fxr receptor",
        "peroxisome proliferator activated receptor",
        "ppar",
        "glucocorticoid receptor",
        "mineralocorticoid receptor",
        "estrogen receptor",
        "androgen receptor",
        "progesterone receptor",
        "vitamin d receptor",
        "chemokine receptor",
        "cxcr",
        "s1p receptor",
        "sphingosine receptor",
        "nmda receptor",
        "gaba receptor",
        "acetylcholine receptor",

        "ion channel",
        "channel blocker",
        "channel opener",
        "channel modulator",
        "voltage gated",
        "voltage activated",
        "calcium channel",
        "sodium channel",
        "potassium channel",
        "chloride channel",
        "anion channel",
        "herg",
        "enac",
        "ryanodine receptor",
        "l type calcium channel",
        "inwardly rectifying potassium channel",
        "calcium activated potassium channel",

        "acetylcholinesterase",
        "cholinesterase",
        "dopamine transporter",
        "serotonin transporter",
        "norepinephrine transporter",
        "gaba transporter",
        "synaptic vesicle transporter",
        "vesicular amine transporter",
        "sv2a",
        "neurotransmitter reuptake",
        "bcr abl",
        "abl kinase",
        "fusion protein inhibitor"

        "angiotensin-converting enzyme",
        "angiotensin converting enzyme",
        "ace inhibitor",
        "lipoxygenase",
        "cyclooxygenase",
        "phosphodiesterase",
        "pde inhibitor",
        "neprilysin",
        "lipase",
        "glucosidase",
        "decarboxylase",
        "monoamine oxidase",
        "thrombin",
        "farnesyltransferase",
        "kinase",
        "phosphatase",
        "ubiquitin ligase",
        "aurora kinase",
        "pdgfr",
        "pdk1",
        "ikk",

        "transporter",
        "solute carrier",
        "slc",
        "cotransporter",
        "sodium chloride cotransporter",
        "sodium potassium chloride cotransporter",
        "npc1l1",
        "amine transporter",
        "renal transporter",

        "insulin receptor",
        "hormone receptor",
        "glucose metabolism",
        "lipid metabolism",
        "fatty acid metabolism",
        "ppar alpha",
        "ppar gamma",
        "ppar delta",
        "metabolic regulator",

        "growth factor receptor",
        "egfr",
        "map kinase",
        "mtor",
        "nf kb",
        "signal transduction",
        "transcription factor",
        "cell signaling",
        "cytokine receptor",

        "tubulin",
        "microtubule",
        "cytoskeleton",
        "spindle assembly",
        "microtubule stabilizer",
        "microtubule inhibitor"
        "potassium transporting atpase",
        "npc1l1",
        "niemmann-pick c1",
        "niemann pick c1",
        "synaptic vesicle glycoprotein",
        "sv2a",
        "sv2a modulator",
        "laxative",
        "potassium-transporting atpase inhibitor",
        "potassium transporting atpase",
        "potassium atpase",
        "synaptic vesicle glycoprotein 2a",
        "cholesterol absorption inhibitor"



    ]):
        return "Affecting Human Physiology"

    # 12 Multi-target
    if any(k in text for k in [
        "multi target",
        "multiple",
        "polypharmacology",
        "pleiotropic"
    ]):
        return "Multi-target / Polypharmacology"

    return "Others"


# ======================================
# 6. Apply categorization
# ======================================
df["MoA_category"] = df["MoA_clean"].apply(categorize_moa)

# ======================================
# 7. Keep only required columns
# ======================================
df_final = df[["Name", "SMILES", "MoA_category"]]

# ======================================
# 8. Rename column
# ======================================
df_final = df_final.rename(columns={"MoA_category": "MoA"})

# ======================================
# 9. Check distribution
# ======================================
print(df["MoA_category"].value_counts())

# ======================================
# 10. Save final file
# ======================================
df_final.to_excel("final_moa_categorized.xlsx", index=False)

print(df[df["MoA_category"]=="Others"]["MoA"].unique())

# ======================================
# 10. Save categorized data into multiple sheets
# ======================================

with pd.ExcelWriter("final_moa_categorized.xlsx", engine="xlsxwriter") as writer:

    # Main categorized dataset
    df_final.to_excel(writer, sheet_name="All_Data_PA", index=False)

    # Human physiology targets
    human_targets = df[df["MoA_category"] == "Affecting Human Physiology"]
    human_targets.to_excel(writer, sheet_name="Human_Physiology_Targets", index=False)

    # Antiviral agents
    antiviral_targets = df[df["MoA_category"] == "Antiviral agents"]
    antiviral_targets.to_excel(writer, sheet_name="Antiviral_Targets", index=False)

    # Antifungal agents
    antifungal_targets = df[df["MoA_category"] == "Antifungal Sterol Biosynthesis Inhibitors"]
    antifungal_targets.to_excel(writer, sheet_name="Antifungal_Targets", index=False)

    # Others
    Others = df[df["MoA_category"] == "Others"]
    Others.to_excel(writer, sheet_name="Others", index=False)

    # ======================================
    # Direct Antibacterial (remove unwanted categories)
    # ======================================
    direct_antibacterial = df[~df["MoA_category"].isin([
        "Affecting Human Physiology",
        "Antiviral agents",
        "Antifungal Sterol Biosynthesis Inhibitors",
        "Others"
    ])]

    direct_antibacterial.to_excel(writer, sheet_name="Direct_Antibacterial", index=False)

print("Excel file created with multiple sheets including Direct_Antibacterial.")

import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

# Load dataset from the correct file and sheet
df = pd.read_excel("final_moa_categorized.xlsx", sheet_name="Direct_Antibacterial")

# Get list of all RDKit descriptors
descriptor_names = [desc[0] for desc in Descriptors.descList]

def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return [None]*len(descriptor_names)

    descriptor_values = []

    for name, function in Descriptors.descList:
        try:
            value = function(mol)
        except:
            value = None
        descriptor_values.append(value)

    return descriptor_values


# Apply descriptor calculation
descriptor_data = df["SMILES"].apply(calculate_descriptors)

# Convert to dataframe
X = pd.DataFrame(descriptor_data.tolist(), columns=descriptor_names)

# Target variable
y = df["MoA_category"]

# Combine if needed
final_df = pd.concat([X, y], axis=1)

# Save descriptor dataset
final_df.to_excel("MoA_rdkit_all_descriptors_calculated.xlsx", index=False)

print("Total descriptors extracted:", len(descriptor_names))
print("Dataset shape:", final_df.shape)


# Load Descriptor data

import pandas as pd

df = pd.read_excel("MoA_rdkit_all_descriptors_calculated.xlsx")

print("Dataset shape:", df.shape)

df.head()

# seperate descriptors and label(moa)

X = df.drop(columns=["MoA_category"])
y = df["MoA_category"]

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

# -------------------------------
#  FILTER RARE CLASSES HERE
# -------------------------------

df_model = X.copy()
df_model["MoA_category"] = y

counts = df_model["MoA_category"].value_counts()
valid_classes = counts[counts >= 3].index

df_filtered = df_model[df_model["MoA_category"].isin(valid_classes)].copy()

# Update X and y AFTER filtering
X = df_filtered.drop(columns=["MoA_category"])
y = df_filtered["MoA_category"]

print("After filtering:", X.shape, y.shape)

import numpy as np

# Replace infinity values
X = X.replace([np.inf, -np.inf], np.nan)

# Convert to float32 to catch values too large for this dtype and re-handle potential new NaNs
X = X.astype(np.float32)

# Replace any new inf values that might have been created due to float32 conversion overflow
X = X.replace([np.inf, -np.inf], np.nan)

# Fill missing values

X = X.fillna(X.median())

# Remove Extremely Large Descriptors
# Some descriptors can reach 10¹² or more, causing overflow.

X.describe().T.sort_values("max", ascending=False).head(10)

# A safe solution is to clip values.

X = X.clip(-1e6, 1e6)

# A safe solution is to clip values.

X = X.astype(np.float64)

# check missing, infinity and large values

import numpy as np

print("NaN values:", np.isnan(X).sum().sum())
print("Infinity values:", np.isinf(X).sum().sum())
print("Max value in dataset:", np.nanmax(X.values))

# Train-Test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# label encoding

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_encoded = le.fit_transform(y_train)

y_test_encoded = le.transform(y_test)

# Feature stability

from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0)

# Fit ONLY on training data
X_train_var = selector.fit_transform(X_train)

# Apply SAME transformation on test data
X_test_var = selector.transform(X_test)

# Convert to DataFrame (optional but good)
selected_columns = X_train.columns[selector.get_support()]

X_train_var = pd.DataFrame(X_train_var, columns=selected_columns)
X_test_var = pd.DataFrame(X_test_var, columns=selected_columns)

print("After removing constant features:", X_train_var.shape[1])

import numpy as np

# Step 1: Compute correlation ONLY on training data
corr_matrix = X_train_var.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]

print("Descriptors to drop:", len(to_drop))

# Step 2: Drop from TRAIN
X_train_corr = X_train_var.drop(columns=to_drop)

# Step 3: Drop SAME columns from TEST
X_test_corr = X_test_var.drop(columns=to_drop)

print("After correlation removal (train):", X_train_corr.shape)
print("After correlation removal (test):", X_test_corr.shape)

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=400,
    random_state=42
)

# ✅ Train ONLY on training data
rf.fit(X_train_corr, y_train)

# Get importance
importance = pd.Series(
    rf.feature_importances_,
    index=X_train_corr.columns
)

# Sort
importance = importance.sort_values(ascending=False)

print("Top features:\n", importance.head(30))

# Select top 50 features
top_features = importance.head(50).index

# Apply SAME features to train and test
X_train_sel = X_train_corr[top_features]
X_test_sel = X_test_corr[top_features]

print("Final train shape:", X_train_sel.shape)
print("Final test shape:", X_test_sel.shape)

# Normalization

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit ONLY on training
X_train_scaled = scaler.fit_transform(X_train_sel)

# Apply to test
X_test_scaled = scaler.transform(X_test_sel)

# Find best K

from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt

bic_scores = []

k_values = [5, 8, 10, 12, 15, 18, 20, 22]

for k in k_values:

    gmm = GaussianMixture(
        n_components=k,
        covariance_type='full',
        random_state=42
    )

    gmm.fit(X_train_scaled)

    bic = gmm.bic(X_train_scaled)

    bic_scores.append(bic)

    print(f"K = {k}, BIC = {bic}")

# Silhouette Score for best K

from sklearn.metrics import silhouette_score

for k in k_values:

    gmm = GaussianMixture(n_components=k, random_state=42)

    labels = gmm.fit_predict(X_train_scaled)

    score = silhouette_score(X_train_scaled, labels)

    print(f"K = {k}, Silhouette Score = {score}")

# Apply GMM

from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=18, random_state=42)

gmm.fit(X_train_scaled)

train_gmm = gmm.predict_proba(X_train_scaled)
test_gmm = gmm.predict_proba(X_test_scaled)

# Combine
X_train_final = np.hstack([X_train_scaled, train_gmm])
X_test_final = np.hstack([X_test_scaled, test_gmm])

# GMM probabilities for TRAIN
gmm_train_probs = gmm.predict_proba(X_train_scaled)

gmm_train_df = pd.DataFrame(
    gmm_train_probs,
    columns=[f"GMM_Cluster_{i}" for i in range(gmm_train_probs.shape[1])]
)

# Convert scaled train to DataFrame
X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=X_train_sel.columns
)

# Concatenate
X_train_enhanced = pd.concat([X_train_scaled_df, gmm_train_df], axis=1)

print("Train shape:", X_train_enhanced.shape)
print(X_train_enhanced.columns[-10:])

# GMM probabilities for TEST
gmm_test_probs = gmm.predict_proba(X_test_scaled)

gmm_test_df = pd.DataFrame(
    gmm_test_probs,
    columns=[f"GMM_Cluster_{i}" for i in range(gmm_test_probs.shape[1])]
)

# Convert scaled test to DataFrame
X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=X_train_sel.columns   # IMPORTANT: use train columns
)

# Concatenate
X_test_enhanced = pd.concat([X_test_scaled_df, gmm_test_df], axis=1)

print("Test shape:", X_test_enhanced.shape)
print(X_test_enhanced.columns[-10:])

cluster_labels = gmm.predict(X_train_scaled)

df_analysis = pd.DataFrame({
    "Cluster": cluster_labels,
    "MoA": y_train.values
})

print(df_analysis.groupby("Cluster")["MoA"].value_counts())

# ---Function for evaluating metrices---

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    classification_report
)

def evaluate_model_full(model_name, y_test, y_pred, y_prob):

    print("\n==============================")
    print("MODEL:", model_name)
    print("==============================")

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    print("\nOverall Confusion Matrix:\n")
    print(cm)

    # Plot confusion matrix
    plt.figure(figsize=(8,6))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=le.classes_,
        yticklabels=le.classes_
    )

    plt.xlabel("Predicted MoA")
    plt.ylabel("Actual MoA")
    plt.title(model_name + " Confusion Matrix")

    plt.show()

    # ------------------------------
    # TP FP FN TN PER CLASS - CALCULATE HERE
    # ------------------------------

    TP = np.diag(cm)

    FP = np.sum(cm, axis=0) - TP

    FN = np.sum(cm, axis=1) - TP

    TN = np.sum(cm) - (TP + FP + FN)

    # --------------------------------
    # PER-MoA CONFUSION MATRICES
    # --------------------------------

    per_moa_cm_list = []

    for i, moa in enumerate(le.classes_):

        tp = TP[i]
        fp = FP[i]
        fn = FN[i]
        tn = TN[i]

        cm_moa = np.array([
            [tp, fn],
            [fp, tn]
        ])

        print("\nConfusion Matrix for MoA:", moa)
        print(cm_moa)

        plt.figure(figsize=(4,4))

        sns.heatmap(
            cm_moa,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=["Predicted "+moa, "Predicted Other"],
            yticklabels=["Actual "+moa, "Actual Other"]
        )

        plt.title(model_name + " - " + moa)

        plt.show()

        per_moa_cm_list.append({
            "MoA": moa,
            "TP": tp,
            "FN": fn,
            "FP": fp,
            "TN": tn
        })

    per_moa_cm_df = pd.DataFrame(per_moa_cm_list)

    per_moa_cm_df.to_excel(model_name+"_per_moa_confusion_matrix.xlsx", index=False)

    # Sensitivity (Recall)
    sensitivity = TP / (TP + FN)

    # Specificity
    specificity = TN / (TN + FP)

    per_class_df = pd.DataFrame({
        "MoA": le.classes_,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "Sensitivity": sensitivity,
        "Specificity": specificity
    })

    print("\nPer-MoA Metrics\n")
    print(per_class_df)

    # Save results
    per_class_df.to_excel(model_name+"_per_MoA_metrics.xlsx", index=False)

    # ------------------------------
    # OVERALL METRICS
    # ------------------------------

    overall_TP = TP.sum()

    overall_FP = FP.sum()

    overall_FN = FN.sum()

    overall_TN = TN.sum()

    overall_sensitivity = overall_TP / (overall_TP + overall_FN)

    overall_specificity = overall_TN / (overall_TN + overall_FP)

    overall_df = pd.DataFrame({
        "Metric": [
            "TP",
            "FP",
            "FN",
            "TN",
            "Sensitivity",
            "Specificity"
        ],
        "Value": [
            overall_TP,
            overall_FP,
            overall_FN,
            overall_TN,
            overall_sensitivity,
            overall_specificity
        ]
    })

    print("\nOverall Metrics\n")
    print(overall_df)

    overall_df.to_excel(model_name+"_overall_metrics.xlsx", index=False)

    # ------------------------------
    # Accuracy
    # ------------------------------

    acc = accuracy_score(y_test, y_pred)

    print("\nAccuracy:", acc)

    # ------------------------------
    # MCC
    # ------------------------------

    mcc = matthews_corrcoef(y_test, y_pred)

    print("MCC:", mcc)

    # ------------------------------
    # ROC AUC (Multiclass)
    # ------------------------------

    from sklearn.preprocessing import label_binarize

    y_test_bin = label_binarize(y_test, classes=np.unique(y_test))

    auc = roc_auc_score(y_test_bin, y_prob, multi_class="ovr")

    print("ROC-AUC:", auc)

    # Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features='sqrt'
)

# ✅ Train on FINAL enhanced features (IMPORTANT)
rf_model.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_rf = rf_model.predict(X_test_enhanced)

# Accuracy
print("Random Forest Accuracy:",
      accuracy_score(y_test_encoded, y_pred_rf))

# Classification Report
print(classification_report(y_test_encoded, y_pred_rf))

# Probabilities (for ROC-AUC)
y_prob_rf = rf_model.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "Random Forest",
    y_test_encoded,
    y_pred_rf,
    y_prob_rf
)

# Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features='sqrt'
)

# ✅ Train on FINAL enhanced features (IMPORTANT)
rf_model.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_rf = rf_model.predict(X_test_enhanced)

# Accuracy
print("Random Forest Accuracy:",
      accuracy_score(y_test_encoded, y_pred_rf))

# Classification Report
print(classification_report(y_test_encoded, y_pred_rf))

# Probabilities (for ROC-AUC)
y_prob_rf = rf_model.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "Random Forest",
    y_test_encoded,
    y_pred_rf,
    y_prob_rf
)


# XGBoost

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    eval_metric="mlogloss"
)

# ✅ Train on FINAL enhanced features
xgb.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_xgb = xgb.predict(X_test_enhanced)

# Accuracy
print("XGBoost Accuracy:",
      accuracy_score(y_test_encoded, y_pred_xgb))

# Classification Report
print(classification_report(y_test_encoded, y_pred_xgb))

# Probabilities (for ROC-AUC)
y_prob_xgb = xgb.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "XGBoost",
    y_test_encoded,
    y_pred_xgb,
    y_prob_xgb
)




k = 20

In [ ]:
import pandas as pd
import re

# ======================================
# 1. Load the Excel file
# ======================================
df = pd.read_excel("final_moa_cateogrization_Peptides+Smiles.xlsx")

# ======================================
# 2. Remove duplicate rows
# ======================================
df = df.drop_duplicates()

# ======================================
# 3. Fill missing MoA with "Unknown"
# ======================================
df["MoA"] = df["MoA"].fillna("Unknown")

# ======================================
# 4. Drop MoA with "Unknown"
# ======================================
df = df[df["MoA"] != "Unknown"]

# ======================================
# 4. Clean MoA text
# ======================================
def clean_moa(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", re.sub(r"[^a-z0-9\s]", " ", text))
    return text.strip()

df["MoA_clean"] = df["MoA"].apply(clean_moa)

# ======================================
# 5. Categorization
# ======================================
def categorize_moa(text):

    text = text.lower()

    if text == "unknown":
        return "Multi-target / Polypharmacology"

    # 1 Membrane / Cell Envelope Disruption
    if any(k in text for k in [
        "membrane",
        "cell membrane",
        "pore",
        "permeability",
        "depolar",
        "lipid",
        "bilayer",
        "detergent",
        "ferriprotoporphyrin",
        "reactive nitro radical",
        "reactive intermediates",
        "membrane depolarization",
        "membrane permeabilization",
        "membrane damage",
        "membrane disrupting",
        "breaks down membrane lipids",
        "destroys cell membranes"
    ]):
        return "Membrane / Cell Envelope Disruption"

    # 2 Cell Wall Biosynthesis Inhibition
    if any(k in text for k in [
        "penicillin-binding protein",
        "pbp",
        "peptidoglycan",
        "cell wall",
        "lipid ii",
        "alanine racemase",
        "beta lactamase",
        "beta-lactamase",
        "penicillin binding protein",
        "fab i",
        "enoyl acyl carrier protein reductase"
    ]):
        return "Cell Wall Biosynthesis Inhibition"

    # 3 Protein Synthesis Inhibition
    if any(k in text for k in [
        "ribosome",
        "70s",
        "30s",
        "50s",
        "translation",
        "elongation factor",
        "trna",
        "protein synthesis",
        "trna synthetase",
        "trna ligase"
    ]):
        return "Protein Synthesis Inhibition"

    # 4 Nucleic Acid Synthesis / Integrity Disruption
    if any(k in text for k in [
        "dna",
        "rna",
        "gyrase",
        "topoisomerase",
        "polymerase",
        "replication",
        "intercalat",
        "dna binding",
        "dna disrupting",
        "dna inhibitor",
        "ribonucleotide reductase"
    ]):
        return "Nucleic Acid Synthesis / Integrity Disruption"

    # 5 Metabolic Pathway Inhibition
    if any(k in text for k in [
        "dihydrofolate reductase",
        "dihydropteroate synthase",
        "folate",
        "biosynthesis",
        "metabolism",
        "amidophosphoribosyltransferase",
        "oxidoreductase",
        "thymidylate synthase",
        "inosine monophosphate dehydrogenase",
        "dxr",
        "reductase",
        "synthase",
        "dehydrogenase",
        "cyclooxygenase",
        "carbonic anhydrase",
        "monoamine oxidase",
        "hmg coa reductase",
        "xanthine dehydrogenase",
        "caspase",
        "matrix metalloproteinase",
        "reductoisomerase",
        "dioxp reductoisomerase",
        "dxr",
        "dipeptidase",
        "diacylglycerol o acyltransferase",
        "dgat",
        "glucosyltransferase",
        "ceramide glucosyltransferase"
    ]):
        return "Metabolic Pathway Inhibition"

    # 6 Redox / Oxidative Stress Induction
    if any(k in text for k in [
        "oxidative",
        "redox",
        "ros",
        "reactive",
        "nitro",
        "oxidase",
        "reducing agent",
        "antioxidant",
        "increases ros",
        "chelating",
        "copper chelating",
        "iron chelating",
        "aluminium chelating",
        "glutathione",
        "glutathione precursor",
        "redox regulator"
    ]):
        return "Redox / Oxidative Stress Induction"

    # 7 Regulatory / Signalling Interference
    if any(k in text for k in [
        "quorum",
        "two component",
        "regulation",
        "signalling",
        "stress response",
        "transcription",
        "toll like receptor",
        "nf kb",
        "mtor",
        "map kinase",
        "cyclin dependent kinase",
        "biofilm related genes",
        "guanylate cyclase",
        "cyclase activator",
        "cyclase inhibitor",
        "cap binding complex",
        "signal transduction modulator",
        "arca",
        "arcb",
        "arcc",
        "arcd",
        "arc gene",
        "arginine deiminase pathway"
    ]):
        return "Regulatory / Signalling Interference"

    # 8 Biofilm Matrix Disruption
    if any(k in text for k in [
        "biofilm",
        "edna",
        "nuclease"
    ]):
        return "Biofilm Matrix Disruption"

    # 9 Antifungal Sterol Biosynthesis
    if any(k in text for k in [
        "cytochrome p450",
        "cyp51",
        "lanosterol",
        "ergosterol",
        "sterol biosynthesis",
        "squalene monooxygenase"
    ]):
        return "Antifungal Sterol Biosynthesis Inhibitors"

    # 10 Antiviral agents
    if any(k in text for k in [
        "hiv",
        "herpesvirus",
        "influenza",
        "hepatitis c",
        "reverse transcriptase",
        "viral protease",
        "viral capsid",
        "neuraminidase",
        "viral polymerase",
        "viral", "hepatitis", "capsid", "viral kinase", "helicase", "primase",
        "hiv protease",
        "hiv 1 protease",
        "viral protease",
        "pul97",
        "viral phosphotransferase",
        "human immunodeficiency virus type 1 protease"
    ]):
        return "Antiviral agents"

    # 11 Affecting Human Physiology
    if any(k in text for k in [
        "adrenergic receptor",
        "dopamine receptor",
        "serotonin transporter",
        "histamine receptor",
        "muscarinic receptor",
        "angiotensin receptor",
        "opioid receptor",
        "cannabinoid receptor",
        "vitamin d receptor",
        "androgen receptor",
        "estrogen receptor",
        "progesterone receptor",
        "glucocorticoid receptor",
        "receptor",
        "agonist",
        "antagonist",
        "partial agonist",
        "inverse agonist",
        "allosteric modulator",
        "allosteric antagonist",
        "positive modulator",
        "negative modulator",
        "retinoid receptor",
        "retinoic acid receptor",
        "retinoid x receptor",
        "adenosine receptor",
        "dopamine receptor",
        "serotonin receptor",
        "histamine receptor",
        "angiotensin receptor",
        "muscarinic receptor",
        "nicotinic receptor",
        "cannabinoid receptor",
        "opioid receptor",
        "purinergic receptor",
        "vanilloid receptor",
        "bile acid receptor",
        "fxr receptor",
        "peroxisome proliferator activated receptor",
        "ppar",
        "glucocorticoid receptor",
        "mineralocorticoid receptor",
        "estrogen receptor",
        "androgen receptor",
        "progesterone receptor",
        "vitamin d receptor",
        "chemokine receptor",
        "cxcr",
        "s1p receptor",
        "sphingosine receptor",
        "nmda receptor",
        "gaba receptor",
        "acetylcholine receptor",

        "ion channel",
        "channel blocker",
        "channel opener",
        "channel modulator",
        "voltage gated",
        "voltage activated",
        "calcium channel",
        "sodium channel",
        "potassium channel",
        "chloride channel",
        "anion channel",
        "herg",
        "enac",
        "ryanodine receptor",
        "l type calcium channel",
        "inwardly rectifying potassium channel",
        "calcium activated potassium channel",

        "acetylcholinesterase",
        "cholinesterase",
        "dopamine transporter",
        "serotonin transporter",
        "norepinephrine transporter",
        "gaba transporter",
        "synaptic vesicle transporter",
        "vesicular amine transporter",
        "sv2a",
        "neurotransmitter reuptake",
        "bcr abl",
        "abl kinase",
        "fusion protein inhibitor"

        "angiotensin-converting enzyme",
        "angiotensin converting enzyme",
        "ace inhibitor",
        "lipoxygenase",
        "cyclooxygenase",
        "phosphodiesterase",
        "pde inhibitor",
        "neprilysin",
        "lipase",
        "glucosidase",
        "decarboxylase",
        "monoamine oxidase",
        "thrombin",
        "farnesyltransferase",
        "kinase",
        "phosphatase",
        "ubiquitin ligase",
        "aurora kinase",
        "pdgfr",
        "pdk1",
        "ikk",

        "transporter",
        "solute carrier",
        "slc",
        "cotransporter",
        "sodium chloride cotransporter",
        "sodium potassium chloride cotransporter",
        "npc1l1",
        "amine transporter",
        "renal transporter",

        "insulin receptor",
        "hormone receptor",
        "glucose metabolism",
        "lipid metabolism",
        "fatty acid metabolism",
        "ppar alpha",
        "ppar gamma",
        "ppar delta",
        "metabolic regulator",

        "growth factor receptor",
        "egfr",
        "map kinase",
        "mtor",
        "nf kb",
        "signal transduction",
        "transcription factor",
        "cell signaling",
        "cytokine receptor",

        "tubulin",
        "microtubule",
        "cytoskeleton",
        "spindle assembly",
        "microtubule stabilizer",
        "microtubule inhibitor"
        "potassium transporting atpase",
        "npc1l1",
        "niemmann-pick c1",
        "niemann pick c1",
        "synaptic vesicle glycoprotein",
        "sv2a",
        "sv2a modulator",
        "laxative",
        "potassium-transporting atpase inhibitor",
        "potassium transporting atpase",
        "potassium atpase",
        "synaptic vesicle glycoprotein 2a",
        "cholesterol absorption inhibitor"



    ]):
        return "Affecting Human Physiology"

    # 12 Multi-target
    if any(k in text for k in [
        "multi target",
        "multiple",
        "polypharmacology",
        "pleiotropic"
    ]):
        return "Multi-target / Polypharmacology"

    return "Others"


# ======================================
# 6. Apply categorization
# ======================================
df["MoA_category"] = df["MoA_clean"].apply(categorize_moa)

# ======================================
# 7. Keep only required columns
# ======================================
df_final = df[["Name", "SMILES", "MoA_category"]]

# ======================================
# 8. Rename column
# ======================================
df_final = df_final.rename(columns={"MoA_category": "MoA"})

# ======================================
# 9. Check distribution
# ======================================
print(df["MoA_category"].value_counts())

# ======================================
# 10. Save final file
# ======================================
df_final.to_excel("final_moa_categorized.xlsx", index=False)

print(df[df["MoA_category"]=="Others"]["MoA"].unique())

# ======================================
# 10. Save categorized data into multiple sheets
# ======================================

with pd.ExcelWriter("final_moa_categorized.xlsx", engine="xlsxwriter") as writer:

    # Main categorized dataset
    df_final.to_excel(writer, sheet_name="All_Data_PA", index=False)

    # Human physiology targets
    human_targets = df[df["MoA_category"] == "Affecting Human Physiology"]
    human_targets.to_excel(writer, sheet_name="Human_Physiology_Targets", index=False)

    # Antiviral agents
    antiviral_targets = df[df["MoA_category"] == "Antiviral agents"]
    antiviral_targets.to_excel(writer, sheet_name="Antiviral_Targets", index=False)

    # Antifungal agents
    antifungal_targets = df[df["MoA_category"] == "Antifungal Sterol Biosynthesis Inhibitors"]
    antifungal_targets.to_excel(writer, sheet_name="Antifungal_Targets", index=False)

    # Others
    Others = df[df["MoA_category"] == "Others"]
    Others.to_excel(writer, sheet_name="Others", index=False)

    # ======================================
    # Direct Antibacterial (remove unwanted categories)
    # ======================================
    direct_antibacterial = df[~df["MoA_category"].isin([
        "Affecting Human Physiology",
        "Antiviral agents",
        "Antifungal Sterol Biosynthesis Inhibitors",
        "Others"
    ])]

    direct_antibacterial.to_excel(writer, sheet_name="Direct_Antibacterial", index=False)

print("Excel file created with multiple sheets including Direct_Antibacterial.")

import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

# Load dataset from the correct file and sheet
df = pd.read_excel("final_moa_categorized.xlsx", sheet_name="Direct_Antibacterial")

# Get list of all RDKit descriptors
descriptor_names = [desc[0] for desc in Descriptors.descList]

def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return [None]*len(descriptor_names)

    descriptor_values = []

    for name, function in Descriptors.descList:
        try:
            value = function(mol)
        except:
            value = None
        descriptor_values.append(value)

    return descriptor_values


# Apply descriptor calculation
descriptor_data = df["SMILES"].apply(calculate_descriptors)

# Convert to dataframe
X = pd.DataFrame(descriptor_data.tolist(), columns=descriptor_names)

# Target variable
y = df["MoA_category"]

# Combine if needed
final_df = pd.concat([X, y], axis=1)

# Save descriptor dataset
final_df.to_excel("MoA_rdkit_all_descriptors_calculated.xlsx", index=False)

print("Total descriptors extracted:", len(descriptor_names))
print("Dataset shape:", final_df.shape)


# Load Descriptor data

import pandas as pd

df = pd.read_excel("MoA_rdkit_all_descriptors_calculated.xlsx")

print("Dataset shape:", df.shape)

df.head()

# seperate descriptors and label(moa)

X = df.drop(columns=["MoA_category"])
y = df["MoA_category"]

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

# -------------------------------
#  FILTER RARE CLASSES HERE
# -------------------------------

df_model = X.copy()
df_model["MoA_category"] = y

counts = df_model["MoA_category"].value_counts()
valid_classes = counts[counts >= 3].index

df_filtered = df_model[df_model["MoA_category"].isin(valid_classes)].copy()

# Update X and y AFTER filtering
X = df_filtered.drop(columns=["MoA_category"])
y = df_filtered["MoA_category"]

print("After filtering:", X.shape, y.shape)

import numpy as np

# Replace infinity values
X = X.replace([np.inf, -np.inf], np.nan)

# Convert to float32 to catch values too large for this dtype and re-handle potential new NaNs
X = X.astype(np.float32)

# Replace any new inf values that might have been created due to float32 conversion overflow
X = X.replace([np.inf, -np.inf], np.nan)

# Fill missing values

X = X.fillna(X.median())

# Remove Extremely Large Descriptors
# Some descriptors can reach 10¹² or more, causing overflow.

X.describe().T.sort_values("max", ascending=False).head(10)

# A safe solution is to clip values.

X = X.clip(-1e6, 1e6)

# A safe solution is to clip values.

X = X.astype(np.float64)

# check missing, infinity and large values

import numpy as np

print("NaN values:", np.isnan(X).sum().sum())
print("Infinity values:", np.isinf(X).sum().sum())
print("Max value in dataset:", np.nanmax(X.values))

# Train-Test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# label encoding

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_encoded = le.fit_transform(y_train)

y_test_encoded = le.transform(y_test)

# Feature stability

from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0)

# Fit ONLY on training data
X_train_var = selector.fit_transform(X_train)

# Apply SAME transformation on test data
X_test_var = selector.transform(X_test)

# Convert to DataFrame (optional but good)
selected_columns = X_train.columns[selector.get_support()]

X_train_var = pd.DataFrame(X_train_var, columns=selected_columns)
X_test_var = pd.DataFrame(X_test_var, columns=selected_columns)

print("After removing constant features:", X_train_var.shape[1])

import numpy as np

# Step 1: Compute correlation ONLY on training data
corr_matrix = X_train_var.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]

print("Descriptors to drop:", len(to_drop))

# Step 2: Drop from TRAIN
X_train_corr = X_train_var.drop(columns=to_drop)

# Step 3: Drop SAME columns from TEST
X_test_corr = X_test_var.drop(columns=to_drop)

print("After correlation removal (train):", X_train_corr.shape)
print("After correlation removal (test):", X_test_corr.shape)

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=400,
    random_state=42
)

# ✅ Train ONLY on training data
rf.fit(X_train_corr, y_train)

# Get importance
importance = pd.Series(
    rf.feature_importances_,
    index=X_train_corr.columns
)

# Sort
importance = importance.sort_values(ascending=False)

print("Top features:\n", importance.head(30))

# Select top 50 features
top_features = importance.head(50).index

# Apply SAME features to train and test
X_train_sel = X_train_corr[top_features]
X_test_sel = X_test_corr[top_features]

print("Final train shape:", X_train_sel.shape)
print("Final test shape:", X_test_sel.shape)

# Normalization

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit ONLY on training
X_train_scaled = scaler.fit_transform(X_train_sel)

# Apply to test
X_test_scaled = scaler.transform(X_test_sel)

# Find best K

from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt

bic_scores = []

k_values = [5, 8, 10, 12, 15, 18, 20, 22]

for k in k_values:

    gmm = GaussianMixture(
        n_components=k,
        covariance_type='full',
        random_state=42
    )

    gmm.fit(X_train_scaled)

    bic = gmm.bic(X_train_scaled)

    bic_scores.append(bic)

    print(f"K = {k}, BIC = {bic}")

# Silhouette Score for best K

from sklearn.metrics import silhouette_score

for k in k_values:

    gmm = GaussianMixture(n_components=k, random_state=42)

    labels = gmm.fit_predict(X_train_scaled)

    score = silhouette_score(X_train_scaled, labels)

    print(f"K = {k}, Silhouette Score = {score}")

# Apply GMM

from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=20, random_state=42)

gmm.fit(X_train_scaled)

train_gmm = gmm.predict_proba(X_train_scaled)
test_gmm = gmm.predict_proba(X_test_scaled)

# Combine
X_train_final = np.hstack([X_train_scaled, train_gmm])
X_test_final = np.hstack([X_test_scaled, test_gmm])

# GMM probabilities for TRAIN
gmm_train_probs = gmm.predict_proba(X_train_scaled)

gmm_train_df = pd.DataFrame(
    gmm_train_probs,
    columns=[f"GMM_Cluster_{i}" for i in range(gmm_train_probs.shape[1])]
)

# Convert scaled train to DataFrame
X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=X_train_sel.columns
)

# Concatenate
X_train_enhanced = pd.concat([X_train_scaled_df, gmm_train_df], axis=1)

print("Train shape:", X_train_enhanced.shape)
print(X_train_enhanced.columns[-10:])

# GMM probabilities for TEST
gmm_test_probs = gmm.predict_proba(X_test_scaled)

gmm_test_df = pd.DataFrame(
    gmm_test_probs,
    columns=[f"GMM_Cluster_{i}" for i in range(gmm_test_probs.shape[1])]
)

# Convert scaled test to DataFrame
X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=X_train_sel.columns   # IMPORTANT: use train columns
)

# Concatenate
X_test_enhanced = pd.concat([X_test_scaled_df, gmm_test_df], axis=1)

print("Test shape:", X_test_enhanced.shape)
print(X_test_enhanced.columns[-10:])

cluster_labels = gmm.predict(X_train_scaled)

df_analysis = pd.DataFrame({
    "Cluster": cluster_labels,
    "MoA": y_train.values
})

print(df_analysis.groupby("Cluster")["MoA"].value_counts())

# ---Function for evaluating metrices---

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    classification_report
)

def evaluate_model_full(model_name, y_test, y_pred, y_prob):

    print("\n==============================")
    print("MODEL:", model_name)
    print("==============================")

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    print("\nOverall Confusion Matrix:\n")
    print(cm)

    # Plot confusion matrix
    plt.figure(figsize=(8,6))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=le.classes_,
        yticklabels=le.classes_
    )

    plt.xlabel("Predicted MoA")
    plt.ylabel("Actual MoA")
    plt.title(model_name + " Confusion Matrix")

    plt.show()

    # ------------------------------
    # TP FP FN TN PER CLASS - CALCULATE HERE
    # ------------------------------

    TP = np.diag(cm)

    FP = np.sum(cm, axis=0) - TP

    FN = np.sum(cm, axis=1) - TP

    TN = np.sum(cm) - (TP + FP + FN)

    # --------------------------------
    # PER-MoA CONFUSION MATRICES
    # --------------------------------

    per_moa_cm_list = []

    for i, moa in enumerate(le.classes_):

        tp = TP[i]
        fp = FP[i]
        fn = FN[i]
        tn = TN[i]

        cm_moa = np.array([
            [tp, fn],
            [fp, tn]
        ])

        print("\nConfusion Matrix for MoA:", moa)
        print(cm_moa)

        plt.figure(figsize=(4,4))

        sns.heatmap(
            cm_moa,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=["Predicted "+moa, "Predicted Other"],
            yticklabels=["Actual "+moa, "Actual Other"]
        )

        plt.title(model_name + " - " + moa)

        plt.show()

        per_moa_cm_list.append({
            "MoA": moa,
            "TP": tp,
            "FN": fn,
            "FP": fp,
            "TN": tn
        })

    per_moa_cm_df = pd.DataFrame(per_moa_cm_list)

    per_moa_cm_df.to_excel(model_name+"_per_moa_confusion_matrix.xlsx", index=False)

    # Sensitivity (Recall)
    sensitivity = TP / (TP + FN)

    # Specificity
    specificity = TN / (TN + FP)

    per_class_df = pd.DataFrame({
        "MoA": le.classes_,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "Sensitivity": sensitivity,
        "Specificity": specificity
    })

    print("\nPer-MoA Metrics\n")
    print(per_class_df)

    # Save results
    per_class_df.to_excel(model_name+"_per_MoA_metrics.xlsx", index=False)

    # ------------------------------
    # OVERALL METRICS
    # ------------------------------

    overall_TP = TP.sum()

    overall_FP = FP.sum()

    overall_FN = FN.sum()

    overall_TN = TN.sum()

    overall_sensitivity = overall_TP / (overall_TP + overall_FN)

    overall_specificity = overall_TN / (overall_TN + overall_FP)

    overall_df = pd.DataFrame({
        "Metric": [
            "TP",
            "FP",
            "FN",
            "TN",
            "Sensitivity",
            "Specificity"
        ],
        "Value": [
            overall_TP,
            overall_FP,
            overall_FN,
            overall_TN,
            overall_sensitivity,
            overall_specificity
        ]
    })

    print("\nOverall Metrics\n")
    print(overall_df)

    overall_df.to_excel(model_name+"_overall_metrics.xlsx", index=False)

    # ------------------------------
    # Accuracy
    # ------------------------------

    acc = accuracy_score(y_test, y_pred)

    print("\nAccuracy:", acc)

    # ------------------------------
    # MCC
    # ------------------------------

    mcc = matthews_corrcoef(y_test, y_pred)

    print("MCC:", mcc)

    # ------------------------------
    # ROC AUC (Multiclass)
    # ------------------------------

    from sklearn.preprocessing import label_binarize

    y_test_bin = label_binarize(y_test, classes=np.unique(y_test))

    auc = roc_auc_score(y_test_bin, y_prob, multi_class="ovr")

    print("ROC-AUC:", auc)

    # Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features='sqrt'
)

# ✅ Train on FINAL enhanced features (IMPORTANT)
rf_model.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_rf = rf_model.predict(X_test_enhanced)

# Accuracy
print("Random Forest Accuracy:",
      accuracy_score(y_test_encoded, y_pred_rf))

# Classification Report
print(classification_report(y_test_encoded, y_pred_rf))

# Probabilities (for ROC-AUC)
y_prob_rf = rf_model.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "Random Forest",
    y_test_encoded,
    y_pred_rf,
    y_prob_rf
)

# Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features='sqrt'
)

# ✅ Train on FINAL enhanced features (IMPORTANT)
rf_model.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_rf = rf_model.predict(X_test_enhanced)

# Accuracy
print("Random Forest Accuracy:",
      accuracy_score(y_test_encoded, y_pred_rf))

# Classification Report
print(classification_report(y_test_encoded, y_pred_rf))

# Probabilities (for ROC-AUC)
y_prob_rf = rf_model.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "Random Forest",
    y_test_encoded,
    y_pred_rf,
    y_prob_rf
)


# XGBoost

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    eval_metric="mlogloss"
)

# ✅ Train on FINAL enhanced features
xgb.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_xgb = xgb.predict(X_test_enhanced)

# Accuracy
print("XGBoost Accuracy:",
      accuracy_score(y_test_encoded, y_pred_xgb))

# Classification Report
print(classification_report(y_test_encoded, y_pred_xgb))

# Probabilities (for ROC-AUC)
y_prob_xgb = xgb.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "XGBoost",
    y_test_encoded,
    y_pred_xgb,
    y_prob_xgb
)




k = 10 f = 60

In [ ]:
import pandas as pd
import re

# ======================================
# 1. Load the Excel file
# ======================================
df = pd.read_excel("final_moa_cateogrization_Peptides+Smiles.xlsx")

# ======================================
# 2. Remove duplicate rows
# ======================================
df = df.drop_duplicates()

# ======================================
# 3. Fill missing MoA with "Unknown"
# ======================================
df["MoA"] = df["MoA"].fillna("Unknown")

# ======================================
# 4. Drop MoA with "Unknown"
# ======================================
df = df[df["MoA"] != "Unknown"]

# ======================================
# 4. Clean MoA text
# ======================================
def clean_moa(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", re.sub(r"[^a-z0-9\s]", " ", text))
    return text.strip()

df["MoA_clean"] = df["MoA"].apply(clean_moa)

# ======================================
# 5. Categorization
# ======================================
def categorize_moa(text):

    text = text.lower()

    if text == "unknown":
        return "Multi-target / Polypharmacology"

    # 1 Membrane / Cell Envelope Disruption
    if any(k in text for k in [
        "membrane",
        "cell membrane",
        "pore",
        "permeability",
        "depolar",
        "lipid",
        "bilayer",
        "detergent",
        "ferriprotoporphyrin",
        "reactive nitro radical",
        "reactive intermediates",
        "membrane depolarization",
        "membrane permeabilization",
        "membrane damage",
        "membrane disrupting",
        "breaks down membrane lipids",
        "destroys cell membranes"
    ]):
        return "Membrane / Cell Envelope Disruption"

    # 2 Cell Wall Biosynthesis Inhibition
    if any(k in text for k in [
        "penicillin-binding protein",
        "pbp",
        "peptidoglycan",
        "cell wall",
        "lipid ii",
        "alanine racemase",
        "beta lactamase",
        "beta-lactamase",
        "penicillin binding protein",
        "fab i",
        "enoyl acyl carrier protein reductase"
    ]):
        return "Cell Wall Biosynthesis Inhibition"

    # 3 Protein Synthesis Inhibition
    if any(k in text for k in [
        "ribosome",
        "70s",
        "30s",
        "50s",
        "translation",
        "elongation factor",
        "trna",
        "protein synthesis",
        "trna synthetase",
        "trna ligase"
    ]):
        return "Protein Synthesis Inhibition"

    # 4 Nucleic Acid Synthesis / Integrity Disruption
    if any(k in text for k in [
        "dna",
        "rna",
        "gyrase",
        "topoisomerase",
        "polymerase",
        "replication",
        "intercalat",
        "dna binding",
        "dna disrupting",
        "dna inhibitor",
        "ribonucleotide reductase"
    ]):
        return "Nucleic Acid Synthesis / Integrity Disruption"

    # 5 Metabolic Pathway Inhibition
    if any(k in text for k in [
        "dihydrofolate reductase",
        "dihydropteroate synthase",
        "folate",
        "biosynthesis",
        "metabolism",
        "amidophosphoribosyltransferase",
        "oxidoreductase",
        "thymidylate synthase",
        "inosine monophosphate dehydrogenase",
        "dxr",
        "reductase",
        "synthase",
        "dehydrogenase",
        "cyclooxygenase",
        "carbonic anhydrase",
        "monoamine oxidase",
        "hmg coa reductase",
        "xanthine dehydrogenase",
        "caspase",
        "matrix metalloproteinase",
        "reductoisomerase",
        "dioxp reductoisomerase",
        "dxr",
        "dipeptidase",
        "diacylglycerol o acyltransferase",
        "dgat",
        "glucosyltransferase",
        "ceramide glucosyltransferase"
    ]):
        return "Metabolic Pathway Inhibition"

    # 6 Redox / Oxidative Stress Induction
    if any(k in text for k in [
        "oxidative",
        "redox",
        "ros",
        "reactive",
        "nitro",
        "oxidase",
        "reducing agent",
        "antioxidant",
        "increases ros",
        "chelating",
        "copper chelating",
        "iron chelating",
        "aluminium chelating",
        "glutathione",
        "glutathione precursor",
        "redox regulator"
    ]):
        return "Redox / Oxidative Stress Induction"

    # 7 Regulatory / Signalling Interference
    if any(k in text for k in [
        "quorum",
        "two component",
        "regulation",
        "signalling",
        "stress response",
        "transcription",
        "toll like receptor",
        "nf kb",
        "mtor",
        "map kinase",
        "cyclin dependent kinase",
        "biofilm related genes",
        "guanylate cyclase",
        "cyclase activator",
        "cyclase inhibitor",
        "cap binding complex",
        "signal transduction modulator",
        "arca",
        "arcb",
        "arcc",
        "arcd",
        "arc gene",
        "arginine deiminase pathway"
    ]):
        return "Regulatory / Signalling Interference"

    # 8 Biofilm Matrix Disruption
    if any(k in text for k in [
        "biofilm",
        "edna",
        "nuclease"
    ]):
        return "Biofilm Matrix Disruption"

    # 9 Antifungal Sterol Biosynthesis
    if any(k in text for k in [
        "cytochrome p450",
        "cyp51",
        "lanosterol",
        "ergosterol",
        "sterol biosynthesis",
        "squalene monooxygenase"
    ]):
        return "Antifungal Sterol Biosynthesis Inhibitors"

    # 10 Antiviral agents
    if any(k in text for k in [
        "hiv",
        "herpesvirus",
        "influenza",
        "hepatitis c",
        "reverse transcriptase",
        "viral protease",
        "viral capsid",
        "neuraminidase",
        "viral polymerase",
        "viral", "hepatitis", "capsid", "viral kinase", "helicase", "primase",
        "hiv protease",
        "hiv 1 protease",
        "viral protease",
        "pul97",
        "viral phosphotransferase",
        "human immunodeficiency virus type 1 protease"
    ]):
        return "Antiviral agents"

    # 11 Affecting Human Physiology
    if any(k in text for k in [
        "adrenergic receptor",
        "dopamine receptor",
        "serotonin transporter",
        "histamine receptor",
        "muscarinic receptor",
        "angiotensin receptor",
        "opioid receptor",
        "cannabinoid receptor",
        "vitamin d receptor",
        "androgen receptor",
        "estrogen receptor",
        "progesterone receptor",
        "glucocorticoid receptor",
        "receptor",
        "agonist",
        "antagonist",
        "partial agonist",
        "inverse agonist",
        "allosteric modulator",
        "allosteric antagonist",
        "positive modulator",
        "negative modulator",
        "retinoid receptor",
        "retinoic acid receptor",
        "retinoid x receptor",
        "adenosine receptor",
        "dopamine receptor",
        "serotonin receptor",
        "histamine receptor",
        "angiotensin receptor",
        "muscarinic receptor",
        "nicotinic receptor",
        "cannabinoid receptor",
        "opioid receptor",
        "purinergic receptor",
        "vanilloid receptor",
        "bile acid receptor",
        "fxr receptor",
        "peroxisome proliferator activated receptor",
        "ppar",
        "glucocorticoid receptor",
        "mineralocorticoid receptor",
        "estrogen receptor",
        "androgen receptor",
        "progesterone receptor",
        "vitamin d receptor",
        "chemokine receptor",
        "cxcr",
        "s1p receptor",
        "sphingosine receptor",
        "nmda receptor",
        "gaba receptor",
        "acetylcholine receptor",

        "ion channel",
        "channel blocker",
        "channel opener",
        "channel modulator",
        "voltage gated",
        "voltage activated",
        "calcium channel",
        "sodium channel",
        "potassium channel",
        "chloride channel",
        "anion channel",
        "herg",
        "enac",
        "ryanodine receptor",
        "l type calcium channel",
        "inwardly rectifying potassium channel",
        "calcium activated potassium channel",

        "acetylcholinesterase",
        "cholinesterase",
        "dopamine transporter",
        "serotonin transporter",
        "norepinephrine transporter",
        "gaba transporter",
        "synaptic vesicle transporter",
        "vesicular amine transporter",
        "sv2a",
        "neurotransmitter reuptake",
        "bcr abl",
        "abl kinase",
        "fusion protein inhibitor"

        "angiotensin-converting enzyme",
        "angiotensin converting enzyme",
        "ace inhibitor",
        "lipoxygenase",
        "cyclooxygenase",
        "phosphodiesterase",
        "pde inhibitor",
        "neprilysin",
        "lipase",
        "glucosidase",
        "decarboxylase",
        "monoamine oxidase",
        "thrombin",
        "farnesyltransferase",
        "kinase",
        "phosphatase",
        "ubiquitin ligase",
        "aurora kinase",
        "pdgfr",
        "pdk1",
        "ikk",

        "transporter",
        "solute carrier",
        "slc",
        "cotransporter",
        "sodium chloride cotransporter",
        "sodium potassium chloride cotransporter",
        "npc1l1",
        "amine transporter",
        "renal transporter",

        "insulin receptor",
        "hormone receptor",
        "glucose metabolism",
        "lipid metabolism",
        "fatty acid metabolism",
        "ppar alpha",
        "ppar gamma",
        "ppar delta",
        "metabolic regulator",

        "growth factor receptor",
        "egfr",
        "map kinase",
        "mtor",
        "nf kb",
        "signal transduction",
        "transcription factor",
        "cell signaling",
        "cytokine receptor",

        "tubulin",
        "microtubule",
        "cytoskeleton",
        "spindle assembly",
        "microtubule stabilizer",
        "microtubule inhibitor"
        "potassium transporting atpase",
        "npc1l1",
        "niemmann-pick c1",
        "niemann pick c1",
        "synaptic vesicle glycoprotein",
        "sv2a",
        "sv2a modulator",
        "laxative",
        "potassium-transporting atpase inhibitor",
        "potassium transporting atpase",
        "potassium atpase",
        "synaptic vesicle glycoprotein 2a",
        "cholesterol absorption inhibitor"



    ]):
        return "Affecting Human Physiology"

    # 12 Multi-target
    if any(k in text for k in [
        "multi target",
        "multiple",
        "polypharmacology",
        "pleiotropic"
    ]):
        return "Multi-target / Polypharmacology"

    return "Others"


# ======================================
# 6. Apply categorization
# ======================================
df["MoA_category"] = df["MoA_clean"].apply(categorize_moa)

# ======================================
# 7. Keep only required columns
# ======================================
df_final = df[["Name", "SMILES", "MoA_category"]]

# ======================================
# 8. Rename column
# ======================================
df_final = df_final.rename(columns={"MoA_category": "MoA"})

# ======================================
# 9. Check distribution
# ======================================
print(df["MoA_category"].value_counts())

# ======================================
# 10. Save final file
# ======================================
df_final.to_excel("final_moa_categorized.xlsx", index=False)

print(df[df["MoA_category"]=="Others"]["MoA"].unique())

# ======================================
# 10. Save categorized data into multiple sheets
# ======================================

with pd.ExcelWriter("final_moa_categorized.xlsx", engine="xlsxwriter") as writer:

    # Main categorized dataset
    df_final.to_excel(writer, sheet_name="All_Data_PA", index=False)

    # Human physiology targets
    human_targets = df[df["MoA_category"] == "Affecting Human Physiology"]
    human_targets.to_excel(writer, sheet_name="Human_Physiology_Targets", index=False)

    # Antiviral agents
    antiviral_targets = df[df["MoA_category"] == "Antiviral agents"]
    antiviral_targets.to_excel(writer, sheet_name="Antiviral_Targets", index=False)

    # Antifungal agents
    antifungal_targets = df[df["MoA_category"] == "Antifungal Sterol Biosynthesis Inhibitors"]
    antifungal_targets.to_excel(writer, sheet_name="Antifungal_Targets", index=False)

    # Others
    Others = df[df["MoA_category"] == "Others"]
    Others.to_excel(writer, sheet_name="Others", index=False)

    # ======================================
    # Direct Antibacterial (remove unwanted categories)
    # ======================================
    direct_antibacterial = df[~df["MoA_category"].isin([
        "Affecting Human Physiology",
        "Antiviral agents",
        "Antifungal Sterol Biosynthesis Inhibitors",
        "Others"
    ])]

    direct_antibacterial.to_excel(writer, sheet_name="Direct_Antibacterial", index=False)

print("Excel file created with multiple sheets including Direct_Antibacterial.")

import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

# Load dataset from the correct file and sheet
df = pd.read_excel("final_moa_categorized.xlsx", sheet_name="Direct_Antibacterial")

# Get list of all RDKit descriptors
descriptor_names = [desc[0] for desc in Descriptors.descList]

def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return [None]*len(descriptor_names)

    descriptor_values = []

    for name, function in Descriptors.descList:
        try:
            value = function(mol)
        except:
            value = None
        descriptor_values.append(value)

    return descriptor_values


# Apply descriptor calculation
descriptor_data = df["SMILES"].apply(calculate_descriptors)

# Convert to dataframe
X = pd.DataFrame(descriptor_data.tolist(), columns=descriptor_names)

# Target variable
y = df["MoA_category"]

# Combine if needed
final_df = pd.concat([X, y], axis=1)

# Save descriptor dataset
final_df.to_excel("MoA_rdkit_all_descriptors_calculated.xlsx", index=False)

print("Total descriptors extracted:", len(descriptor_names))
print("Dataset shape:", final_df.shape)


# Load Descriptor data

import pandas as pd

df = pd.read_excel("MoA_rdkit_all_descriptors_calculated.xlsx")

print("Dataset shape:", df.shape)

df.head()

# seperate descriptors and label(moa)

X = df.drop(columns=["MoA_category"])
y = df["MoA_category"]

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

# -------------------------------
#  FILTER RARE CLASSES HERE
# -------------------------------

df_model = X.copy()
df_model["MoA_category"] = y

counts = df_model["MoA_category"].value_counts()
valid_classes = counts[counts >= 3].index

df_filtered = df_model[df_model["MoA_category"].isin(valid_classes)].copy()

# Update X and y AFTER filtering
X = df_filtered.drop(columns=["MoA_category"])
y = df_filtered["MoA_category"]

print("After filtering:", X.shape, y.shape)

import numpy as np

# Replace infinity values
X = X.replace([np.inf, -np.inf], np.nan)

# Convert to float32 to catch values too large for this dtype and re-handle potential new NaNs
X = X.astype(np.float32)

# Replace any new inf values that might have been created due to float32 conversion overflow
X = X.replace([np.inf, -np.inf], np.nan)

# Fill missing values

X = X.fillna(X.median())

# Remove Extremely Large Descriptors
# Some descriptors can reach 10¹² or more, causing overflow.

X.describe().T.sort_values("max", ascending=False).head(10)

# A safe solution is to clip values.

X = X.clip(-1e6, 1e6)

# A safe solution is to clip values.

X = X.astype(np.float64)

# check missing, infinity and large values

import numpy as np

print("NaN values:", np.isnan(X).sum().sum())
print("Infinity values:", np.isinf(X).sum().sum())
print("Max value in dataset:", np.nanmax(X.values))

# Train-Test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# label encoding

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_encoded = le.fit_transform(y_train)

y_test_encoded = le.transform(y_test)

# Feature stability

from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0)

# Fit ONLY on training data
X_train_var = selector.fit_transform(X_train)

# Apply SAME transformation on test data
X_test_var = selector.transform(X_test)

# Convert to DataFrame (optional but good)
selected_columns = X_train.columns[selector.get_support()]

X_train_var = pd.DataFrame(X_train_var, columns=selected_columns)
X_test_var = pd.DataFrame(X_test_var, columns=selected_columns)

print("After removing constant features:", X_train_var.shape[1])

import numpy as np

# Step 1: Compute correlation ONLY on training data
corr_matrix = X_train_var.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]

print("Descriptors to drop:", len(to_drop))

# Step 2: Drop from TRAIN
X_train_corr = X_train_var.drop(columns=to_drop)

# Step 3: Drop SAME columns from TEST
X_test_corr = X_test_var.drop(columns=to_drop)

print("After correlation removal (train):", X_train_corr.shape)
print("After correlation removal (test):", X_test_corr.shape)

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=400,
    random_state=42
)

# ✅ Train ONLY on training data
rf.fit(X_train_corr, y_train)

# Get importance
importance = pd.Series(
    rf.feature_importances_,
    index=X_train_corr.columns
)

# Sort
importance = importance.sort_values(ascending=False)

print("Top features:\n", importance.head(30))

# Select top 60 features
top_features = importance.head(60).index

# Apply SAME features to train and test
X_train_sel = X_train_corr[top_features]
X_test_sel = X_test_corr[top_features]

print("Final train shape:", X_train_sel.shape)
print("Final test shape:", X_test_sel.shape)

# Normalization

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit ONLY on training
X_train_scaled = scaler.fit_transform(X_train_sel)

# Apply to test
X_test_scaled = scaler.transform(X_test_sel)

# Find best K

from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt

bic_scores = []

k_values = [5, 8, 10, 12, 15, 18, 20, 22]

for k in k_values:

    gmm = GaussianMixture(
        n_components=k,
        covariance_type='full',
        random_state=42
    )

    gmm.fit(X_train_scaled)

    bic = gmm.bic(X_train_scaled)

    bic_scores.append(bic)

    print(f"K = {k}, BIC = {bic}")

# Silhouette Score for best K

from sklearn.metrics import silhouette_score

for k in k_values:

    gmm = GaussianMixture(n_components=k, random_state=42)

    labels = gmm.fit_predict(X_train_scaled)

    score = silhouette_score(X_train_scaled, labels)

    print(f"K = {k}, Silhouette Score = {score}")

# Apply GMM

from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=10, random_state=42)

gmm.fit(X_train_scaled)

train_gmm = gmm.predict_proba(X_train_scaled)
test_gmm = gmm.predict_proba(X_test_scaled)

# Combine
X_train_final = np.hstack([X_train_scaled, train_gmm])
X_test_final = np.hstack([X_test_scaled, test_gmm])

# GMM probabilities for TRAIN
gmm_train_probs = gmm.predict_proba(X_train_scaled)

gmm_train_df = pd.DataFrame(
    gmm_train_probs,
    columns=[f"GMM_Cluster_{i}" for i in range(gmm_train_probs.shape[1])]
)

# Convert scaled train to DataFrame
X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=X_train_sel.columns
)

# Concatenate
X_train_enhanced = pd.concat([X_train_scaled_df, gmm_train_df], axis=1)

print("Train shape:", X_train_enhanced.shape)
print(X_train_enhanced.columns[-10:])

# GMM probabilities for TEST
gmm_test_probs = gmm.predict_proba(X_test_scaled)

gmm_test_df = pd.DataFrame(
    gmm_test_probs,
    columns=[f"GMM_Cluster_{i}" for i in range(gmm_test_probs.shape[1])]
)

# Convert scaled test to DataFrame
X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=X_train_sel.columns   # IMPORTANT: use train columns
)

# Concatenate
X_test_enhanced = pd.concat([X_test_scaled_df, gmm_test_df], axis=1)

print("Test shape:", X_test_enhanced.shape)
print(X_test_enhanced.columns[-10:])

cluster_labels = gmm.predict(X_train_scaled)

df_analysis = pd.DataFrame({
    "Cluster": cluster_labels,
    "MoA": y_train.values
})

print(df_analysis.groupby("Cluster")["MoA"].value_counts())

# ---Function for evaluating metrices---

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    classification_report
)

def evaluate_model_full(model_name, y_test, y_pred, y_prob):

    print("\n==============================")
    print("MODEL:", model_name)
    print("==============================")

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    print("\nOverall Confusion Matrix:\n")
    print(cm)

    # Plot confusion matrix
    plt.figure(figsize=(8,6))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=le.classes_,
        yticklabels=le.classes_
    )

    plt.xlabel("Predicted MoA")
    plt.ylabel("Actual MoA")
    plt.title(model_name + " Confusion Matrix")

    plt.show()

    # ------------------------------
    # TP FP FN TN PER CLASS - CALCULATE HERE
    # ------------------------------

    TP = np.diag(cm)

    FP = np.sum(cm, axis=0) - TP

    FN = np.sum(cm, axis=1) - TP

    TN = np.sum(cm) - (TP + FP + FN)

    # --------------------------------
    # PER-MoA CONFUSION MATRICES
    # --------------------------------

    per_moa_cm_list = []

    for i, moa in enumerate(le.classes_):

        tp = TP[i]
        fp = FP[i]
        fn = FN[i]
        tn = TN[i]

        cm_moa = np.array([
            [tp, fn],
            [fp, tn]
        ])

        print("\nConfusion Matrix for MoA:", moa)
        print(cm_moa)

        plt.figure(figsize=(4,4))

        sns.heatmap(
            cm_moa,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=["Predicted "+moa, "Predicted Other"],
            yticklabels=["Actual "+moa, "Actual Other"]
        )

        plt.title(model_name + " - " + moa)

        plt.show()

        per_moa_cm_list.append({
            "MoA": moa,
            "TP": tp,
            "FN": fn,
            "FP": fp,
            "TN": tn
        })

    per_moa_cm_df = pd.DataFrame(per_moa_cm_list)

    per_moa_cm_df.to_excel(model_name+"_per_moa_confusion_matrix.xlsx", index=False)

    # Sensitivity (Recall)
    sensitivity = TP / (TP + FN)

    # Specificity
    specificity = TN / (TN + FP)

    per_class_df = pd.DataFrame({
        "MoA": le.classes_,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "Sensitivity": sensitivity,
        "Specificity": specificity
    })

    print("\nPer-MoA Metrics\n")
    print(per_class_df)

    # Save results
    per_class_df.to_excel(model_name+"_per_MoA_metrics.xlsx", index=False)

    # ------------------------------
    # OVERALL METRICS
    # ------------------------------

    overall_TP = TP.sum()

    overall_FP = FP.sum()

    overall_FN = FN.sum()

    overall_TN = TN.sum()

    overall_sensitivity = overall_TP / (overall_TP + overall_FN)

    overall_specificity = overall_TN / (overall_TN + overall_FP)

    overall_df = pd.DataFrame({
        "Metric": [
            "TP",
            "FP",
            "FN",
            "TN",
            "Sensitivity",
            "Specificity"
        ],
        "Value": [
            overall_TP,
            overall_FP,
            overall_FN,
            overall_TN,
            overall_sensitivity,
            overall_specificity
        ]
    })

    print("\nOverall Metrics\n")
    print(overall_df)

    overall_df.to_excel(model_name+"_overall_metrics.xlsx", index=False)

    # ------------------------------
    # Accuracy
    # ------------------------------

    acc = accuracy_score(y_test, y_pred)

    print("\nAccuracy:", acc)

    # ------------------------------
    # MCC
    # ------------------------------

    mcc = matthews_corrcoef(y_test, y_pred)

    print("MCC:", mcc)

    # ------------------------------
    # ROC AUC (Multiclass)
    # ------------------------------

    from sklearn.preprocessing import label_binarize

    y_test_bin = label_binarize(y_test, classes=np.unique(y_test))

    auc = roc_auc_score(y_test_bin, y_prob, multi_class="ovr")

    print("ROC-AUC:", auc)

    # Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features='sqrt'
)

# ✅ Train on FINAL enhanced features (IMPORTANT)
rf_model.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_rf = rf_model.predict(X_test_enhanced)

# Accuracy
print("Random Forest Accuracy:",
      accuracy_score(y_test_encoded, y_pred_rf))

# Classification Report
print(classification_report(y_test_encoded, y_pred_rf))

# Probabilities (for ROC-AUC)
y_prob_rf = rf_model.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "Random Forest",
    y_test_encoded,
    y_pred_rf,
    y_prob_rf
)

# Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features='sqrt'
)

# ✅ Train on FINAL enhanced features (IMPORTANT)
rf_model.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_rf = rf_model.predict(X_test_enhanced)

# Accuracy
print("Random Forest Accuracy:",
      accuracy_score(y_test_encoded, y_pred_rf))

# Classification Report
print(classification_report(y_test_encoded, y_pred_rf))

# Probabilities (for ROC-AUC)
y_prob_rf = rf_model.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "Random Forest",
    y_test_encoded,
    y_pred_rf,
    y_prob_rf
)


# XGBoost

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    eval_metric="mlogloss"
)

# ✅ Train on FINAL enhanced features
xgb.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_xgb = xgb.predict(X_test_enhanced)

# Accuracy
print("XGBoost Accuracy:",
      accuracy_score(y_test_encoded, y_pred_xgb))

# Classification Report
print(classification_report(y_test_encoded, y_pred_xgb))

# Probabilities (for ROC-AUC)
y_prob_xgb = xgb.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "XGBoost",
    y_test_encoded,
    y_pred_xgb,
    y_prob_xgb
)




k = 10 f = 70

In [ ]:
import pandas as pd
import re

# ======================================
# 1. Load the Excel file
# ======================================
df = pd.read_excel("final_moa_cateogrization_Peptides+Smiles.xlsx")

# ======================================
# 2. Remove duplicate rows
# ======================================
df = df.drop_duplicates()

# ======================================
# 3. Fill missing MoA with "Unknown"
# ======================================
df["MoA"] = df["MoA"].fillna("Unknown")

# ======================================
# 4. Drop MoA with "Unknown"
# ======================================
df = df[df["MoA"] != "Unknown"]

# ======================================
# 4. Clean MoA text
# ======================================
def clean_moa(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", re.sub(r"[^a-z0-9\s]", " ", text))
    return text.strip()

df["MoA_clean"] = df["MoA"].apply(clean_moa)

# ======================================
# 5. Categorization
# ======================================
def categorize_moa(text):

    text = text.lower()

    if text == "unknown":
        return "Multi-target / Polypharmacology"

    # 1 Membrane / Cell Envelope Disruption
    if any(k in text for k in [
        "membrane",
        "cell membrane",
        "pore",
        "permeability",
        "depolar",
        "lipid",
        "bilayer",
        "detergent",
        "ferriprotoporphyrin",
        "reactive nitro radical",
        "reactive intermediates",
        "membrane depolarization",
        "membrane permeabilization",
        "membrane damage",
        "membrane disrupting",
        "breaks down membrane lipids",
        "destroys cell membranes"
    ]):
        return "Membrane / Cell Envelope Disruption"

    # 2 Cell Wall Biosynthesis Inhibition
    if any(k in text for k in [
        "penicillin-binding protein",
        "pbp",
        "peptidoglycan",
        "cell wall",
        "lipid ii",
        "alanine racemase",
        "beta lactamase",
        "beta-lactamase",
        "penicillin binding protein",
        "fab i",
        "enoyl acyl carrier protein reductase"
    ]):
        return "Cell Wall Biosynthesis Inhibition"

    # 3 Protein Synthesis Inhibition
    if any(k in text for k in [
        "ribosome",
        "70s",
        "30s",
        "50s",
        "translation",
        "elongation factor",
        "trna",
        "protein synthesis",
        "trna synthetase",
        "trna ligase"
    ]):
        return "Protein Synthesis Inhibition"

    # 4 Nucleic Acid Synthesis / Integrity Disruption
    if any(k in text for k in [
        "dna",
        "rna",
        "gyrase",
        "topoisomerase",
        "polymerase",
        "replication",
        "intercalat",
        "dna binding",
        "dna disrupting",
        "dna inhibitor",
        "ribonucleotide reductase"
    ]):
        return "Nucleic Acid Synthesis / Integrity Disruption"

    # 5 Metabolic Pathway Inhibition
    if any(k in text for k in [
        "dihydrofolate reductase",
        "dihydropteroate synthase",
        "folate",
        "biosynthesis",
        "metabolism",
        "amidophosphoribosyltransferase",
        "oxidoreductase",
        "thymidylate synthase",
        "inosine monophosphate dehydrogenase",
        "dxr",
        "reductase",
        "synthase",
        "dehydrogenase",
        "cyclooxygenase",
        "carbonic anhydrase",
        "monoamine oxidase",
        "hmg coa reductase",
        "xanthine dehydrogenase",
        "caspase",
        "matrix metalloproteinase",
        "reductoisomerase",
        "dioxp reductoisomerase",
        "dxr",
        "dipeptidase",
        "diacylglycerol o acyltransferase",
        "dgat",
        "glucosyltransferase",
        "ceramide glucosyltransferase"
    ]):
        return "Metabolic Pathway Inhibition"

    # 6 Redox / Oxidative Stress Induction
    if any(k in text for k in [
        "oxidative",
        "redox",
        "ros",
        "reactive",
        "nitro",
        "oxidase",
        "reducing agent",
        "antioxidant",
        "increases ros",
        "chelating",
        "copper chelating",
        "iron chelating",
        "aluminium chelating",
        "glutathione",
        "glutathione precursor",
        "redox regulator"
    ]):
        return "Redox / Oxidative Stress Induction"

    # 7 Regulatory / Signalling Interference
    if any(k in text for k in [
        "quorum",
        "two component",
        "regulation",
        "signalling",
        "stress response",
        "transcription",
        "toll like receptor",
        "nf kb",
        "mtor",
        "map kinase",
        "cyclin dependent kinase",
        "biofilm related genes",
        "guanylate cyclase",
        "cyclase activator",
        "cyclase inhibitor",
        "cap binding complex",
        "signal transduction modulator",
        "arca",
        "arcb",
        "arcc",
        "arcd",
        "arc gene",
        "arginine deiminase pathway"
    ]):
        return "Regulatory / Signalling Interference"

    # 8 Biofilm Matrix Disruption
    if any(k in text for k in [
        "biofilm",
        "edna",
        "nuclease"
    ]):
        return "Biofilm Matrix Disruption"

    # 9 Antifungal Sterol Biosynthesis
    if any(k in text for k in [
        "cytochrome p450",
        "cyp51",
        "lanosterol",
        "ergosterol",
        "sterol biosynthesis",
        "squalene monooxygenase"
    ]):
        return "Antifungal Sterol Biosynthesis Inhibitors"

    # 10 Antiviral agents
    if any(k in text for k in [
        "hiv",
        "herpesvirus",
        "influenza",
        "hepatitis c",
        "reverse transcriptase",
        "viral protease",
        "viral capsid",
        "neuraminidase",
        "viral polymerase",
        "viral", "hepatitis", "capsid", "viral kinase", "helicase", "primase",
        "hiv protease",
        "hiv 1 protease",
        "viral protease",
        "pul97",
        "viral phosphotransferase",

        "human immunodeficiency virus type 1 protease"
    ]):
        return "Antiviral agents"

    # 11 Affecting Human Physiology
    if any(k in text for k in [
        "adrenergic receptor",
        "dopamine receptor",
        "serotonin transporter",
        "histamine receptor",
        "muscarinic receptor",
        "angiotensin receptor",
        "opioid receptor",
        "cannabinoid receptor",
        "vitamin d receptor",
        "androgen receptor",
        "estrogen receptor",
        "progesterone receptor",
        "glucocorticoid receptor",
        "receptor",
        "agonist",
        "antagonist",
        "partial agonist",
        "inverse agonist",
        "allosteric modulator",
        "allosteric antagonist",
        "positive modulator",
        "negative modulator",
        "retinoid receptor",
        "retinoic acid receptor",
        "retinoid x receptor",
        "adenosine receptor",
        "dopamine receptor",
        "serotonin receptor",
        "histamine receptor",
        "angiotensin receptor",
        "muscarinic receptor",
        "nicotinic receptor",
        "cannabinoid receptor",
        "opioid receptor",
        "purinergic receptor",
        "vanilloid receptor",
        "bile acid receptor",
        "fxr receptor",
        "peroxisome proliferator activated receptor",
        "ppar",
        "glucocorticoid receptor",
        "mineralocorticoid receptor",
        "estrogen receptor",
        "androgen receptor",
        "progesterone receptor",
        "vitamin d receptor",
        "chemokine receptor",
        "cxcr",
        "s1p receptor",
        "sphingosine receptor",
        "nmda receptor",
        "gaba receptor",
        "acetylcholine receptor",

        "ion channel",
        "channel blocker",
        "channel opener",
        "channel modulator",
        "voltage gated",
        "voltage activated",
        "calcium channel",
        "sodium channel",
        "potassium channel",
        "chloride channel",
        "anion channel",
        "herg",
        "enac",
        "ryanodine receptor",
        "l type calcium channel",
        "inwardly rectifying potassium channel",
        "calcium activated potassium channel",

        "acetylcholinesterase",
        "cholinesterase",
        "dopamine transporter",
        "serotonin transporter",
        "norepinephrine transporter",
        "gaba transporter",
        "synaptic vesicle transporter",
        "vesicular amine transporter",
        "sv2a",
        "neurotransmitter reuptake",
        "bcr abl",
        "abl kinase",
        "fusion protein inhibitor"

        "angiotensin-converting enzyme",
        "angiotensin converting enzyme",
        "ace inhibitor",
        "lipoxygenase",
        "cyclooxygenase",
        "phosphodiesterase",
        "pde inhibitor",
        "neprilysin",
        "lipase",
        "glucosidase",
        "decarboxylase",
        "monoamine oxidase",
        "thrombin",
        "farnesyltransferase",
        "kinase",
        "phosphatase",
        "ubiquitin ligase",
        "aurora kinase",
        "pdgfr",
        "pdk1",
        "ikk",

        "transporter",
        "solute carrier",
        "slc",
        "cotransporter",
        "sodium chloride cotransporter",
        "sodium potassium chloride cotransporter",
        "npc1l1",
        "amine transporter",
        "renal transporter",

        "insulin receptor",
        "hormone receptor",
        "glucose metabolism",
        "lipid metabolism",
        "fatty acid metabolism",
        "ppar alpha",
        "ppar gamma",
        "ppar delta",
        "metabolic regulator",

        "growth factor receptor",
        "egfr",
        "map kinase",
        "mtor",
        "nf kb",
        "signal transduction",
        "transcription factor",
        "cell signaling",
        "cytokine receptor",

        "tubulin",
        "microtubule",
        "cytoskeleton",
        "spindle assembly",
        "microtubule stabilizer",
        "microtubule inhibitor"
        "potassium transporting atpase",
        "npc1l1",
        "niemmann-pick c1",
        "niemann pick c1",
        "synaptic vesicle glycoprotein",
        "sv2a",
        "sv2a modulator",
        "laxative",
        "potassium-transporting atpase inhibitor",
        "potassium transporting atpase",
        "potassium atpase",
        "synaptic vesicle glycoprotein 2a",
        "cholesterol absorption inhibitor"



    ]):
        return "Affecting Human Physiology"

    # 12 Multi-target
    if any(k in text for k in [
        "multi target",
        "multiple",
        "polypharmacology",
        "pleiotropic"
    ]):
        return "Multi-target / Polypharmacology"

    return "Others"


# ======================================
# 6. Apply categorization
# ======================================
df["MoA_category"] = df["MoA_clean"].apply(categorize_moa)

# ======================================
# 7. Keep only required columns
# ======================================
df_final = df[["Name", "SMILES", "MoA_category"]]

# ======================================
# 8. Rename column
# ======================================
df_final = df_final.rename(columns={"MoA_category": "MoA"})

# ======================================
# 9. Check distribution
# ======================================
print(df["MoA_category"].value_counts())

# ======================================
# 10. Save final file
# ======================================
df_final.to_excel("final_moa_categorized.xlsx", index=False)

print(df[df["MoA_category"]=="Others"]["MoA"].unique())

# ======================================
# 10. Save categorized data into multiple sheets
# ======================================

with pd.ExcelWriter("final_moa_categorized.xlsx", engine="xlsxwriter") as writer:

    # Main categorized dataset
    df_final.to_excel(writer, sheet_name="All_Data_PA", index=False)

    # Human physiology targets
    human_targets = df[df["MoA_category"] == "Affecting Human Physiology"]
    human_targets.to_excel(writer, sheet_name="Human_Physiology_Targets", index=False)

    # Antiviral agents
    antiviral_targets = df[df["MoA_category"] == "Antiviral agents"]
    antiviral_targets.to_excel(writer, sheet_name="Antiviral_Targets", index=False)

    # Antifungal agents
    antifungal_targets = df[df["MoA_category"] == "Antifungal Sterol Biosynthesis Inhibitors"]
    antifungal_targets.to_excel(writer, sheet_name="Antifungal_Targets", index=False)

    # Others
    Others = df[df["MoA_category"] == "Others"]
    Others.to_excel(writer, sheet_name="Others", index=False)

    # ======================================
    # Direct Antibacterial (remove unwanted categories)
    # ======================================
    direct_antibacterial = df[~df["MoA_category"].isin([
        "Affecting Human Physiology",
        "Antiviral agents",
        "Antifungal Sterol Biosynthesis Inhibitors",
        "Others"
    ])]

    direct_antibacterial.to_excel(writer, sheet_name="Direct_Antibacterial", index=False)

print("Excel file created with multiple sheets including Direct_Antibacterial.")

import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

# Load dataset from the correct file and sheet
df = pd.read_excel("final_moa_categorized.xlsx", sheet_name="Direct_Antibacterial")

# Get list of all RDKit descriptors
descriptor_names = [desc[0] for desc in Descriptors.descList]

def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return [None]*len(descriptor_names)

    descriptor_values = []

    for name, function in Descriptors.descList:
        try:
            value = function(mol)
        except:
            value = None
        descriptor_values.append(value)

    return descriptor_values


# Apply descriptor calculation
descriptor_data = df["SMILES"].apply(calculate_descriptors)

# Convert to dataframe
X = pd.DataFrame(descriptor_data.tolist(), columns=descriptor_names)

# Target variable
y = df["MoA_category"]

# Combine if needed
final_df = pd.concat([X, y], axis=1)

# Save descriptor dataset
final_df.to_excel("MoA_rdkit_all_descriptors_calculated.xlsx", index=False)

print("Total descriptors extracted:", len(descriptor_names))
print("Dataset shape:", final_df.shape)


# Load Descriptor data

import pandas as pd

df = pd.read_excel("MoA_rdkit_all_descriptors_calculated.xlsx")

print("Dataset shape:", df.shape)

df.head()

# seperate descriptors and label(moa)

X = df.drop(columns=["MoA_category"])
y = df["MoA_category"]

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

# -------------------------------
#  FILTER RARE CLASSES HERE
# -------------------------------

df_model = X.copy()
df_model["MoA_category"] = y

counts = df_model["MoA_category"].value_counts()
valid_classes = counts[counts >= 3].index

df_filtered = df_model[df_model["MoA_category"].isin(valid_classes)].copy()

# Update X and y AFTER filtering
X = df_filtered.drop(columns=["MoA_category"])
y = df_filtered["MoA_category"]

print("After filtering:", X.shape, y.shape)

import numpy as np

# Replace infinity values
X = X.replace([np.inf, -np.inf], np.nan)

# Convert to float32 to catch values too large for this dtype and re-handle potential new NaNs
X = X.astype(np.float32)

# Replace any new inf values that might have been created due to float32 conversion overflow
X = X.replace([np.inf, -np.inf], np.nan)

# Fill missing values

X = X.fillna(X.median())

# Remove Extremely Large Descriptors
# Some descriptors can reach 10¹² or more, causing overflow.

X.describe().T.sort_values("max", ascending=False).head(10)

# A safe solution is to clip values.

X = X.clip(-1e6, 1e6)

# A safe solution is to clip values.

X = X.astype(np.float64)

# check missing, infinity and large values

import numpy as np

print("NaN values:", np.isnan(X).sum().sum())
print("Infinity values:", np.isinf(X).sum().sum())
print("Max value in dataset:", np.nanmax(X.values))

# Train-Test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# label encoding

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_encoded = le.fit_transform(y_train)

y_test_encoded = le.transform(y_test)

# Feature stability

from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0)

# Fit ONLY on training data
X_train_var = selector.fit_transform(X_train)

# Apply SAME transformation on test data
X_test_var = selector.transform(X_test)

# Convert to DataFrame (optional but good)
selected_columns = X_train.columns[selector.get_support()]

X_train_var = pd.DataFrame(X_train_var, columns=selected_columns)
X_test_var = pd.DataFrame(X_test_var, columns=selected_columns)

print("After removing constant features:", X_train_var.shape[1])

import numpy as np

# Step 1: Compute correlation ONLY on training data
corr_matrix = X_train_var.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]

print("Descriptors to drop:", len(to_drop))

# Step 2: Drop from TRAIN
X_train_corr = X_train_var.drop(columns=to_drop)

# Step 3: Drop SAME columns from TEST
X_test_corr = X_test_var.drop(columns=to_drop)

print("After correlation removal (train):", X_train_corr.shape)
print("After correlation removal (test):", X_test_corr.shape)

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=400,
    random_state=42
)

# ✅ Train ONLY on training data
rf.fit(X_train_corr, y_train)

# Get importance
importance = pd.Series(
    rf.feature_importances_,
    index=X_train_corr.columns
)

# Sort
importance = importance.sort_values(ascending=False)

print("Top features:\n", importance.head(30))

# Select top 70 features
top_features = importance.head(70).index

# Apply SAME features to train and test
X_train_sel = X_train_corr[top_features]
X_test_sel = X_test_corr[top_features]

print("Final train shape:", X_train_sel.shape)
print("Final test shape:", X_test_sel.shape)

# Normalization

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit ONLY on training
X_train_scaled = scaler.fit_transform(X_train_sel)

# Apply to test
X_test_scaled = scaler.transform(X_test_sel)

# Find best K

from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt

bic_scores = []

k_values = [5, 8, 10, 12, 15, 18, 20, 22]

for k in k_values:

    gmm = GaussianMixture(
        n_components=k,
        covariance_type='full',
        random_state=42
    )

    gmm.fit(X_train_scaled)

    bic = gmm.bic(X_train_scaled)

    bic_scores.append(bic)

    print(f"K = {k}, BIC = {bic}")

# Silhouette Score for best K

from sklearn.metrics import silhouette_score

for k in k_values:

    gmm = GaussianMixture(n_components=k, random_state=42)

    labels = gmm.fit_predict(X_train_scaled)

    score = silhouette_score(X_train_scaled, labels)

    print(f"K = {k}, Silhouette Score = {score}")

# Apply GMM

from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=10, random_state=42)

gmm.fit(X_train_scaled)

train_gmm = gmm.predict_proba(X_train_scaled)
test_gmm = gmm.predict_proba(X_test_scaled)

# Combine
X_train_final = np.hstack([X_train_scaled, train_gmm])
X_test_final = np.hstack([X_test_scaled, test_gmm])

# GMM probabilities for TRAIN
gmm_train_probs = gmm.predict_proba(X_train_scaled)

gmm_train_df = pd.DataFrame(
    gmm_train_probs,
    columns=[f"GMM_Cluster_{i}" for i in range(gmm_train_probs.shape[1])]
)

# Convert scaled train to DataFrame
X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=X_train_sel.columns
)

# Concatenate
X_train_enhanced = pd.concat([X_train_scaled_df, gmm_train_df], axis=1)

print("Train shape:", X_train_enhanced.shape)
print(X_train_enhanced.columns[-10:])

# GMM probabilities for TEST
gmm_test_probs = gmm.predict_proba(X_test_scaled)

gmm_test_df = pd.DataFrame(
    gmm_test_probs,
    columns=[f"GMM_Cluster_{i}" for i in range(gmm_test_probs.shape[1])]
)

# Convert scaled test to DataFrame
X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=X_train_sel.columns   # IMPORTANT: use train columns
)

# Concatenate
X_test_enhanced = pd.concat([X_test_scaled_df, gmm_test_df], axis=1)

print("Test shape:", X_test_enhanced.shape)
print(X_test_enhanced.columns[-10:])

cluster_labels = gmm.predict(X_train_scaled)

df_analysis = pd.DataFrame({
    "Cluster": cluster_labels,
    "MoA": y_train.values
})

print(df_analysis.groupby("Cluster")["MoA"].value_counts())

# ---Function for evaluating metrices---

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    classification_report
)

def evaluate_model_full(model_name, y_test, y_pred, y_prob):

    print("\n==============================")
    print("MODEL:", model_name)
    print("==============================")

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    print("\nOverall Confusion Matrix:\n")
    print(cm)

    # Plot confusion matrix
    plt.figure(figsize=(8,6))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=le.classes_,
        yticklabels=le.classes_
    )

    plt.xlabel("Predicted MoA")
    plt.ylabel("Actual MoA")
    plt.title(model_name + " Confusion Matrix")

    plt.show()

    # ------------------------------
    # TP FP FN TN PER CLASS - CALCULATE HERE
    # ------------------------------

    TP = np.diag(cm)

    FP = np.sum(cm, axis=0) - TP

    FN = np.sum(cm, axis=1) - TP

    TN = np.sum(cm) - (TP + FP + FN)

    # --------------------------------
    # PER-MoA CONFUSION MATRICES
    # --------------------------------

    per_moa_cm_list = []

    for i, moa in enumerate(le.classes_):

        tp = TP[i]
        fp = FP[i]
        fn = FN[i]
        tn = TN[i]

        cm_moa = np.array([
            [tp, fn],
            [fp, tn]
        ])

        print("\nConfusion Matrix for MoA:", moa)
        print(cm_moa)

        plt.figure(figsize=(4,4))

        sns.heatmap(
            cm_moa,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=["Predicted "+moa, "Predicted Other"],
            yticklabels=["Actual "+moa, "Actual Other"]
        )

        plt.title(model_name + " - " + moa)

        plt.show()

        per_moa_cm_list.append({
            "MoA": moa,
            "TP": tp,
            "FN": fn,
            "FP": fp,
            "TN": tn
        })

    per_moa_cm_df = pd.DataFrame(per_moa_cm_list)

    per_moa_cm_df.to_excel(model_name+"_per_moa_confusion_matrix.xlsx", index=False)

    # Sensitivity (Recall)
    sensitivity = TP / (TP + FN)

    # Specificity
    specificity = TN / (TN + FP)

    per_class_df = pd.DataFrame({
        "MoA": le.classes_,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "Sensitivity": sensitivity,
        "Specificity": specificity
    })

    print("\nPer-MoA Metrics\n")
    print(per_class_df)

    # Save results
    per_class_df.to_excel(model_name+"_per_MoA_metrics.xlsx", index=False)

    # ------------------------------
    # OVERALL METRICS
    # ------------------------------

    overall_TP = TP.sum()

    overall_FP = FP.sum()

    overall_FN = FN.sum()

    overall_TN = TN.sum()

    overall_sensitivity = overall_TP / (overall_TP + overall_FN)

    overall_specificity = overall_TN / (overall_TN + overall_FP)

    overall_df = pd.DataFrame({
        "Metric": [
            "TP",
            "FP",
            "FN",
            "TN",
            "Sensitivity",
            "Specificity"
        ],
        "Value": [
            overall_TP,
            overall_FP,
            overall_FN,
            overall_TN,
            overall_sensitivity,
            overall_specificity
        ]
    })

    print("\nOverall Metrics\n")
    print(overall_df)

    overall_df.to_excel(model_name+"_overall_metrics.xlsx", index=False)

    # ------------------------------
    # Accuracy
    # ------------------------------

    acc = accuracy_score(y_test, y_pred)

    print("\nAccuracy:", acc)

    # ------------------------------
    # MCC
    # ------------------------------

    mcc = matthews_corrcoef(y_test, y_pred)

    print("MCC:", mcc)

    # ------------------------------
    # ROC AUC (Multiclass)
    # ------------------------------

    from sklearn.preprocessing import label_binarize

    y_test_bin = label_binarize(y_test, classes=np.unique(y_test))

    auc = roc_auc_score(y_test_bin, y_prob, multi_class="ovr")

    print("ROC-AUC:", auc)

    # Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features='sqrt'
)

# ✅ Train on FINAL enhanced features (IMPORTANT)
rf_model.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_rf = rf_model.predict(X_test_enhanced)

# Accuracy
print("Random Forest Accuracy:",
      accuracy_score(y_test_encoded, y_pred_rf))

# Classification Report
print(classification_report(y_test_encoded, y_pred_rf))

# Probabilities (for ROC-AUC)
y_prob_rf = rf_model.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "Random Forest",
    y_test_encoded,
    y_pred_rf,
    y_prob_rf
)

# Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features='sqrt'
)

# ✅ Train on FINAL enhanced features (IMPORTANT)
rf_model.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_rf = rf_model.predict(X_test_enhanced)

# Accuracy
print("Random Forest Accuracy:",
      accuracy_score(y_test_encoded, y_pred_rf))

# Classification Report
print(classification_report(y_test_encoded, y_pred_rf))

# Probabilities (for ROC-AUC)
y_prob_rf = rf_model.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "Random Forest",
    y_test_encoded,
    y_pred_rf,
    y_prob_rf
)


# XGBoost

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    eval_metric="mlogloss"
)

# ✅ Train on FINAL enhanced features
xgb.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_xgb = xgb.predict(X_test_enhanced)

# Accuracy
print("XGBoost Accuracy:",
      accuracy_score(y_test_encoded, y_pred_xgb))

# Classification Report
print(classification_report(y_test_encoded, y_pred_xgb))

# Probabilities (for ROC-AUC)
y_prob_xgb = xgb.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "XGBoost",
    y_test_encoded,
    y_pred_xgb,
    y_prob_xgb
)




k = 10, f = 100

In [ ]:
import pandas as pd
import re

# ======================================
# 1. Load the Excel file
# ======================================
df = pd.read_excel("final_moa_cateogrization_Peptides+Smiles.xlsx")

# ======================================
# 2. Remove duplicate rows
# ======================================
df = df.drop_duplicates()

# ======================================
# 3. Fill missing MoA with "Unknown"
# ======================================
df["MoA"] = df["MoA"].fillna("Unknown")

# ======================================
# 4. Drop MoA with "Unknown"
# ======================================
df = df[df["MoA"] != "Unknown"]

# ======================================
# 4. Clean MoA text
# ======================================
def clean_moa(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", re.sub(r"[^a-z0-9\s]", " ", text))
    return text.strip()

df["MoA_clean"] = df["MoA"].apply(clean_moa)

# ======================================
# 5. Categorization
# ======================================
def categorize_moa(text):

    text = text.lower()

    if text == "unknown":
        return "Multi-target / Polypharmacology"

    # 1 Membrane / Cell Envelope Disruption
    if any(k in text for k in [
        "membrane",
        "cell membrane",
        "pore",
        "permeability",
        "depolar",
        "lipid",
        "bilayer",
        "detergent",
        "ferriprotoporphyrin",
        "reactive nitro radical",
        "reactive intermediates",
        "membrane depolarization",
        "membrane permeabilization",
        "membrane damage",
        "membrane disrupting",
        "breaks down membrane lipids",
        "destroys cell membranes"
    ]):
        return "Membrane / Cell Envelope Disruption"

    # 2 Cell Wall Biosynthesis Inhibition
    if any(k in text for k in [
        "penicillin-binding protein",
        "pbp",
        "peptidoglycan",
        "cell wall",
        "lipid ii",
        "alanine racemase",
        "beta lactamase",
        "beta-lactamase",
        "penicillin binding protein",
        "fab i",
        "enoyl acyl carrier protein reductase"
    ]):
        return "Cell Wall Biosynthesis Inhibition"

    # 3 Protein Synthesis Inhibition
    if any(k in text for k in [
        "ribosome",
        "70s",
        "30s",
        "50s",
        "translation",
        "elongation factor",
        "trna",
        "protein synthesis",
        "trna synthetase",
        "trna ligase"
    ]):
        return "Protein Synthesis Inhibition"

    # 4 Nucleic Acid Synthesis / Integrity Disruption
    if any(k in text for k in [
        "dna",
        "rna",
        "gyrase",
        "topoisomerase",
        "polymerase",
        "replication",
        "intercalat",
        "dna binding",
        "dna disrupting",
        "dna inhibitor",
        "ribonucleotide reductase"
    ]):
        return "Nucleic Acid Synthesis / Integrity Disruption"

    # 5 Metabolic Pathway Inhibition
    if any(k in text for k in [
        "dihydrofolate reductase",
        "dihydropteroate synthase",
        "folate",
        "biosynthesis",
        "metabolism",
        "amidophosphoribosyltransferase",
        "oxidoreductase",
        "thymidylate synthase",
        "inosine monophosphate dehydrogenase",
        "dxr",
        "reductase",
        "synthase",
        "dehydrogenase",
        "cyclooxygenase",
        "carbonic anhydrase",
        "monoamine oxidase",
        "hmg coa reductase",
        "xanthine dehydrogenase",
        "caspase",
        "matrix metalloproteinase",
        "reductoisomerase",
        "dioxp reductoisomerase",
        "dxr",
        "dipeptidase",
        "diacylglycerol o acyltransferase",
        "dgat",
        "glucosyltransferase",
        "ceramide glucosyltransferase"
    ]):
        return "Metabolic Pathway Inhibition"

    # 6 Redox / Oxidative Stress Induction
    if any(k in text for k in [
        "oxidative",
        "redox",
        "ros",
        "reactive",
        "nitro",
        "oxidase",
        "reducing agent",
        "antioxidant",
        "increases ros",
        "chelating",
        "copper chelating",
        "iron chelating",
        "aluminium chelating",
        "glutathione",
        "glutathione precursor",
        "redox regulator"
    ]):
        return "Redox / Oxidative Stress Induction"

    # 7 Regulatory / Signalling Interference
    if any(k in text for k in [
        "quorum",
        "two component",
        "regulation",
        "signalling",
        "stress response",
        "transcription",
        "toll like receptor",
        "nf kb",
        "mtor",
        "map kinase",
        "cyclin dependent kinase",
        "biofilm related genes",
        "guanylate cyclase",
        "cyclase activator",
        "cyclase inhibitor",
        "cap binding complex",
        "signal transduction modulator",
        "arca",
        "arcb",
        "arcc",
        "arcd",
        "arc gene",
        "arginine deiminase pathway"
    ]):
        return "Regulatory / Signalling Interference"

    # 8 Biofilm Matrix Disruption
    if any(k in text for k in [
        "biofilm",
        "edna",
        "nuclease"
    ]):
        return "Biofilm Matrix Disruption"

    # 9 Antifungal Sterol Biosynthesis
    if any(k in text for k in [
        "cytochrome p450",
        "cyp51",
        "lanosterol",
        "ergosterol",
        "sterol biosynthesis",
        "squalene monooxygenase"
    ]):
        return "Antifungal Sterol Biosynthesis Inhibitors"

    # 10 Antiviral agents
    if any(k in text for k in [
        "hiv",
        "herpesvirus",
        "influenza",
        "hepatitis c",
        "reverse transcriptase",
        "viral protease",
        "viral capsid",
        "neuraminidase",
        "viral polymerase",
        "viral", "hepatitis", "capsid", "viral kinase", "helicase", "primase",
        "hiv protease",
        "hiv 1 protease",
        "viral protease",
        "pul97",
        "viral phosphotransferase",

        "human immunodeficiency virus type 1 protease"
    ]):
        return "Antiviral agents"

    # 11 Affecting Human Physiology
    if any(k in text for k in [
        "adrenergic receptor",
        "dopamine receptor",
        "serotonin transporter",
        "histamine receptor",
        "muscarinic receptor",
        "angiotensin receptor",
        "opioid receptor",
        "cannabinoid receptor",
        "vitamin d receptor",
        "androgen receptor",
        "estrogen receptor",
        "progesterone receptor",
        "glucocorticoid receptor",
        "receptor",
        "agonist",
        "antagonist",
        "partial agonist",
        "inverse agonist",
        "allosteric modulator",
        "allosteric antagonist",
        "positive modulator",
        "negative modulator",
        "retinoid receptor",
        "retinoic acid receptor",
        "retinoid x receptor",
        "adenosine receptor",
        "dopamine receptor",
        "serotonin receptor",
        "histamine receptor",
        "angiotensin receptor",
        "muscarinic receptor",
        "nicotinic receptor",
        "cannabinoid receptor",
        "opioid receptor",
        "purinergic receptor",
        "vanilloid receptor",
        "bile acid receptor",
        "fxr receptor",
        "peroxisome proliferator activated receptor",
        "ppar",
        "glucocorticoid receptor",
        "mineralocorticoid receptor",
        "estrogen receptor",
        "androgen receptor",
        "progesterone receptor",
        "vitamin d receptor",
        "chemokine receptor",
        "cxcr",
        "s1p receptor",
        "sphingosine receptor",
        "nmda receptor",
        "gaba receptor",
        "acetylcholine receptor",

        "ion channel",
        "channel blocker",
        "channel opener",
        "channel modulator",
        "voltage gated",
        "voltage activated",
        "calcium channel",
        "sodium channel",
        "potassium channel",
        "chloride channel",
        "anion channel",
        "herg",
        "enac",
        "ryanodine receptor",
        "l type calcium channel",
        "inwardly rectifying potassium channel",
        "calcium activated potassium channel",

        "acetylcholinesterase",
        "cholinesterase",
        "dopamine transporter",
        "serotonin transporter",
        "norepinephrine transporter",
        "gaba transporter",
        "synaptic vesicle transporter",
        "vesicular amine transporter",
        "sv2a",
        "neurotransmitter reuptake",
        "bcr abl",
        "abl kinase",
        "fusion protein inhibitor"

        "angiotensin-converting enzyme",
        "angiotensin converting enzyme",
        "ace inhibitor",
        "lipoxygenase",
        "cyclooxygenase",
        "phosphodiesterase",
        "pde inhibitor",
        "neprilysin",
        "lipase",
        "glucosidase",
        "decarboxylase",
        "monoamine oxidase",
        "thrombin",
        "farnesyltransferase",
        "kinase",
        "phosphatase",
        "ubiquitin ligase",
        "aurora kinase",
        "pdgfr",
        "pdk1",
        "ikk",

        "transporter",
        "solute carrier",
        "slc",
        "cotransporter",
        "sodium chloride cotransporter",
        "sodium potassium chloride cotransporter",
        "npc1l1",
        "amine transporter",
        "renal transporter",

        "insulin receptor",
        "hormone receptor",
        "glucose metabolism",
        "lipid metabolism",
        "fatty acid metabolism",
        "ppar alpha",
        "ppar gamma",
        "ppar delta",
        "metabolic regulator",

        "growth factor receptor",
        "egfr",
        "map kinase",
        "mtor",
        "nf kb",
        "signal transduction",
        "transcription factor",
        "cell signaling",
        "cytokine receptor",

        "tubulin",
        "microtubule",
        "cytoskeleton",
        "spindle assembly",
        "microtubule stabilizer",
        "microtubule inhibitor"
        "potassium transporting atpase",
        "npc1l1",
        "niemmann-pick c1",
        "niemann pick c1",
        "synaptic vesicle glycoprotein",
        "sv2a",
        "sv2a modulator",
        "laxative",
        "potassium-transporting atpase inhibitor",
        "potassium transporting atpase",
        "potassium atpase",
        "synaptic vesicle glycoprotein 2a",
        "cholesterol absorption inhibitor"



    ]):
        return "Affecting Human Physiology"

    # 12 Multi-target
    if any(k in text for k in [
        "multi target",
        "multiple",
        "polypharmacology",
        "pleiotropic"
    ]):
        return "Multi-target / Polypharmacology"

    return "Others"


# ======================================
# 6. Apply categorization
# ======================================
df["MoA_category"] = df["MoA_clean"].apply(categorize_moa)

# ======================================
# 7. Keep only required columns
# ======================================
df_final = df[["Name", "SMILES", "MoA_category"]]

# ======================================
# 8. Rename column
# ======================================
df_final = df_final.rename(columns={"MoA_category": "MoA"})

# ======================================
# 9. Check distribution
# ======================================
print(df["MoA_category"].value_counts())

# ======================================
# 10. Save final file
# ======================================
df_final.to_excel("final_moa_categorized.xlsx", index=False)

print(df[df["MoA_category"]=="Others"]["MoA"].unique())

# ======================================
# 10. Save categorized data into multiple sheets
# ======================================

with pd.ExcelWriter("final_moa_categorized.xlsx", engine="xlsxwriter") as writer:

    # Main categorized dataset
    df_final.to_excel(writer, sheet_name="All_Data_PA", index=False)

    # Human physiology targets
    human_targets = df[df["MoA_category"] == "Affecting Human Physiology"]
    human_targets.to_excel(writer, sheet_name="Human_Physiology_Targets", index=False)

    # Antiviral agents
    antiviral_targets = df[df["MoA_category"] == "Antiviral agents"]
    antiviral_targets.to_excel(writer, sheet_name="Antiviral_Targets", index=False)

    # Antifungal agents
    antifungal_targets = df[df["MoA_category"] == "Antifungal Sterol Biosynthesis Inhibitors"]
    antifungal_targets.to_excel(writer, sheet_name="Antifungal_Targets", index=False)

    # Others
    Others = df[df["MoA_category"] == "Others"]
    Others.to_excel(writer, sheet_name="Others", index=False)

    # ======================================
    # Direct Antibacterial (remove unwanted categories)
    # ======================================
    direct_antibacterial = df[~df["MoA_category"].isin([
        "Affecting Human Physiology",
        "Antiviral agents",
        "Antifungal Sterol Biosynthesis Inhibitors",
        "Others"
    ])]

    direct_antibacterial.to_excel(writer, sheet_name="Direct_Antibacterial", index=False)

print("Excel file created with multiple sheets including Direct_Antibacterial.")

import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

# Load dataset from the correct file and sheet
df = pd.read_excel("final_moa_categorized.xlsx", sheet_name="Direct_Antibacterial")

# Get list of all RDKit descriptors
descriptor_names = [desc[0] for desc in Descriptors.descList]

def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return [None]*len(descriptor_names)

    descriptor_values = []

    for name, function in Descriptors.descList:
        try:
            value = function(mol)
        except:
            value = None
        descriptor_values.append(value)

    return descriptor_values


# Apply descriptor calculation
descriptor_data = df["SMILES"].apply(calculate_descriptors)

# Convert to dataframe
X = pd.DataFrame(descriptor_data.tolist(), columns=descriptor_names)

# Target variable
y = df["MoA_category"]

# Combine if needed
final_df = pd.concat([X, y], axis=1)

# Save descriptor dataset
final_df.to_excel("MoA_rdkit_all_descriptors_calculated.xlsx", index=False)

print("Total descriptors extracted:", len(descriptor_names))
print("Dataset shape:", final_df.shape)


# Load Descriptor data

import pandas as pd

df = pd.read_excel("MoA_rdkit_all_descriptors_calculated.xlsx")

print("Dataset shape:", df.shape)

df.head()

# seperate descriptors and label(moa)

X = df.drop(columns=["MoA_category"])
y = df["MoA_category"]

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

# -------------------------------
#  FILTER RARE CLASSES HERE
# -------------------------------

df_model = X.copy()
df_model["MoA_category"] = y

counts = df_model["MoA_category"].value_counts()
valid_classes = counts[counts >= 3].index

df_filtered = df_model[df_model["MoA_category"].isin(valid_classes)].copy()

# Update X and y AFTER filtering
X = df_filtered.drop(columns=["MoA_category"])
y = df_filtered["MoA_category"]

print("After filtering:", X.shape, y.shape)

import numpy as np

# Replace infinity values
X = X.replace([np.inf, -np.inf], np.nan)

# Convert to float32 to catch values too large for this dtype and re-handle potential new NaNs
X = X.astype(np.float32)

# Replace any new inf values that might have been created due to float32 conversion overflow
X = X.replace([np.inf, -np.inf], np.nan)

# Fill missing values

X = X.fillna(X.median())

# Remove Extremely Large Descriptors
# Some descriptors can reach 10¹² or more, causing overflow.

X.describe().T.sort_values("max", ascending=False).head(10)

# A safe solution is to clip values.

X = X.clip(-1e6, 1e6)

# A safe solution is to clip values.

X = X.astype(np.float64)

# check missing, infinity and large values

import numpy as np

print("NaN values:", np.isnan(X).sum().sum())
print("Infinity values:", np.isinf(X).sum().sum())
print("Max value in dataset:", np.nanmax(X.values))

# Train-Test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# label encoding

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_encoded = le.fit_transform(y_train)

y_test_encoded = le.transform(y_test)

# Feature stability

from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0)

# Fit ONLY on training data
X_train_var = selector.fit_transform(X_train)

# Apply SAME transformation on test data
X_test_var = selector.transform(X_test)

# Convert to DataFrame (optional but good)
selected_columns = X_train.columns[selector.get_support()]

X_train_var = pd.DataFrame(X_train_var, columns=selected_columns)
X_test_var = pd.DataFrame(X_test_var, columns=selected_columns)

print("After removing constant features:", X_train_var.shape[1])

import numpy as np

# Step 1: Compute correlation ONLY on training data
corr_matrix = X_train_var.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]

print("Descriptors to drop:", len(to_drop))

# Step 2: Drop from TRAIN
X_train_corr = X_train_var.drop(columns=to_drop)

# Step 3: Drop SAME columns from TEST
X_test_corr = X_test_var.drop(columns=to_drop)

print("After correlation removal (train):", X_train_corr.shape)
print("After correlation removal (test):", X_test_corr.shape)

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=400,
    random_state=42
)

# ✅ Train ONLY on training data
rf.fit(X_train_corr, y_train)

# Get importance
importance = pd.Series(
    rf.feature_importances_,
    index=X_train_corr.columns
)

# Sort
importance = importance.sort_values(ascending=False)

print("Top features:\n", importance.head(30))

# Select top 100 features
top_features = importance.head(100).index

# Apply SAME features to train and test
X_train_sel = X_train_corr[top_features]
X_test_sel = X_test_corr[top_features]

print("Final train shape:", X_train_sel.shape)
print("Final test shape:", X_test_sel.shape)

# Normalization

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit ONLY on training
X_train_scaled = scaler.fit_transform(X_train_sel)

# Apply to test
X_test_scaled = scaler.transform(X_test_sel)

# Find best K

from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt

bic_scores = []

k_values = [5, 8, 10, 12, 15, 18, 20, 22]

for k in k_values:

    gmm = GaussianMixture(
        n_components=k,
        covariance_type='full',
        random_state=42
    )

    gmm.fit(X_train_scaled)

    bic = gmm.bic(X_train_scaled)

    bic_scores.append(bic)

    print(f"K = {k}, BIC = {bic}")

# Silhouette Score for best K

from sklearn.metrics import silhouette_score

for k in k_values:

    gmm = GaussianMixture(n_components=k, random_state=42)

    labels = gmm.fit_predict(X_train_scaled)

    score = silhouette_score(X_train_scaled, labels)

    print(f"K = {k}, Silhouette Score = {score}")

# Apply GMM

from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=10, random_state=42)

gmm.fit(X_train_scaled)

train_gmm = gmm.predict_proba(X_train_scaled)
test_gmm = gmm.predict_proba(X_test_scaled)

# Combine
X_train_final = np.hstack([X_train_scaled, train_gmm])
X_test_final = np.hstack([X_test_scaled, test_gmm])

# GMM probabilities for TRAIN
gmm_train_probs = gmm.predict_proba(X_train_scaled)

gmm_train_df = pd.DataFrame(
    gmm_train_probs,
    columns=[f"GMM_Cluster_{i}" for i in range(gmm_train_probs.shape[1])]
)

# Convert scaled train to DataFrame
X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=X_train_sel.columns
)

# Concatenate
X_train_enhanced = pd.concat([X_train_scaled_df, gmm_train_df], axis=1)

print("Train shape:", X_train_enhanced.shape)
print(X_train_enhanced.columns[-10:])

# GMM probabilities for TEST
gmm_test_probs = gmm.predict_proba(X_test_scaled)

gmm_test_df = pd.DataFrame(
    gmm_test_probs,
    columns=[f"GMM_Cluster_{i}" for i in range(gmm_test_probs.shape[1])]
)

# Convert scaled test to DataFrame
X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=X_train_sel.columns   # IMPORTANT: use train columns
)

# Concatenate
X_test_enhanced = pd.concat([X_test_scaled_df, gmm_test_df], axis=1)

print("Test shape:", X_test_enhanced.shape)
print(X_test_enhanced.columns[-10:])

cluster_labels = gmm.predict(X_train_scaled)

df_analysis = pd.DataFrame({
    "Cluster": cluster_labels,
    "MoA": y_train.values
})

print(df_analysis.groupby("Cluster")["MoA"].value_counts())

# ---Function for evaluating metrices---

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    classification_report
)

def evaluate_model_full(model_name, y_test, y_pred, y_prob):

    print("\n==============================")
    print("MODEL:", model_name)
    print("==============================")

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    print("\nOverall Confusion Matrix:\n")
    print(cm)

    # Plot confusion matrix
    plt.figure(figsize=(8,6))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=le.classes_,
        yticklabels=le.classes_
    )

    plt.xlabel("Predicted MoA")
    plt.ylabel("Actual MoA")
    plt.title(model_name + " Confusion Matrix")

    plt.show()

    # ------------------------------
    # TP FP FN TN PER CLASS - CALCULATE HERE
    # ------------------------------

    TP = np.diag(cm)

    FP = np.sum(cm, axis=0) - TP

    FN = np.sum(cm, axis=1) - TP

    TN = np.sum(cm) - (TP + FP + FN)

    # --------------------------------
    # PER-MoA CONFUSION MATRICES
    # --------------------------------

    per_moa_cm_list = []

    for i, moa in enumerate(le.classes_):

        tp = TP[i]
        fp = FP[i]
        fn = FN[i]
        tn = TN[i]

        cm_moa = np.array([
            [tp, fn],
            [fp, tn]
        ])

        print("\nConfusion Matrix for MoA:", moa)
        print(cm_moa)

        plt.figure(figsize=(4,4))

        sns.heatmap(
            cm_moa,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=["Predicted "+moa, "Predicted Other"],
            yticklabels=["Actual "+moa, "Actual Other"]
        )

        plt.title(model_name + " - " + moa)

        plt.show()

        per_moa_cm_list.append({
            "MoA": moa,
            "TP": tp,
            "FN": fn,
            "FP": fp,
            "TN": tn
        })

    per_moa_cm_df = pd.DataFrame(per_moa_cm_list)

    per_moa_cm_df.to_excel(model_name+"_per_moa_confusion_matrix.xlsx", index=False)

    # Sensitivity (Recall)
    sensitivity = TP / (TP + FN)

    # Specificity
    specificity = TN / (TN + FP)

    per_class_df = pd.DataFrame({
        "MoA": le.classes_,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "Sensitivity": sensitivity,
        "Specificity": specificity
    })

    print("\nPer-MoA Metrics\n")
    print(per_class_df)

    # Save results
    per_class_df.to_excel(model_name+"_per_MoA_metrics.xlsx", index=False)

    # ------------------------------
    # OVERALL METRICS
    # ------------------------------

    overall_TP = TP.sum()

    overall_FP = FP.sum()

    overall_FN = FN.sum()

    overall_TN = TN.sum()

    overall_sensitivity = overall_TP / (overall_TP + overall_FN)

    overall_specificity = overall_TN / (overall_TN + overall_FP)

    overall_df = pd.DataFrame({
        "Metric": [
            "TP",
            "FP",
            "FN",
            "TN",
            "Sensitivity",
            "Specificity"
        ],
        "Value": [
            overall_TP,
            overall_FP,
            overall_FN,
            overall_TN,
            overall_sensitivity,
            overall_specificity
        ]
    })

    print("\nOverall Metrics\n")
    print(overall_df)

    overall_df.to_excel(model_name+"_overall_metrics.xlsx", index=False)

    # ------------------------------
    # Accuracy
    # ------------------------------

    acc = accuracy_score(y_test, y_pred)

    print("\nAccuracy:", acc)

    # ------------------------------
    # MCC
    # ------------------------------

    mcc = matthews_corrcoef(y_test, y_pred)

    print("MCC:", mcc)

    # ------------------------------
    # ROC AUC (Multiclass)
    # ------------------------------

    from sklearn.preprocessing import label_binarize

    y_test_bin = label_binarize(y_test, classes=np.unique(y_test))

    auc = roc_auc_score(y_test_bin, y_prob, multi_class="ovr")

    print("ROC-AUC:", auc)

    # Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features='sqrt'
)

# ✅ Train on FINAL enhanced features (IMPORTANT)
rf_model.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_rf = rf_model.predict(X_test_enhanced)

# Accuracy
print("Random Forest Accuracy:",
      accuracy_score(y_test_encoded, y_pred_rf))

# Classification Report
print(classification_report(y_test_encoded, y_pred_rf))

# Probabilities (for ROC-AUC)
y_prob_rf = rf_model.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "Random Forest",
    y_test_encoded,
    y_pred_rf,
    y_prob_rf
)

# Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features='sqrt'
)

# ✅ Train on FINAL enhanced features (IMPORTANT)
rf_model.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_rf = rf_model.predict(X_test_enhanced)

# Accuracy
print("Random Forest Accuracy:",
      accuracy_score(y_test_encoded, y_pred_rf))

# Classification Report
print(classification_report(y_test_encoded, y_pred_rf))

# Probabilities (for ROC-AUC)
y_prob_rf = rf_model.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "Random Forest",
    y_test_encoded,
    y_pred_rf,
    y_prob_rf
)


# XGBoost

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    eval_metric="mlogloss"
)

# ✅ Train on FINAL enhanced features
xgb.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_xgb = xgb.predict(X_test_enhanced)

# Accuracy
print("XGBoost Accuracy:",
      accuracy_score(y_test_encoded, y_pred_xgb))

# Classification Report
print(classification_report(y_test_encoded, y_pred_xgb))

# Probabilities (for ROC-AUC)
y_prob_xgb = xgb.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "XGBoost",
    y_test_encoded,
    y_pred_xgb,
    y_prob_xgb
)




k = 12 f = 100

In [ ]:
import pandas as pd
import re

# ======================================
# 1. Load the Excel file
# ======================================
df = pd.read_excel("final_moa_cateogrization_Peptides+Smiles.xlsx")

# ======================================
# 2. Remove duplicate rows
# ======================================
df = df.drop_duplicates()

# ======================================
# 3. Fill missing MoA with "Unknown"
# ======================================
df["MoA"] = df["MoA"].fillna("Unknown")

# ======================================
# 4. Drop MoA with "Unknown"
# ======================================
df = df[df["MoA"] != "Unknown"]

# ======================================
# 4. Clean MoA text
# ======================================
def clean_moa(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", re.sub(r"[^a-z0-9\s]", " ", text))
    return text.strip()

df["MoA_clean"] = df["MoA"].apply(clean_moa)

# ======================================
# 5. Categorization
# ======================================
def categorize_moa(text):

    text = text.lower()

    if text == "unknown":
        return "Multi-target / Polypharmacology"

    # 1 Membrane / Cell Envelope Disruption
    if any(k in text for k in [
        "membrane",
        "cell membrane",
        "pore",
        "permeability",
        "depolar",
        "lipid",
        "bilayer",
        "detergent",
        "ferriprotoporphyrin",
        "reactive nitro radical",
        "reactive intermediates",
        "membrane depolarization",
        "membrane permeabilization",
        "membrane damage",
        "membrane disrupting",
        "breaks down membrane lipids",
        "destroys cell membranes"
    ]):
        return "Membrane / Cell Envelope Disruption"

    # 2 Cell Wall Biosynthesis Inhibition
    if any(k in text for k in [
        "penicillin-binding protein",
        "pbp",
        "peptidoglycan",
        "cell wall",
        "lipid ii",
        "alanine racemase",
        "beta lactamase",
        "beta-lactamase",
        "penicillin binding protein",
        "fab i",
        "enoyl acyl carrier protein reductase"
    ]):
        return "Cell Wall Biosynthesis Inhibition"

    # 3 Protein Synthesis Inhibition
    if any(k in text for k in [
        "ribosome",
        "70s",
        "30s",
        "50s",
        "translation",
        "elongation factor",
        "trna",
        "protein synthesis",
        "trna synthetase",
        "trna ligase"
    ]):
        return "Protein Synthesis Inhibition"

    # 4 Nucleic Acid Synthesis / Integrity Disruption
    if any(k in text for k in [
        "dna",
        "rna",
        "gyrase",
        "topoisomerase",
        "polymerase",
        "replication",
        "intercalat",
        "dna binding",
        "dna disrupting",
        "dna inhibitor",
        "ribonucleotide reductase"
    ]):
        return "Nucleic Acid Synthesis / Integrity Disruption"

    # 5 Metabolic Pathway Inhibition
    if any(k in text for k in [
        "dihydrofolate reductase",
        "dihydropteroate synthase",
        "folate",
        "biosynthesis",
        "metabolism",
        "amidophosphoribosyltransferase",
        "oxidoreductase",
        "thymidylate synthase",
        "inosine monophosphate dehydrogenase",
        "dxr",
        "reductase",
        "synthase",
        "dehydrogenase",
        "cyclooxygenase",
        "carbonic anhydrase",
        "monoamine oxidase",
        "hmg coa reductase",
        "xanthine dehydrogenase",
        "caspase",
        "matrix metalloproteinase",
        "reductoisomerase",
        "dioxp reductoisomerase",
        "dxr",
        "dipeptidase",
        "diacylglycerol o acyltransferase",
        "dgat",
        "glucosyltransferase",
        "ceramide glucosyltransferase"
    ]):
        return "Metabolic Pathway Inhibition"

    # 6 Redox / Oxidative Stress Induction
    if any(k in text for k in [
        "oxidative",
        "redox",
        "ros",
        "reactive",
        "nitro",
        "oxidase",
        "reducing agent",
        "antioxidant",
        "increases ros",
        "chelating",
        "copper chelating",
        "iron chelating",
        "aluminium chelating",
        "glutathione",
        "glutathione precursor",
        "redox regulator"
    ]):
        return "Redox / Oxidative Stress Induction"

    # 7 Regulatory / Signalling Interference
    if any(k in text for k in [
        "quorum",
        "two component",
        "regulation",
        "signalling",
        "stress response",
        "transcription",
        "toll like receptor",
        "nf kb",
        "mtor",
        "map kinase",
        "cyclin dependent kinase",
        "biofilm related genes",
        "guanylate cyclase",
        "cyclase activator",
        "cyclase inhibitor",
        "cap binding complex",
        "signal transduction modulator",
        "arca",
        "arcb",
        "arcc",
        "arcd",
        "arc gene",
        "arginine deiminase pathway"
    ]):
        return "Regulatory / Signalling Interference"

    # 8 Biofilm Matrix Disruption
    if any(k in text for k in [
        "biofilm",
        "edna",
        "nuclease"
    ]):
        return "Biofilm Matrix Disruption"

    # 9 Antifungal Sterol Biosynthesis
    if any(k in text for k in [
        "cytochrome p450",
        "cyp51",
        "lanosterol",
        "ergosterol",
        "sterol biosynthesis",
        "squalene monooxygenase"
    ]):
        return "Antifungal Sterol Biosynthesis Inhibitors"

    # 10 Antiviral agents
    if any(k in text for k in [
        "hiv",
        "herpesvirus",
        "influenza",
        "hepatitis c",
        "reverse transcriptase",
        "viral protease",
        "viral capsid",
        "neuraminidase",
        "viral polymerase",
        "viral", "hepatitis", "capsid", "viral kinase", "helicase", "primase",
        "hiv protease",
        "hiv 1 protease",
        "viral protease",
        "pul97",
        "viral phosphotransferase",
        "human immunodeficiency virus type 1 protease"
    ]):
        return "Antiviral agents"

    # 11 Affecting Human Physiology
    if any(k in text for k in [
        "adrenergic receptor",
        "dopamine receptor",
        "serotonin transporter",
        "histamine receptor",
        "muscarinic receptor",
        "angiotensin receptor",
        "opioid receptor",
        "cannabinoid receptor",
        "vitamin d receptor",
        "androgen receptor",
        "estrogen receptor",
        "progesterone receptor",
        "glucocorticoid receptor",
        "receptor",
        "agonist",
        "antagonist",
        "partial agonist",
        "inverse agonist",
        "allosteric modulator",
        "allosteric antagonist",
        "positive modulator",
        "negative modulator",
        "retinoid receptor",
        "retinoic acid receptor",
        "retinoid x receptor",
        "adenosine receptor",
        "dopamine receptor",
        "serotonin receptor",
        "histamine receptor",
        "angiotensin receptor",
        "muscarinic receptor",
        "nicotinic receptor",
        "cannabinoid receptor",
        "opioid receptor",
        "purinergic receptor",
        "vanilloid receptor",
        "bile acid receptor",
        "fxr receptor",
        "peroxisome proliferator activated receptor",
        "ppar",
        "glucocorticoid receptor",
        "mineralocorticoid receptor",
        "estrogen receptor",
        "androgen receptor",
        "progesterone receptor",
        "vitamin d receptor",
        "chemokine receptor",
        "cxcr",
        "s1p receptor",
        "sphingosine receptor",
        "nmda receptor",
        "gaba receptor",
        "acetylcholine receptor",

        "ion channel",
        "channel blocker",
        "channel opener",
        "channel modulator",
        "voltage gated",
        "voltage activated",
        "calcium channel",
        "sodium channel",
        "potassium channel",
        "chloride channel",
        "anion channel",
        "herg",
        "enac",
        "ryanodine receptor",
        "l type calcium channel",
        "inwardly rectifying potassium channel",
        "calcium activated potassium channel",

        "acetylcholinesterase",
        "cholinesterase",
        "dopamine transporter",
        "serotonin transporter",
        "norepinephrine transporter",
        "gaba transporter",
        "synaptic vesicle transporter",
        "vesicular amine transporter",
        "sv2a",
        "neurotransmitter reuptake",
        "bcr abl",
        "abl kinase",
        "fusion protein inhibitor"

        "angiotensin-converting enzyme",
        "angiotensin converting enzyme",
        "ace inhibitor",
        "lipoxygenase",
        "cyclooxygenase",
        "phosphodiesterase",
        "pde inhibitor",
        "neprilysin",
        "lipase",
        "glucosidase",
        "decarboxylase",
        "monoamine oxidase",
        "thrombin",
        "farnesyltransferase",
        "kinase",
        "phosphatase",
        "ubiquitin ligase",
        "aurora kinase",
        "pdgfr",
        "pdk1",
        "ikk",

        "transporter",
        "solute carrier",
        "slc",
        "cotransporter",
        "sodium chloride cotransporter",
        "sodium potassium chloride cotransporter",
        "npc1l1",
        "amine transporter",
        "renal transporter",

        "insulin receptor",
        "hormone receptor",
        "glucose metabolism",
        "lipid metabolism",
        "fatty acid metabolism",
        "ppar alpha",
        "ppar gamma",
        "ppar delta",
        "metabolic regulator",

        "growth factor receptor",
        "egfr",
        "map kinase",
        "mtor",
        "nf kb",
        "signal transduction",
        "transcription factor",
        "cell signaling",
        "cytokine receptor",

        "tubulin",
        "microtubule",
        "cytoskeleton",
        "spindle assembly",
        "microtubule stabilizer",
        "microtubule inhibitor"
        "potassium transporting atpase",
        "npc1l1",
        "niemmann-pick c1",
        "niemann pick c1",
        "synaptic vesicle glycoprotein",
        "sv2a",
        "sv2a modulator",
        "laxative",
        "potassium-transporting atpase inhibitor",
        "potassium transporting atpase",
        "potassium atpase",
        "synaptic vesicle glycoprotein 2a",
        "cholesterol absorption inhibitor"



    ]):
        return "Affecting Human Physiology"

    # 12 Multi-target
    if any(k in text for k in [
        "multi target",
        "multiple",
        "polypharmacology",
        "pleiotropic"
    ]):
        return "Multi-target / Polypharmacology"

    return "Others"


# ======================================
# 6. Apply categorization
# ======================================
df["MoA_category"] = df["MoA_clean"].apply(categorize_moa)

# ======================================
# 7. Keep only required columns
# ======================================
df_final = df[["Name", "SMILES", "MoA_category"]]

# ======================================
# 8. Rename column
# ======================================
df_final = df_final.rename(columns={"MoA_category": "MoA"})

# ======================================
# 9. Check distribution
# ======================================
print(df["MoA_category"].value_counts())

# ======================================
# 10. Save final file
# ======================================
df_final.to_excel("final_moa_categorized.xlsx", index=False)

print(df[df["MoA_category"]=="Others"]["MoA"].unique())

# ======================================
# 10. Save categorized data into multiple sheets
# ======================================

with pd.ExcelWriter("final_moa_categorized.xlsx", engine="xlsxwriter") as writer:

    # Main categorized dataset
    df_final.to_excel(writer, sheet_name="All_Data_PA", index=False)

    # Human physiology targets
    human_targets = df[df["MoA_category"] == "Affecting Human Physiology"]
    human_targets.to_excel(writer, sheet_name="Human_Physiology_Targets", index=False)

    # Antiviral agents
    antiviral_targets = df[df["MoA_category"] == "Antiviral agents"]
    antiviral_targets.to_excel(writer, sheet_name="Antiviral_Targets", index=False)

    # Antifungal agents
    antifungal_targets = df[df["MoA_category"] == "Antifungal Sterol Biosynthesis Inhibitors"]
    antifungal_targets.to_excel(writer, sheet_name="Antifungal_Targets", index=False)

    # Others
    Others = df[df["MoA_category"] == "Others"]
    Others.to_excel(writer, sheet_name="Others", index=False)

    # ======================================
    # Direct Antibacterial (remove unwanted categories)
    # ======================================
    direct_antibacterial = df[~df["MoA_category"].isin([
        "Affecting Human Physiology",
        "Antiviral agents",
        "Antifungal Sterol Biosynthesis Inhibitors",
        "Others"
    ])]

    direct_antibacterial.to_excel(writer, sheet_name="Direct_Antibacterial", index=False)

print("Excel file created with multiple sheets including Direct_Antibacterial.")

import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

# Load dataset from the correct file and sheet
df = pd.read_excel("final_moa_categorized.xlsx", sheet_name="Direct_Antibacterial")

# Get list of all RDKit descriptors
descriptor_names = [desc[0] for desc in Descriptors.descList]

def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return [None]*len(descriptor_names)

    descriptor_values = []

    for name, function in Descriptors.descList:
        try:
            value = function(mol)
        except:
            value = None
        descriptor_values.append(value)

    return descriptor_values


# Apply descriptor calculation
descriptor_data = df["SMILES"].apply(calculate_descriptors)

# Convert to dataframe
X = pd.DataFrame(descriptor_data.tolist(), columns=descriptor_names)

# Target variable
y = df["MoA_category"]

# Combine if needed
final_df = pd.concat([X, y], axis=1)

# Save descriptor dataset
final_df.to_excel("MoA_rdkit_all_descriptors_calculated.xlsx", index=False)

print("Total descriptors extracted:", len(descriptor_names))
print("Dataset shape:", final_df.shape)


# Load Descriptor data

import pandas as pd

df = pd.read_excel("MoA_rdkit_all_descriptors_calculated.xlsx")

print("Dataset shape:", df.shape)

df.head()

# seperate descriptors and label(moa)

X = df.drop(columns=["MoA_category"])
y = df["MoA_category"]

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

# -------------------------------
#  FILTER RARE CLASSES HERE
# -------------------------------

df_model = X.copy()
df_model["MoA_category"] = y

counts = df_model["MoA_category"].value_counts()
valid_classes = counts[counts >= 3].index

df_filtered = df_model[df_model["MoA_category"].isin(valid_classes)].copy()

# Update X and y AFTER filtering
X = df_filtered.drop(columns=["MoA_category"])
y = df_filtered["MoA_category"]

print("After filtering:", X.shape, y.shape)

import numpy as np

# Replace infinity values
X = X.replace([np.inf, -np.inf], np.nan)

# Convert to float32 to catch values too large for this dtype and re-handle potential new NaNs
X = X.astype(np.float32)

# Replace any new inf values that might have been created due to float32 conversion overflow
X = X.replace([np.inf, -np.inf], np.nan)

# Fill missing values

X = X.fillna(X.median())

# Remove Extremely Large Descriptors
# Some descriptors can reach 10¹² or more, causing overflow.

X.describe().T.sort_values("max", ascending=False).head(10)

# A safe solution is to clip values.

X = X.clip(-1e6, 1e6)

# A safe solution is to clip values.

X = X.astype(np.float64)

# check missing, infinity and large values

import numpy as np

print("NaN values:", np.isnan(X).sum().sum())
print("Infinity values:", np.isinf(X).sum().sum())
print("Max value in dataset:", np.nanmax(X.values))

# Train-Test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# label encoding

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_encoded = le.fit_transform(y_train)

y_test_encoded = le.transform(y_test)

# Feature stability

from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0)

# Fit ONLY on training data
X_train_var = selector.fit_transform(X_train)

# Apply SAME transformation on test data
X_test_var = selector.transform(X_test)

# Convert to DataFrame (optional but good)
selected_columns = X_train.columns[selector.get_support()]

X_train_var = pd.DataFrame(X_train_var, columns=selected_columns)
X_test_var = pd.DataFrame(X_test_var, columns=selected_columns)

print("After removing constant features:", X_train_var.shape[1])

import numpy as np

# Step 1: Compute correlation ONLY on training data
corr_matrix = X_train_var.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]

print("Descriptors to drop:", len(to_drop))

# Step 2: Drop from TRAIN
X_train_corr = X_train_var.drop(columns=to_drop)

# Step 3: Drop SAME columns from TEST
X_test_corr = X_test_var.drop(columns=to_drop)

print("After correlation removal (train):", X_train_corr.shape)
print("After correlation removal (test):", X_test_corr.shape)

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=400,
    random_state=42
)

# ✅ Train ONLY on training data
rf.fit(X_train_corr, y_train)

# Get importance
importance = pd.Series(
    rf.feature_importances_,
    index=X_train_corr.columns
)

# Sort
importance = importance.sort_values(ascending=False)

print("Top features:\n", importance.head(30))

# Select top 100 features
top_features = importance.head(100).index

# Apply SAME features to train and test
X_train_sel = X_train_corr[top_features]
X_test_sel = X_test_corr[top_features]

print("Final train shape:", X_train_sel.shape)
print("Final test shape:", X_test_sel.shape)

# Normalization

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit ONLY on training
X_train_scaled = scaler.fit_transform(X_train_sel)

# Apply to test
X_test_scaled = scaler.transform(X_test_sel)

# Find best K

from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt

bic_scores = []

k_values = [5, 8, 10, 12, 15, 18, 20, 22]

for k in k_values:

    gmm = GaussianMixture(
        n_components=k,
        covariance_type='full',
        random_state=42
    )

    gmm.fit(X_train_scaled)

    bic = gmm.bic(X_train_scaled)

    bic_scores.append(bic)

    print(f"K = {k}, BIC = {bic}")

# Silhouette Score for best K

from sklearn.metrics import silhouette_score

for k in k_values:

    gmm = GaussianMixture(n_components=k, random_state=42)

    labels = gmm.fit_predict(X_train_scaled)

    score = silhouette_score(X_train_scaled, labels)

    print(f"K = {k}, Silhouette Score = {score}")

# Apply GMM

from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=12, random_state=42)

gmm.fit(X_train_scaled)

train_gmm = gmm.predict_proba(X_train_scaled)
test_gmm = gmm.predict_proba(X_test_scaled)

# Combine
X_train_final = np.hstack([X_train_scaled, train_gmm])
X_test_final = np.hstack([X_test_scaled, test_gmm])

# GMM probabilities for TRAIN
gmm_train_probs = gmm.predict_proba(X_train_scaled)

gmm_train_df = pd.DataFrame(
    gmm_train_probs,
    columns=[f"GMM_Cluster_{i}" for i in range(gmm_train_probs.shape[1])]
)

# Convert scaled train to DataFrame
X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=X_train_sel.columns
)

# Concatenate
X_train_enhanced = pd.concat([X_train_scaled_df, gmm_train_df], axis=1)

print("Train shape:", X_train_enhanced.shape)
print(X_train_enhanced.columns[-10:])

# GMM probabilities for TEST
gmm_test_probs = gmm.predict_proba(X_test_scaled)

gmm_test_df = pd.DataFrame(
    gmm_test_probs,
    columns=[f"GMM_Cluster_{i}" for i in range(gmm_test_probs.shape[1])]
)

# Convert scaled test to DataFrame
X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=X_train_sel.columns   # IMPORTANT: use train columns
)

# Concatenate
X_test_enhanced = pd.concat([X_test_scaled_df, gmm_test_df], axis=1)

print("Test shape:", X_test_enhanced.shape)
print(X_test_enhanced.columns[-10:])

cluster_labels = gmm.predict(X_train_scaled)

df_analysis = pd.DataFrame({
    "Cluster": cluster_labels,
    "MoA": y_train.values
})

print(df_analysis.groupby("Cluster")["MoA"].value_counts())

# ---Function for evaluating metrices---

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    classification_report
)

def evaluate_model_full(model_name, y_test, y_pred, y_prob):

    print("\n==============================")
    print("MODEL:", model_name)
    print("==============================")

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    print("\nOverall Confusion Matrix:\n")
    print(cm)

    # Plot confusion matrix
    plt.figure(figsize=(8,6))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=le.classes_,
        yticklabels=le.classes_
    )

    plt.xlabel("Predicted MoA")
    plt.ylabel("Actual MoA")
    plt.title(model_name + " Confusion Matrix")

    plt.show()

    # ------------------------------
    # TP FP FN TN PER CLASS - CALCULATE HERE
    # ------------------------------

    TP = np.diag(cm)

    FP = np.sum(cm, axis=0) - TP

    FN = np.sum(cm, axis=1) - TP

    TN = np.sum(cm) - (TP + FP + FN)

    # --------------------------------
    # PER-MoA CONFUSION MATRICES
    # --------------------------------

    per_moa_cm_list = []

    for i, moa in enumerate(le.classes_):

        tp = TP[i]
        fp = FP[i]
        fn = FN[i]
        tn = TN[i]

        cm_moa = np.array([
            [tp, fn],
            [fp, tn]
        ])

        print("\nConfusion Matrix for MoA:", moa)
        print(cm_moa)

        plt.figure(figsize=(4,4))

        sns.heatmap(
            cm_moa,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=["Predicted "+moa, "Predicted Other"],
            yticklabels=["Actual "+moa, "Actual Other"]
        )

        plt.title(model_name + " - " + moa)

        plt.show()

        per_moa_cm_list.append({
            "MoA": moa,
            "TP": tp,
            "FN": fn,
            "FP": fp,
            "TN": tn
        })

    per_moa_cm_df = pd.DataFrame(per_moa_cm_list)

    per_moa_cm_df.to_excel(model_name+"_per_moa_confusion_matrix.xlsx", index=False)

    # Sensitivity (Recall)
    sensitivity = TP / (TP + FN)

    # Specificity
    specificity = TN / (TN + FP)

    per_class_df = pd.DataFrame({
        "MoA": le.classes_,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "Sensitivity": sensitivity,
        "Specificity": specificity
    })

    print("\nPer-MoA Metrics\n")
    print(per_class_df)

    # Save results
    per_class_df.to_excel(model_name+"_per_MoA_metrics.xlsx", index=False)

    # ------------------------------
    # OVERALL METRICS
    # ------------------------------

    overall_TP = TP.sum()

    overall_FP = FP.sum()

    overall_FN = FN.sum()

    overall_TN = TN.sum()

    overall_sensitivity = overall_TP / (overall_TP + overall_FN)

    overall_specificity = overall_TN / (overall_TN + overall_FP)

    overall_df = pd.DataFrame({
        "Metric": [
            "TP",
            "FP",
            "FN",
            "TN",
            "Sensitivity",
            "Specificity"
        ],
        "Value": [
            overall_TP,
            overall_FP,
            overall_FN,
            overall_TN,
            overall_sensitivity,
            overall_specificity
        ]
    })

    print("\nOverall Metrics\n")
    print(overall_df)

    overall_df.to_excel(model_name+"_overall_metrics.xlsx", index=False)

    # ------------------------------
    # Accuracy
    # ------------------------------

    acc = accuracy_score(y_test, y_pred)

    print("\nAccuracy:", acc)

    # ------------------------------
    # MCC
    # ------------------------------

    mcc = matthews_corrcoef(y_test, y_pred)

    print("MCC:", mcc)

    # ------------------------------
    # ROC AUC (Multiclass)
    # ------------------------------

    from sklearn.preprocessing import label_binarize

    y_test_bin = label_binarize(y_test, classes=np.unique(y_test))

    auc = roc_auc_score(y_test_bin, y_prob, multi_class="ovr")

    print("ROC-AUC:", auc)

    # Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features='sqrt'
)

# ✅ Train on FINAL enhanced features (IMPORTANT)
rf_model.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_rf = rf_model.predict(X_test_enhanced)

# Accuracy
print("Random Forest Accuracy:",
      accuracy_score(y_test_encoded, y_pred_rf))

# Classification Report
print(classification_report(y_test_encoded, y_pred_rf))

# Probabilities (for ROC-AUC)
y_prob_rf = rf_model.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "Random Forest",
    y_test_encoded,
    y_pred_rf,
    y_prob_rf
)

# Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features='sqrt'
)

# ✅ Train on FINAL enhanced features (IMPORTANT)
rf_model.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_rf = rf_model.predict(X_test_enhanced)

# Accuracy
print("Random Forest Accuracy:",
      accuracy_score(y_test_encoded, y_pred_rf))

# Classification Report
print(classification_report(y_test_encoded, y_pred_rf))

# Probabilities (for ROC-AUC)
y_prob_rf = rf_model.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "Random Forest",
    y_test_encoded,
    y_pred_rf,
    y_prob_rf
)


# XGBoost

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    eval_metric="mlogloss"
)

# ✅ Train on FINAL enhanced features
xgb.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_xgb = xgb.predict(X_test_enhanced)

# Accuracy
print("XGBoost Accuracy:",
      accuracy_score(y_test_encoded, y_pred_xgb))

# Classification Report
print(classification_report(y_test_encoded, y_pred_xgb))

# Probabilities (for ROC-AUC)
y_prob_xgb = xgb.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "XGBoost",
    y_test_encoded,
    y_pred_xgb,
    y_prob_xgb
)




k = 10 f = 125

In [ ]:
import pandas as pd
import re

# ======================================
# 1. Load the Excel file
# ======================================
df = pd.read_excel("final_moa_cateogrization_Peptides+Smiles.xlsx")

# ======================================
# 2. Remove duplicate rows
# ======================================
df = df.drop_duplicates()

# ======================================
# 3. Fill missing MoA with "Unknown"
# ======================================
df["MoA"] = df["MoA"].fillna("Unknown")

# ======================================
# 4. Drop MoA with "Unknown"
# ======================================
df = df[df["MoA"] != "Unknown"]

# ======================================
# 4. Clean MoA text
# ======================================
def clean_moa(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", re.sub(r"[^a-z0-9\s]", " ", text))
    return text.strip()

df["MoA_clean"] = df["MoA"].apply(clean_moa)

# ======================================
# 5. Categorization
# ======================================
def categorize_moa(text):

    text = text.lower()

    if text == "unknown":
        return "Multi-target / Polypharmacology"

    # 1 Membrane / Cell Envelope Disruption
    if any(k in text for k in [
        "membrane",
        "cell membrane",
        "pore",
        "permeability",
        "depolar",
        "lipid",
        "bilayer",
        "detergent",
        "ferriprotoporphyrin",
        "reactive nitro radical",
        "reactive intermediates",
        "membrane depolarization",
        "membrane permeabilization",
        "membrane damage",
        "membrane disrupting",
        "breaks down membrane lipids",
        "destroys cell membranes"
    ]):
        return "Membrane / Cell Envelope Disruption"

    # 2 Cell Wall Biosynthesis Inhibition
    if any(k in text for k in [
        "penicillin-binding protein",
        "pbp",
        "peptidoglycan",
        "cell wall",
        "lipid ii",
        "alanine racemase",
        "beta lactamase",
        "beta-lactamase",
        "penicillin binding protein",
        "fab i",
        "enoyl acyl carrier protein reductase"
    ]):
        return "Cell Wall Biosynthesis Inhibition"

    # 3 Protein Synthesis Inhibition
    if any(k in text for k in [
        "ribosome",
        "70s",
        "30s",
        "50s",
        "translation",
        "elongation factor",
        "trna",
        "protein synthesis",
        "trna synthetase",
        "trna ligase"
    ]):
        return "Protein Synthesis Inhibition"

    # 4 Nucleic Acid Synthesis / Integrity Disruption
    if any(k in text for k in [
        "dna",
        "rna",
        "gyrase",
        "topoisomerase",
    ]):
        return "Nucleic Acid Synthesis / Integrity Disruption"

    if any(k in text for k in [
        "dihydrofolate reductase",
        "dihydropteroate synthase",
        "folate",
        "biosynthesis",
        "metabolism",
        "amidophosphoribosyltransferase",
        "oxidoreductase",
        "thymidylate synthase",
        "inosine monophosphate dehydrogenase",
        "dxr",
        "reductase",
        "synthase",
        "dehydrogenase",
        "cyclooxygenase",
        "carbonic anhydrase",
        "monoamine oxidase",
        "hmg coa reductase",
        "xanthine dehydrogenase",
        "caspase",
        "matrix metalloproteinase",
        "reductoisomerase",
        "dioxp reductoisomerase",
        "dxr",
        "dipeptidase",
        "diacylglycerol o acyltransferase",
        "dgat",
        "glucosyltransferase",
        "ceramide glucosyltransferase"
    ]):
        return "Metabolic Pathway Inhibition"

    # 6 Redox / Oxidative Stress Induction
    if any(k in text for k in [
        "oxidative",
        "redox",
        "ros",
        "reactive",
        "nitro",
        "oxidase",
        "reducing agent",
        "antioxidant",
        "increases ros",
        "chelating",
        "copper chelating",
        "iron chelating",
        "aluminium chelating",
        "glutathione",
        "glutathione precursor",
        "redox regulator"
    ]):
        return "Redox / Oxidative Stress Induction"

    # 7 Regulatory / Signalling Interference
    if any(k in text for k in [
        "quorum",
        "two component",
        "regulation",
        "signalling",
        "stress response",
        "transcription",
        "toll like receptor",
        "nf kb",
        "mtor",
        "map kinase",
        "cyclin dependent kinase",
        "biofilm related genes",
        "guanylate cyclase",
        "cyclase activator",
        "cyclase inhibitor",
        "cap binding complex",
        "signal transduction modulator",
        "arca",
        "arcb",
        "arcc",
        "arcd",
        "arc gene",
        "arginine deiminase pathway"
    ]):
        return "Regulatory / Signalling Interference"

    # 8 Biofilm Matrix Disruption
    if any(k in text for k in [
        "biofilm",
        "edna",
        "nuclease"
    ]):
        return "Biofilm Matrix Disruption"

    # 9 Antifungal Sterol Biosynthesis
    if any(k in text for k in [
        "cytochrome p450",
        "cyp51",
        "lanosterol",
        "ergosterol",
        "sterol biosynthesis",
        "squalene monooxygenase"
    ]):
        return "Antifungal Sterol Biosynthesis Inhibitors"

    # 10 Antiviral agents
    if any(k in text for k in [
        "hiv",
        "herpesvirus",
        "influenza",
        "hepatitis c",
        "reverse transcriptase",
        "viral protease",
        "viral capsid",
        "neuraminidase",
        "viral polymerase",
        "viral", "hepatitis", "capsid", "viral kinase", "helicase", "primase",
        "hiv protease",
        "hiv 1 protease",
        "viral protease",
        "pul97",
        "viral phosphotransferase",
        "human immunodeficiency virus type 1 protease"
    ]):
        return "Antiviral agents"

    # 11 Affecting Human Physiology
    if any(k in text for k in [
        "adrenergic receptor",
        "dopamine receptor",
        "serotonin transporter",
        "histamine receptor",
        "muscarinic receptor",
        "angiotensin receptor",
        "opioid receptor",
        "cannabinoid receptor",
        "vitamin d receptor",
        "androgen receptor",
        "estrogen receptor",
        "progesterone receptor",
        "glucocorticoid receptor",
        "receptor",
        "agonist",
        "antagonist",
        "partial agonist",
        "inverse agonist",
        "allosteric modulator",
        "allosteric antagonist",
        "positive modulator",
        "negative modulator",
        "retinoid receptor",
        "retinoic acid receptor",
        "retinoid x receptor",
        "adenosine receptor",
        "dopamine receptor",
        "serotonin receptor",
        "histamine receptor",
        "angiotensin receptor",
        "muscarinic receptor",
        "nicotinic receptor",
        "cannabinoid receptor",
        "opioid receptor",
        "purinergic receptor",
        "vanilloid receptor",
        "bile acid receptor",
        "fxr receptor",
        "peroxisome proliferator activated receptor",
        "ppar",
        "glucocorticoid receptor",
        "mineralocorticoid receptor",
        "estrogen receptor",
        "androgen receptor",
        "progesterone receptor",
        "vitamin d receptor",
        "chemokine receptor",
        "cxcr",
        "s1p receptor",
        "sphingosine receptor",
        "nmda receptor",
        "gaba receptor",
        "acetylcholine receptor",

        "ion channel",
        "channel blocker",
        "channel opener",
        "channel modulator",
        "voltage gated",
        "voltage activated",
        "calcium channel",
        "sodium channel",
        "potassium channel",
        "chloride channel",
        "anion channel",
        "herg",
        "enac",
        "ryanodine receptor",
        "l type calcium channel",
        "inwardly rectifying potassium channel",
        "calcium activated potassium channel",

        "acetylcholinesterase",
        "cholinesterase",
        "dopamine transporter",
        "serotonin transporter",
        "norepinephrine transporter",
        "gaba transporter",
        "synaptic vesicle transporter",
        "vesicular amine transporter",
        "sv2a",
        "neurotransmitter reuptake",
        "bcr abl",
        "abl kinase",
        "fusion protein inhibitor"

        "angiotensin-converting enzyme",
        "angiotensin converting enzyme",
        "ace inhibitor",
        "lipoxygenase",
        "cyclooxygenase",
        "phosphodiesterase",
        "pde inhibitor",
        "neprilysin",
        "lipase",
        "glucosidase",
        "decarboxylase",
        "monoamine oxidase",
        "thrombin",
        "farnesyltransferase",
        "kinase",
        "phosphatase",
        "ubiquitin ligase",
        "aurora kinase",
        "pdgfr",
        "pdk1",
        "ikk",

        "transporter",
        "solute carrier",
        "slc",
        "cotransporter",
        "sodium chloride cotransporter",
        "sodium potassium chloride cotransporter",
        "npc1l1",
        "amine transporter",
        "renal transporter",

        "insulin receptor",
        "hormone receptor",
        "glucose metabolism",
        "lipid metabolism",
        "fatty acid metabolism",
        "ppar alpha",
        "ppar gamma",
        "ppar delta",
        "metabolic regulator",

        "growth factor receptor",
        "egfr",
        "map kinase",
        "mtor",
        "nf kb",
        "signal transduction",
        "transcription factor",
        "cell signaling",
        "cytokine receptor",

        "tubulin",
        "microtubule",
        "cytoskeleton",
        "spindle assembly",
        "microtubule stabilizer",
        "microtubule inhibitor"
        "potassium transporting atpase",
        "npc1l1",
        "niemmann-pick c1",
        "niemann pick c1",
        "synaptic vesicle glycoprotein",
        "sv2a",
        "sv2a modulator",
        "laxative",
        "potassium-transporting atpase inhibitor",
        "potassium transporting atpase",
        "potassium atpase",
        "synaptic vesicle glycoprotein 2a",
        "cholesterol absorption inhibitor"



    ]):
        return "Affecting Human Physiology"

    # 12 Multi-target
    if any(k in text for k in [
        "multi target",
        "multiple",
        "polypharmacology",
        "pleiotropic"
    ]):
        return "Multi-target / Polypharmacology"

    return "Others"


# ======================================
# 6. Apply categorization
# ======================================
df["MoA_category"] = df["MoA_clean"].apply(categorize_moa)

# ======================================
# 7. Keep only required columns
# ======================================
df_final = df[["Name", "SMILES", "MoA_category"]]

# ======================================
# 8. Rename column
# ======================================
df_final = df_final.rename(columns={"MoA_category": "MoA"})

# ======================================
# 9. Check distribution
# ======================================
print(df["MoA_category"].value_counts())

# ======================================
# 10. Save final file
# ======================================
df_final.to_excel("final_moa_categorized.xlsx", index=False)

print(df[df["MoA_category"]=="Others"]["MoA"].unique())

# ======================================
# 10. Save categorized data into multiple sheets
# ======================================

with pd.ExcelWriter("final_moa_categorized.xlsx", engine="xlsxwriter") as writer:

    # Main categorized dataset
    df_final.to_excel(writer, sheet_name="All_Data_PA", index=False)

    # Human physiology targets
    human_targets = df[df["MoA_category"] == "Affecting Human Physiology"]
    human_targets.to_excel(writer, sheet_name="Human_Physiology_Targets", index=False)

    # Antiviral agents
    antiviral_targets = df[df["MoA_category"] == "Antiviral agents"]
    antiviral_targets.to_excel(writer, sheet_name="Antiviral_Targets", index=False)

    # Antifungal agents
    antifungal_targets = df[df["MoA_category"] == "Antifungal Sterol Biosynthesis Inhibitors"]
    antifungal_targets.to_excel(writer, sheet_name="Antifungal_Targets", index=False)

    # Others
    Others = df[df["MoA_category"] == "Others"]
    Others.to_excel(writer, sheet_name="Others", index=False)

    # ======================================
    # Direct Antibacterial (remove unwanted categories)
    # ======================================
    direct_antibacterial = df[~df["MoA_category"].isin([
        "Affecting Human Physiology",
        "Antiviral agents",
        "Antifungal Sterol Biosynthesis Inhibitors",
        "Others"
    ])]

    direct_antibacterial.to_excel(writer, sheet_name="Direct_Antibacterial", index=False)

print("Excel file created with multiple sheets including Direct_Antibacterial.")

import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

# Load dataset from the correct file and sheet
df = pd.read_excel("final_moa_categorized.xlsx", sheet_name="Direct_Antibacterial")

# Get list of all RDKit descriptors
descriptor_names = [desc[0] for desc in Descriptors.descList]

def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return [None]*len(descriptor_names)

    descriptor_values = []

    for name, function in Descriptors.descList:
        try:
            value = function(mol)
        except:
            value = None
        descriptor_values.append(value)

    return descriptor_values


# Apply descriptor calculation
descriptor_data = df["SMILES"].apply(calculate_descriptors)

# Convert to dataframe
X = pd.DataFrame(descriptor_data.tolist(), columns=descriptor_names)

# Target variable
y = df["MoA_category"]

# Combine if needed
final_df = pd.concat([X, y], axis=1)

# Save descriptor dataset
final_df.to_excel("MoA_rdkit_all_descriptors_calculated.xlsx", index=False)

print("Total descriptors extracted:", len(descriptor_names))
print("Dataset shape:", final_df.shape)


# Load Descriptor data

import pandas as pd

df = pd.read_excel("MoA_rdkit_all_descriptors_calculated.xlsx")

print("Dataset shape:", df.shape)

df.head()

# seperate descriptors and label(moa)

X = df.drop(columns=["MoA_category"])
y = df["MoA_category"]

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

# -------------------------------
#  FILTER RARE CLASSES HERE
# -------------------------------

df_model = X.copy()
df_model["MoA_category"] = y

counts = df_model["MoA_category"].value_counts()
valid_classes = counts[counts >= 3].index

df_filtered = df_model[df_model["MoA_category"].isin(valid_classes)].copy()

# Update X and y AFTER filtering
X = df_filtered.drop(columns=["MoA_category"])
y = df_filtered["MoA_category"]

print("After filtering:", X.shape, y.shape)

import numpy as np

# Replace infinity values
X = X.replace([np.inf, -np.inf], np.nan)

# Convert to float32 to catch values too large for this dtype and re-handle potential new NaNs
X = X.astype(np.float32)

# Replace any new inf values that might have been created due to float32 conversion overflow
X = X.replace([np.inf, -np.inf], np.nan)

# Fill missing values

X = X.fillna(X.median())

# Remove Extremely Large Descriptors
# Some descriptors can reach 10¹² or more, causing overflow.

X.describe().T.sort_values("max", ascending=False).head(10)

# A safe solution is to clip values.

X = X.clip(-1e6, 1e6)

# A safe solution is to clip values.

X = X.astype(np.float64)

# check missing, infinity and large values

import numpy as np

print("NaN values:", np.isnan(X).sum().sum())
print("Infinity values:", np.isinf(X).sum().sum())
print("Max value in dataset:", np.nanmax(X.values))

# Train-Test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# label encoding

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_encoded = le.fit_transform(y_train)

y_test_encoded = le.transform(y_test)

# Feature stability

from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0)

# Fit ONLY on training data
X_train_var = selector.fit_transform(X_train)

# Apply SAME transformation on test data
X_test_var = selector.transform(X_test)

# Convert to DataFrame (optional but good)
selected_columns = X_train.columns[selector.get_support()]

X_train_var = pd.DataFrame(X_train_var, columns=selected_columns)
X_test_var = pd.DataFrame(X_test_var, columns=selected_columns)

print("After removing constant features:", X_train_var.shape[1])

import numpy as np

# Step 1: Compute correlation ONLY on training data
corr_matrix = X_train_var.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]

print("Descriptors to drop:", len(to_drop))

# Step 2: Drop from TRAIN
X_train_corr = X_train_var.drop(columns=to_drop)

# Step 3: Drop SAME columns from TEST
X_test_corr = X_test_var.drop(columns=to_drop)

print("After correlation removal (train):", X_train_corr.shape)
print("After correlation removal (test):", X_test_corr.shape)

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=400,
    random_state=42
)

# ✅ Train ONLY on training data
rf.fit(X_train_corr, y_train)

# Get importance
importance = pd.Series(
    rf.feature_importances_,
    index=X_train_corr.columns
)

# Sort
importance = importance.sort_values(ascending=False)

print("Top features:\n", importance.head(30))

# Select top 125 features
top_features = importance.head(125).index

# Apply SAME features to train and test
X_train_sel = X_train_corr[top_features]
X_test_sel = X_test_corr[top_features]

print("Final train shape:", X_train_sel.shape)
print("Final test shape:", X_test_sel.shape)

# Normalization

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit ONLY on training
X_train_scaled = scaler.fit_transform(X_train_sel)

# Apply to test
X_test_scaled = scaler.transform(X_test_sel)

# Find best K

from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt

bic_scores = []

k_values = [5, 8, 10, 12, 15, 18, 20, 22]

for k in k_values:

    gmm = GaussianMixture(
        n_components=k,
        covariance_type='full',
        random_state=42
    )

    gmm.fit(X_train_scaled)

    bic = gmm.bic(X_train_scaled)

    bic_scores.append(bic)

    print(f"K = {k}, BIC = {bic}")

# Silhouette Score for best K

from sklearn.metrics import silhouette_score

for k in k_values:

    gmm = GaussianMixture(n_components=k, random_state=42)

    labels = gmm.fit_predict(X_train_scaled)

    score = silhouette_score(X_train_scaled, labels)

    print(f"K = {k}, Silhouette Score = {score}")

# Apply GMM

from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=10, random_state=42)

gmm.fit(X_train_scaled)

train_gmm = gmm.predict_proba(X_train_scaled)
test_gmm = gmm.predict_proba(X_test_scaled)

# Combine
X_train_final = np.hstack([X_train_scaled, train_gmm])
X_test_final = np.hstack([X_test_scaled, test_gmm])

# GMM probabilities for TRAIN
gmm_train_probs = gmm.predict_proba(X_train_scaled)

gmm_train_df = pd.DataFrame(
    gmm_train_probs,
    columns=[f"GMM_Cluster_{i}" for i in range(gmm_train_probs.shape[1])]
)

# Convert scaled train to DataFrame
X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=X_train_sel.columns
)

# Concatenate
X_train_enhanced = pd.concat([X_train_scaled_df, gmm_train_df], axis=1)

print("Train shape:", X_train_enhanced.shape)
print(X_train_enhanced.columns[-10:])

# GMM probabilities for TEST
gmm_test_probs = gmm.predict_proba(X_test_scaled)

gmm_test_df = pd.DataFrame(
    gmm_test_probs,
    columns=[f"GMM_Cluster_{i}" for i in range(gmm_test_probs.shape[1])]
)

# Convert scaled test to DataFrame
X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=X_train_sel.columns   # IMPORTANT: use train columns
)

# Concatenate
X_test_enhanced = pd.concat([X_test_scaled_df, gmm_test_df], axis=1)

print("Test shape:", X_test_enhanced.shape)
print(X_test_enhanced.columns[-10:])

cluster_labels = gmm.predict(X_train_scaled)

df_analysis = pd.DataFrame({
    "Cluster": cluster_labels,
    "MoA": y_train.values
})

print(df_analysis.groupby("Cluster")["MoA"].value_counts())

# ---Function for evaluating metrices---

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    classification_report
)

def evaluate_model_full(model_name, y_test, y_pred, y_prob):

    print("\n==============================")
    print("MODEL:", model_name)
    print("==============================")

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    print("\nOverall Confusion Matrix:\n")
    print(cm)

    # Plot confusion matrix
    plt.figure(figsize=(8,6))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=le.classes_,
        yticklabels=le.classes_
    )

    plt.xlabel("Predicted MoA")
    plt.ylabel("Actual MoA")
    plt.title(model_name + " Confusion Matrix")

    plt.show()

    # ------------------------------
    # TP FP FN TN PER CLASS - CALCULATE HERE
    # ------------------------------

    TP = np.diag(cm)

    FP = np.sum(cm, axis=0) - TP

    FN = np.sum(cm, axis=1) - TP

    TN = np.sum(cm) - (TP + FP + FN)

    # --------------------------------
    # PER-MoA CONFUSION MATRICES
    # --------------------------------

    per_moa_cm_list = []

    for i, moa in enumerate(le.classes_):

        tp = TP[i]
        fp = FP[i]
        fn = FN[i]
        tn = TN[i]

        cm_moa = np.array([
            [tp, fn],
            [fp, tn]
        ])

        print("\nConfusion Matrix for MoA:", moa)
        print(cm_moa)

        plt.figure(figsize=(4,4))

        sns.heatmap(
            cm_moa,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=["Predicted "+moa, "Predicted Other"],
            yticklabels=["Actual "+moa, "Actual Other"]
        )

        plt.title(model_name + " - " + moa)

        plt.show()

        per_moa_cm_list.append({
            "MoA": moa,
            "TP": tp,
            "FN": fn,
            "FP": fp,
            "TN": tn
        })

    per_moa_cm_df = pd.DataFrame(per_moa_cm_list)

    per_moa_cm_df.to_excel(model_name+"_per_moa_confusion_matrix.xlsx", index=False)

    # Sensitivity (Recall)
    sensitivity = TP / (TP + FN)

    # Specificity
    specificity = TN / (TN + FP)

    per_class_df = pd.DataFrame({
        "MoA": le.classes_,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "Sensitivity": sensitivity,
        "Specificity": specificity
    })

    print("\nPer-MoA Metrics\n")
    print(per_class_df)

    # Save results
    per_class_df.to_excel(model_name+"_per_MoA_metrics.xlsx", index=False)

    # ------------------------------
    # OVERALL METRICS
    # ------------------------------

    overall_TP = TP.sum()

    overall_FP = FP.sum()

    overall_FN = FN.sum()

    overall_TN = TN.sum()

    overall_sensitivity = overall_TP / (overall_TP + overall_FN)

    overall_specificity = overall_TN / (overall_TN + overall_FP)

    overall_df = pd.DataFrame({
        "Metric": [
            "TP",
            "FP",
            "FN",
            "TN",
            "Sensitivity",
            "Specificity"
        ],
        "Value": [
            overall_TP,
            overall_FP,
            overall_FN,
            overall_TN,
            overall_sensitivity,
            overall_specificity
        ]
    })

    print("\nOverall Metrics\n")
    print(overall_df)

    overall_df.to_excel(model_name+"_overall_metrics.xlsx", index=False)

    # ------------------------------
    # Accuracy
    # ------------------------------

    acc = accuracy_score(y_test, y_pred)

    print("\nAccuracy:", acc)

    # ------------------------------
    # MCC
    # ------------------------------

    mcc = matthews_corrcoef(y_test, y_pred)

    print("MCC:", mcc)

    # ------------------------------
    # ROC AUC (Multiclass)
    # ------------------------------

    from sklearn.preprocessing import label_binarize

    y_test_bin = label_binarize(y_test, classes=np.unique(y_test))

    auc = roc_auc_score(y_test_bin, y_prob, multi_class="ovr")

    print("ROC-AUC:", auc)

    # Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features='sqrt'
)

# ✅ Train on FINAL enhanced features (IMPORTANT)
rf_model.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_rf = rf_model.predict(X_test_enhanced)

# Accuracy
print("Random Forest Accuracy:",
      accuracy_score(y_test_encoded, y_pred_rf))

# Classification Report
print(classification_report(y_test_encoded, y_pred_rf))

# Probabilities (for ROC-AUC)
y_prob_rf = rf_model.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "Random Forest",
    y_test_encoded,
    y_pred_rf,
    y_prob_rf
)

# Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features='sqrt'
)

# ✅ Train on FINAL enhanced features (IMPORTANT)
rf_model.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_rf = rf_model.predict(X_test_enhanced)

# Accuracy
print("Random Forest Accuracy:",
      accuracy_score(y_test_encoded, y_pred_rf))

# Classification Report
print(classification_report(y_test_encoded, y_pred_rf))

# Probabilities (for ROC-AUC)
y_prob_rf = rf_model.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "Random Forest",
    y_test_encoded,
    y_pred_rf,
    y_prob_rf
)


# XGBoost

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    eval_metric="mlogloss"
)

# ✅ Train on FINAL enhanced features
xgb.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_xgb = xgb.predict(X_test_enhanced)

# Accuracy
print("XGBoost Accuracy:",
      accuracy_score(y_test_encoded, y_pred_xgb))

# Classification Report
print(classification_report(y_test_encoded, y_pred_xgb))

# Probabilities (for ROC-AUC)
y_prob_xgb = xgb.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "XGBoost",
    y_test_encoded,
    y_pred_xgb,
    y_prob_xgb
)




added more small molecules and peptides k = 10 f = 60

In [ ]:
import pandas as pd
import re

# ======================================
# 1. Load the Excel file
# ======================================
df = pd.read_excel("More_smallM_Pep.xlsx")

# ======================================
# 2. Remove duplicate rows
# ======================================
df = df.drop_duplicates()

# ======================================
# 3. Fill missing MoA with "Unknown"
# ======================================
df["MoA"] = df["MoA"].fillna("Unknown")

# ======================================
# 4. Drop MoA with "Unknown"
# ======================================
df = df[df["MoA"] != "Unknown"]

# ======================================
# 4. Clean MoA text
# ======================================
def clean_moa(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", re.sub(r"[^a-z0-9\s]", " ", text))
    return text.strip()

df["MoA_clean"] = df["MoA"].apply(clean_moa)

# ======================================
# 5. Categorization
# ======================================
def categorize_moa(text):

    text = text.lower()

    if text == "unknown":
        return "Multi-target / Polypharmacology"

    # 1 Membrane / Cell Envelope Disruption
    if any(k in text for k in [
        "membrane",
        "cell membrane",
        "pore",
        "permeability",
        "depolar",
        "lipid",
        "bilayer",
        "detergent",
        "ferriprotoporphyrin",
        "reactive nitro radical",
        "reactive intermediates",
        "membrane depolarization",
        "membrane permeabilization",
        "membrane damage",
        "membrane disrupting",
        "breaks down membrane lipids",
        "destroys cell membranes"
    ]):
        return "Membrane / Cell Envelope Disruption"

    # 2 Cell Wall Biosynthesis Inhibition
    if any(k in text for k in [
        "penicillin-binding protein",
        "pbp",
        "peptidoglycan",
        "cell wall",
        "lipid ii",
        "alanine racemase",
        "beta lactamase",
        "beta-lactamase",
        "penicillin binding protein",
        "fab i",
        "enoyl acyl carrier protein reductase"
    ]):
        return "Cell Wall Biosynthesis Inhibition"

    # 3 Protein Synthesis Inhibition
    if any(k in text for k in [
        "ribosome",
        "70s",
        "30s",
        "50s",
        "translation",
        "elongation factor",
        "trna",
        "protein synthesis",
        "trna synthetase",
        "trna ligase"
    ]):
        return "Protein Synthesis Inhibition"

    # 4 Nucleic Acid Synthesis / Integrity Disruption
    if any(k in text for k in [
        "dna",
        "rna",
        "gyrase",
        "topoisomerase",
    ]):
        return "Nucleic Acid Synthesis / Integrity Disruption"

    if any(k in text for k in [
        "dihydrofolate reductase",
        "dihydropteroate synthase",
        "folate",
        "biosynthesis",
        "metabolism",
        "amidophosphoribosyltransferase",
        "oxidoreductase",
        "thymidylate synthase",
        "inosine monophosphate dehydrogenase",
        "dxr",
        "reductase",
        "synthase",
        "dehydrogenase",
        "cyclooxygenase",
        "carbonic anhydrase",
        "monoamine oxidase",
        "hmg coa reductase",
        "xanthine dehydrogenase",
        "caspase",
        "matrix metalloproteinase",
        "reductoisomerase",
        "dioxp reductoisomerase",
        "dxr",
        "dipeptidase",
        "diacylglycerol o acyltransferase",
        "dgat",
        "glucosyltransferase",
        "ceramide glucosyltransferase"
    ]):
        return "Metabolic Pathway Inhibition"

    # 6 Redox / Oxidative Stress Induction
    if any(k in text for k in [
        "oxidative",
        "redox",
        "ros",
        "reactive",
        "nitro",
        "oxidase",
        "reducing agent",
        "antioxidant",
        "increases ros",
        "chelating",
        "copper chelating",
        "iron chelating",
        "aluminium chelating",
        "glutathione",
        "glutathione precursor",
        "redox regulator"
    ]):
        return "Redox / Oxidative Stress Induction"

    # 7 Regulatory / Signalling Interference
    if any(k in text for k in [
        "quorum",
        "two component",
        "regulation",
        "signalling",
        "stress response",
        "transcription",
        "toll like receptor",
        "nf kb",
        "mtor",
        "map kinase",
        "cyclin dependent kinase",
        "biofilm related genes",
        "guanylate cyclase",
        "cyclase activator",
        "cyclase inhibitor",
        "cap binding complex",
        "signal transduction modulator",
        "arca",
        "arcb",
        "arcc",
        "arcd",
        "arc gene",
        "arginine deiminase pathway"
    ]):
        return "Regulatory / Signalling Interference"

    # 8 Biofilm Matrix Disruption
    if any(k in text for k in [
        "biofilm",
        "edna",
        "nuclease"
    ]):
        return "Biofilm Matrix Disruption"

    # 9 Antifungal Sterol Biosynthesis
    if any(k in text for k in [
        "cytochrome p450",
        "cyp51",
        "lanosterol",
        "ergosterol",
        "sterol biosynthesis",
        "squalene monooxygenase"
    ]):
        return "Antifungal Sterol Biosynthesis Inhibitors"

    # 10 Antiviral agents
    if any(k in text for k in [
        "hiv",
        "herpesvirus",
        "influenza",
        "hepatitis c",
        "reverse transcriptase",
        "viral protease",
        "viral capsid",
        "neuraminidase",
        "viral polymerase",
        "viral", "hepatitis", "capsid", "viral kinase", "helicase", "primase",
        "hiv protease",
        "hiv 1 protease",
        "viral protease",
        "pul97",
        "viral phosphotransferase",
        "human immunodeficiency virus type 1 protease"
    ]):
        return "Antiviral agents"

    # 11 Affecting Human Physiology
    if any(k in text for k in [
        "adrenergic receptor",
        "dopamine receptor",
        "serotonin transporter",
        "histamine receptor",
        "muscarinic receptor",
        "angiotensin receptor",
        "opioid receptor",
        "cannabinoid receptor",
        "vitamin d receptor",
        "androgen receptor",
        "estrogen receptor",
        "progesterone receptor",
        "glucocorticoid receptor",
        "receptor",
        "agonist",
        "antagonist",
        "partial agonist",
        "inverse agonist",
        "allosteric modulator",
        "allosteric antagonist",
        "positive modulator",
        "negative modulator",
        "retinoid receptor",
        "retinoic acid receptor",
        "retinoid x receptor",
        "adenosine receptor",
        "dopamine receptor",
        "serotonin receptor",
        "histamine receptor",
        "angiotensin receptor",
        "muscarinic receptor",
        "nicotinic receptor",
        "cannabinoid receptor",
        "opioid receptor",
        "purinergic receptor",
        "vanilloid receptor",
        "bile acid receptor",
        "fxr receptor",
        "peroxisome proliferator activated receptor",
        "ppar",
        "glucocorticoid receptor",
        "mineralocorticoid receptor",
        "estrogen receptor",
        "androgen receptor",
        "progesterone receptor",
        "vitamin d receptor",
        "chemokine receptor",
        "cxcr",
        "s1p receptor",
        "sphingosine receptor",
        "nmda receptor",
        "gaba receptor",
        "acetylcholine receptor",

        "ion channel",
        "channel blocker",
        "channel opener",
        "channel modulator",
        "voltage gated",
        "voltage activated",
        "calcium channel",
        "sodium channel",
        "potassium channel",
        "chloride channel",
        "anion channel",
        "herg",
        "enac",
        "ryanodine receptor",
        "l type calcium channel",
        "inwardly rectifying potassium channel",
        "calcium activated potassium channel",

        "acetylcholinesterase",
        "cholinesterase",
        "dopamine transporter",
        "serotonin transporter",
        "norepinephrine transporter",
        "gaba transporter",
        "synaptic vesicle transporter",
        "vesicular amine transporter",
        "sv2a",
        "neurotransmitter reuptake",
        "bcr abl",
        "abl kinase",
        "fusion protein inhibitor"

        "angiotensin-converting enzyme",
        "angiotensin converting enzyme",
        "ace inhibitor",
        "lipoxygenase",
        "cyclooxygenase",
        "phosphodiesterase",
        "pde inhibitor",
        "neprilysin",
        "lipase",
        "glucosidase",
        "decarboxylase",
        "monoamine oxidase",
        "thrombin",
        "farnesyltransferase",
        "kinase",
        "phosphatase",
        "ubiquitin ligase",
        "aurora kinase",
        "pdgfr",
        "pdk1",
        "ikk",

        "transporter",
        "solute carrier",
        "slc",
        "cotransporter",
        "sodium chloride cotransporter",
        "sodium potassium chloride cotransporter",
        "npc1l1",
        "amine transporter",
        "renal transporter",

        "insulin receptor",
        "hormone receptor",
        "glucose metabolism",
        "lipid metabolism",
        "fatty acid metabolism",
        "ppar alpha",
        "ppar gamma",
        "ppar delta",
        "metabolic regulator",

        "growth factor receptor",
        "egfr",
        "map kinase",
        "mtor",
        "nf kb",
        "signal transduction",
        "transcription factor",
        "cell signaling",
        "cytokine receptor",

        "tubulin",
        "microtubule",
        "cytoskeleton",
        "spindle assembly",
        "microtubule stabilizer",
        "microtubule inhibitor"
        "potassium transporting atpase",
        "npc1l1",
        "niemmann-pick c1",
        "niemann pick c1",
        "synaptic vesicle glycoprotein",
        "sv2a",
        "sv2a modulator",
        "laxative",
        "potassium-transporting atpase inhibitor",
        "potassium transporting atpase",
        "potassium atpase",
        "synaptic vesicle glycoprotein 2a",
        "cholesterol absorption inhibitor"



    ]):
        return "Affecting Human Physiology"

    # 12 Multi-target
    if any(k in text for k in [
        "multi target",
        "multiple",
        "polypharmacology",
        "pleiotropic"
    ]):
        return "Multi-target / Polypharmacology"

    return "Others"


# ======================================
# 6. Apply categorization
# ======================================
df["MoA_category"] = df["MoA_clean"].apply(categorize_moa)

# ======================================
# 7. Keep only required columns
# ======================================
df_final = df[["Name", "SMILES", "MoA_category"]]

# ======================================
# 8. Rename column
# ======================================
df_final = df_final.rename(columns={"MoA_category": "MoA"})

# ======================================
# 9. Check distribution
# ======================================
print(df["MoA_category"].value_counts())

# ======================================
# 10. Save final file
# ======================================
df_final.to_excel("final_moa_categorized.xlsx", index=False)

print(df[df["MoA_category"]=="Others"]["MoA"].unique())

# ======================================
# 10. Save categorized data into multiple sheets
# ======================================

with pd.ExcelWriter("final_moa_categorized.xlsx", engine="xlsxwriter") as writer:

    # Main categorized dataset
    df_final.to_excel(writer, sheet_name="All_Data_PA", index=False)

    # Human physiology targets
    human_targets = df[df["MoA_category"] == "Affecting Human Physiology"]
    human_targets.to_excel(writer, sheet_name="Human_Physiology_Targets", index=False)

    # Antiviral agents
    antiviral_targets = df[df["MoA_category"] == "Antiviral agents"]
    antiviral_targets.to_excel(writer, sheet_name="Antiviral_Targets", index=False)

    # Antifungal agents
    antifungal_targets = df[df["MoA_category"] == "Antifungal Sterol Biosynthesis Inhibitors"]
    antifungal_targets.to_excel(writer, sheet_name="Antifungal_Targets", index=False)

    # Others
    Others = df[df["MoA_category"] == "Others"]
    Others.to_excel(writer, sheet_name="Others", index=False)

    # ======================================
    # Direct Antibacterial (remove unwanted categories)
    # ======================================
    direct_antibacterial = df[~df["MoA_category"].isin([
        "Affecting Human Physiology",
        "Antiviral agents",
        "Antifungal Sterol Biosynthesis Inhibitors",
        "Others"
    ])]

    direct_antibacterial.to_excel(writer, sheet_name="Direct_Antibacterial", index=False)

print("Excel file created with multiple sheets including Direct_Antibacterial.")

import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

# Load dataset from the correct file and sheet
df = pd.read_excel("final_moa_categorized.xlsx", sheet_name="Direct_Antibacterial")

# Get list of all RDKit descriptors
descriptor_names = [desc[0] for desc in Descriptors.descList]

def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return [None]*len(descriptor_names)

    descriptor_values = []

    for name, function in Descriptors.descList:
        try:
            value = function(mol)
        except:
            value = None
        descriptor_values.append(value)

    return descriptor_values

# Apply descriptor calculation
descriptor_data = df["SMILES"]
# Explicitly convert SMILES column to string type to avoid issues with mixed types
descriptor_data = descriptor_data.astype(str).apply(calculate_descriptors)

# Convert to dataframe
X = pd.DataFrame(descriptor_data.tolist(), columns=descriptor_names)

# Target variable
y = df["MoA_category"]

# Combine if needed
final_df = pd.concat([X, y], axis=1)

# Save descriptor dataset
final_df.to_excel("MoA_rdkit_all_descriptors_calculated.xlsx", index=False)

print("Total descriptors extracted:", len(descriptor_names))
print("Dataset shape:", final_df.shape)


# Load Descriptor data

import pandas as pd

df = pd.read_excel("MoA_rdkit_all_descriptors_calculated.xlsx")

print("Dataset shape:", df.shape)

df.head()

# seperate descriptors and label(moa)

X = df.drop(columns=["MoA_category"])
y = df["MoA_category"]

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

# -------------------------------
#  FILTER RARE CLASSES HERE
# -------------------------------

df_model = X.copy()
df_model["MoA_category"] = y

counts = df_model["MoA_category"].value_counts()
valid_classes = counts[counts >= 3].index

df_filtered = df_model[df_model["MoA_category"].isin(valid_classes)].copy()

# Update X and y AFTER filtering
X = df_filtered.drop(columns=["MoA_category"])
y = df_filtered["MoA_category"]

print("After filtering:", X.shape, y.shape)

import numpy as np

# Replace infinity values
X = X.replace([np.inf, -np.inf], np.nan)

# Convert to float32 to catch values too large for this dtype and re-handle potential new NaNs
X = X.astype(np.float32)

# Replace any new inf values that might have been created due to float32 conversion overflow
X = X.replace([np.inf, -np.inf], np.nan)

# Fill missing values

X = X.fillna(X.median())

# Remove Extremely Large Descriptors
# Some descriptors can reach 10¹² or more, causing overflow.

X.describe().T.sort_values("max", ascending=False).head(10)

# A safe solution is to clip values.

X = X.clip(-1e6, 1e6)

# A safe solution is to clip values.

X = X.astype(np.float64)

# check missing, infinity and large values

import numpy as np

print("NaN values:", np.isnan(X).sum().sum())
print("Infinity values:", np.isinf(X).sum().sum())
print("Max value in dataset:", np.nanmax(X.values))

# Train-Test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# label encoding

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_encoded = le.fit_transform(y_train)

y_test_encoded = le.transform(y_test)

# Feature stability

from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0)

# Fit ONLY on training data
X_train_var = selector.fit_transform(X_train)

# Apply SAME transformation on test data
X_test_var = selector.transform(X_test)

# Convert to DataFrame (optional but good)
selected_columns = X_train.columns[selector.get_support()]

X_train_var = pd.DataFrame(X_train_var, columns=selected_columns)
X_test_var = pd.DataFrame(X_test_var, columns=selected_columns)

print("After removing constant features:", X_train_var.shape[1])

import numpy as np

# Step 1: Compute correlation ONLY on training data
corr_matrix = X_train_var.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]

print("Descriptors to drop:", len(to_drop))

# Step 2: Drop from TRAIN
X_train_corr = X_train_var.drop(columns=to_drop)

# Step 3: Drop SAME columns from TEST
X_test_corr = X_test_var.drop(columns=to_drop)

print("After correlation removal (train):", X_train_corr.shape)
print("After correlation removal (test):", X_test_corr.shape)

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=400,
    random_state=42
)

# ✅ Train ONLY on training data
rf.fit(X_train_corr, y_train)

# Get importance
importance = pd.Series(
    rf.feature_importances_,
    index=X_train_corr.columns
)

# Sort
importance = importance.sort_values(ascending=False)

print("Top features:\n", importance.head(30))

# Select top 60 features
top_features = importance.head(60).index

# Apply SAME features to train and test
X_train_sel = X_train_corr[top_features]
X_test_sel = X_test_corr[top_features]

print("Final train shape:", X_train_sel.shape)
print("Final test shape:", X_test_sel.shape)

# Normalization

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit ONLY on training
X_train_scaled = scaler.fit_transform(X_train_sel)

# Apply to test
X_test_scaled = scaler.transform(X_test_sel)

# Find best K

from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt

bic_scores = []

k_values = [5, 8, 10, 12, 15, 18, 20, 22]

for k in k_values:

    gmm = GaussianMixture(
        n_components=k,
        covariance_type='full',
        random_state=42
    )

    gmm.fit(X_train_scaled)

    bic = gmm.bic(X_train_scaled)

    bic_scores.append(bic)

    print(f"K = {k}, BIC = {bic}")

# Silhouette Score for best K

from sklearn.metrics import silhouette_score

for k in k_values:

    gmm = GaussianMixture(n_components=k, random_state=42)

    labels = gmm.fit_predict(X_train_scaled)

    score = silhouette_score(X_train_scaled, labels)

    print(f"K = {k}, Silhouette Score = {score}")

# Apply GMM

from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=20, random_state=42)

gmm.fit(X_train_scaled)

train_gmm = gmm.predict_proba(X_train_scaled)
test_gmm = gmm.predict_proba(X_test_scaled)

# Combine
X_train_final = np.hstack([X_train_scaled, train_gmm])
X_test_final = np.hstack([X_test_scaled, test_gmm])

# GMM probabilities for TRAIN
gmm_train_probs = gmm.predict_proba(X_train_scaled)

gmm_train_df = pd.DataFrame(
    gmm_train_probs,
    columns=[f"GMM_Cluster_{i}" for i in range(gmm_train_probs.shape[1])]
)

# Convert scaled train to DataFrame
X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=X_train_sel.columns
)

# Concatenate
X_train_enhanced = pd.concat([X_train_scaled_df, gmm_train_df], axis=1)

print("Train shape:", X_train_enhanced.shape)
print(X_train_enhanced.columns[-10:])

# GMM probabilities for TEST
gmm_test_probs = gmm.predict_proba(X_test_scaled)

gmm_test_df = pd.DataFrame(
    gmm_test_probs,
    columns=[f"GMM_Cluster_{i}" for i in range(gmm_test_probs.shape[1])]
)

# Convert scaled test to DataFrame
X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=X_train_sel.columns   # IMPORTANT: use train columns
)

# Concatenate
X_test_enhanced = pd.concat([X_test_scaled_df, gmm_test_df], axis=1)

print("Test shape:", X_test_enhanced.shape)
print(X_test_enhanced.columns[-10:])

cluster_labels = gmm.predict(X_train_scaled)

df_analysis = pd.DataFrame({
    "Cluster": cluster_labels,
    "MoA": y_train.values
})

print(df_analysis.groupby("Cluster")["MoA"].value_counts())

# ---Function for evaluating metrices---

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    classification_report
)

def evaluate_model_full(model_name, y_test, y_pred, y_prob):

    print("\n==============================")
    print("MODEL:", model_name)
    print("==============================")

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    print("\nOverall Confusion Matrix:\n")
    print(cm)

    # Plot confusion matrix
    plt.figure(figsize=(8,6))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=le.classes_,
        yticklabels=le.classes_
    )

    plt.xlabel("Predicted MoA")
    plt.ylabel("Actual MoA")
    plt.title(model_name + " Confusion Matrix")

    plt.show()

    # ------------------------------
    # TP FP FN TN PER CLASS - CALCULATE HERE
    # ------------------------------

    TP = np.diag(cm)

    FP = np.sum(cm, axis=0) - TP

    FN = np.sum(cm, axis=1) - TP

    TN = np.sum(cm) - (TP + FP + FN)

    # --------------------------------
    # PER-MoA CONFUSION MATRICES
    # --------------------------------

    per_moa_cm_list = []

    for i, moa in enumerate(le.classes_):

        tp = TP[i]
        fp = FP[i]
        fn = FN[i]
        tn = TN[i]

        cm_moa = np.array([
            [tp, fn],
            [fp, tn]
        ])

        print("\nConfusion Matrix for MoA:", moa)
        print(cm_moa)

        plt.figure(figsize=(4,4))

        sns.heatmap(
            cm_moa,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=["Predicted "+moa, "Predicted Other"],
            yticklabels=["Actual "+moa, "Actual Other"]
        )

        plt.title(model_name + " - " + moa)

        plt.show()

        per_moa_cm_list.append({
            "MoA": moa,
            "TP": tp,
            "FN": fn,
            "FP": fp,
            "TN": tn
        })

    per_moa_cm_df = pd.DataFrame(per_moa_cm_list)

    per_moa_cm_df.to_excel(model_name+"_per_moa_confusion_matrix.xlsx", index=False)

    # Sensitivity (Recall)
    sensitivity = TP / (TP + FN)

    # Specificity
    specificity = TN / (TN + FP)

    per_class_df = pd.DataFrame({
        "MoA": le.classes_,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "Sensitivity": sensitivity,
        "Specificity": specificity
    })

    print("\nPer-MoA Metrics\n")
    print(per_class_df)

    # Save results
    per_class_df.to_excel(model_name+"_per_MoA_metrics.xlsx", index=False)

    # ------------------------------
    # OVERALL METRICS
    # ------------------------------

    overall_TP = TP.sum()

    overall_FP = FP.sum()

    overall_FN = FN.sum()

    overall_TN = TN.sum()

    overall_sensitivity = overall_TP / (overall_TP + overall_FN)

    overall_specificity = overall_TN / (overall_TN + overall_FP)

    overall_df = pd.DataFrame({
        "Metric": [
            "TP",
            "FP",
            "FN",
            "TN",
            "Sensitivity",
            "Specificity"
        ],
        "Value": [
            overall_TP,
            overall_FP,
            overall_FN,
            overall_TN,
            overall_sensitivity,
            overall_specificity
        ]
    })

    print("\nOverall Metrics\n")
    print(overall_df)

    overall_df.to_excel(model_name+"_overall_metrics.xlsx", index=False)

    # ------------------------------
    # Accuracy
    # ------------------------------

    acc = accuracy_score(y_test, y_pred)

    print("\nAccuracy:", acc)

    # ------------------------------
    # MCC
    # ------------------------------

    mcc = matthews_corrcoef(y_test, y_pred)

    print("MCC:", mcc)

    # ------------------------------
    # ROC AUC (Multiclass)
    # ------------------------------

    from sklearn.preprocessing import label_binarize

    y_test_bin = label_binarize(y_test, classes=np.unique(y_test))

    auc = roc_auc_score(y_test_bin, y_prob, multi_class="ovr")

    print("ROC-AUC:", auc)

    # Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features='sqrt'
)

# ✅ Train on FINAL enhanced features (IMPORTANT)
rf_model.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_rf = rf_model.predict(X_test_enhanced)

# Accuracy
print("Random Forest Accuracy:",
      accuracy_score(y_test_encoded, y_pred_rf))

# Classification Report
print(classification_report(y_test_encoded, y_pred_rf))

# Probabilities (for ROC-AUC)
y_prob_rf = rf_model.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "Random Forest",
    y_test_encoded,
    y_pred_rf,
    y_prob_rf
)

# Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features='sqrt'
)

# ✅ Train on FINAL enhanced features (IMPORTANT)
rf_model.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_rf = rf_model.predict(X_test_enhanced)

# Accuracy
print("Random Forest Accuracy:",
      accuracy_score(y_test_encoded, y_pred_rf))

# Classification Report
print(classification_report(y_test_encoded, y_pred_rf))

# Probabilities (for ROC-AUC)
y_prob_rf = rf_model.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "Random Forest",
    y_test_encoded,
    y_pred_rf,
    y_prob_rf
)


# XGBoost

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    eval_metric="mlogloss"
)

# ✅ Train on FINAL enhanced features
xgb.fit(X_train_enhanced, y_train_encoded)

# ✅ Predictions
y_pred_xgb = xgb.predict(X_test_enhanced)

# Accuracy
print("XGBoost Accuracy:",
      accuracy_score(y_test_encoded, y_pred_xgb))

# Classification Report
print(classification_report(y_test_encoded, y_pred_xgb))

# Probabilities (for ROC-AUC)
y_prob_xgb = xgb.predict_proba(X_test_enhanced)

# Full evaluation
evaluate_model_full(
    "XGBoost",
    y_test_encoded,
    y_pred_xgb,
    y_prob_xgb
)




WITHOUT GMM

In [ ]:
import pandas as pd
import re

# ======================================
# 1. Load the Excel file
# ======================================
df = pd.read_excel("final_moa_cateogrization_Peptides+Smiles.xlsx")

# ======================================
# 2. Remove duplicate rows
# ======================================
df = df.drop_duplicates()

# ======================================
# 3. Fill missing MoA with "Unknown"
# ======================================
df["MoA"] = df["MoA"].fillna("Unknown")

# ======================================
# 4. Drop MoA with "Unknown"
# ======================================
df = df[df["MoA"] != "Unknown"]

# ======================================
# 4. Clean MoA text
# ======================================
def clean_moa(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", re.sub(r"[^a-z0-9\s]", " ", text))
    return text.strip()

df["MoA_clean"] = df["MoA"].apply(clean_moa)

# ======================================
# 5. Categorization
# ======================================
def categorize_moa(text):

    text = text.lower()

    if text == "unknown":
        return "Multi-target / Polypharmacology"

    # 1 Membrane / Cell Envelope Disruption
    if any(k in text for k in [
        "membrane",
        "cell membrane",
        "pore",
        "permeability",
        "depolar",
        "lipid",
        "bilayer",
        "detergent",
        "ferriprotoporphyrin",
        "reactive nitro radical",
        "reactive intermediates",
        "membrane depolarization",
        "membrane permeabilization",
        "membrane damage",
        "membrane disrupting",
        "breaks down membrane lipids",
        "destroys cell membranes"
    ]):
        return "Membrane / Cell Envelope Disruption"

    # 2 Cell Wall Biosynthesis Inhibition
    if any(k in text for k in [
        "penicillin-binding protein",
        "pbp",
        "peptidoglycan",
        "cell wall",
        "lipid ii",
        "alanine racemase",
        "beta lactamase",
        "beta-lactamase",
        "penicillin binding protein",
        "fab i",
        "enoyl acyl carrier protein reductase"
    ]):
        return "Cell Wall Biosynthesis Inhibition"

    # 3 Protein Synthesis Inhibition
    if any(k in text for k in [
        "ribosome",
        "70s",
        "30s",
        "50s",
        "translation",
        "elongation factor",
        "trna",
        "protein synthesis",
        "trna synthetase",
        "trna ligase"
    ]):
        return "Protein Synthesis Inhibition"

    # 4 Nucleic Acid Synthesis / Integrity Disruption
    if any(k in text for k in [
        "dna",
        "rna",
        "gyrase",
        "topoisomerase",
        "polymerase",
        "replication",
        "intercalat",
        "dna binding",
        "dna disrupting",
        "dna inhibitor",
        "ribonucleotide reductase"
    ]):
        return "Nucleic Acid Synthesis / Integrity Disruption"

    # 5 Metabolic Pathway Inhibition
    if any(k in text for k in [
        "dihydrofolate reductase",
        "dihydropteroate synthase",
        "folate",
        "biosynthesis",
        "metabolism",
        "amidophosphoribosyltransferase",
        "oxidoreductase",
        "thymidylate synthase",
        "inosine monophosphate dehydrogenase",
        "dxr",
        "reductase",
        "synthase",
        "dehydrogenase",
        "cyclooxygenase",
        "carbonic anhydrase",
        "monoamine oxidase",
        "hmg coa reductase",
        "xanthine dehydrogenase",
        "caspase",
        "matrix metalloproteinase",
        "reductoisomerase",
        "dioxp reductoisomerase",
        "dxr",
        "dipeptidase",
        "diacylglycerol o acyltransferase",
        "dgat",
        "glucosyltransferase",
        "ceramide glucosyltransferase"
    ]):
        return "Metabolic Pathway Inhibition"

    # 6 Redox / Oxidative Stress Induction
    if any(k in text for k in [
        "oxidative",
        "redox",
        "ros",
        "reactive",
        "nitro",
        "oxidase",
        "reducing agent",
        "antioxidant",
        "increases ros",
        "chelating",
        "copper chelating",
        "iron chelating",
        "aluminium chelating",
        "glutathione",
        "glutathione precursor",
        "redox regulator"
    ]):
        return "Redox / Oxidative Stress Induction"

    # 7 Regulatory / Signalling Interference
    if any(k in text for k in [
        "quorum",
        "two component",
        "regulation",
        "signalling",
        "stress response",
        "transcription",
        "toll like receptor",
        "nf kb",
        "mtor",
        "map kinase",
        "cyclin dependent kinase",
        "biofilm related genes",
        "guanylate cyclase",
        "cyclase activator",
        "cyclase inhibitor",
        "cap binding complex",
        "signal transduction modulator",
        "arca",
        "arcb",
        "arcc",
        "arcd",
        "arc gene",
        "arginine deiminase pathway"
    ]):
        return "Regulatory / Signalling Interference"

    # 8 Biofilm Matrix Disruption
    if any(k in text for k in [
        "biofilm",
        "edna",
        "nuclease"
    ]):
        return "Biofilm Matrix Disruption"

    # 9 Antifungal Sterol Biosynthesis
    if any(k in text for k in [
        "cytochrome p450",
        "cyp51",
        "lanosterol",
        "ergosterol",
        "sterol biosynthesis",
        "squalene monooxygenase"
    ]):
        return "Antifungal Sterol Biosynthesis Inhibitors"

    # 10 Antiviral agents
    if any(k in text for k in [
        "hiv",
        "herpesvirus",
        "influenza",
        "hepatitis c",
        "reverse transcriptase",
        "viral protease",
        "viral capsid",
        "neuraminidase",
        "viral polymerase",
        "viral", "hepatitis", "capsid", "viral kinase", "helicase", "primase",
        "hiv protease",
        "hiv 1 protease",
        "viral protease",
        "pul97",
        "viral phosphotransferase",
        "human immunodeficiency virus type 1 protease"
    ]):
        return "Antiviral agents"

    # 11 Affecting Human Physiology
    if any(k in text for k in [
        "adrenergic receptor",
        "dopamine receptor",
        "serotonin transporter",
        "histamine receptor",
        "muscarinic receptor",
        "angiotensin receptor",
        "opioid receptor",
        "cannabinoid receptor",
        "vitamin d receptor",
        "androgen receptor",
        "estrogen receptor",
        "progesterone receptor",
        "glucocorticoid receptor",
        "receptor",
        "agonist",
        "antagonist",
        "partial agonist",
        "inverse agonist",
        "allosteric modulator",
        "allosteric antagonist",
        "positive modulator",
        "negative modulator",
        "retinoid receptor",
        "retinoic acid receptor",
        "retinoid x receptor",
        "adenosine receptor",
        "dopamine receptor",
        "serotonin receptor",
        "histamine receptor",
        "angiotensin receptor",
        "muscarinic receptor",
        "nicotinic receptor",
        "cannabinoid receptor",
        "opioid receptor",
        "purinergic receptor",
        "vanilloid receptor",
        "bile acid receptor",
        "fxr receptor",
        "peroxisome proliferator activated receptor",
        "ppar",
        "glucocorticoid receptor",
        "mineralocorticoid receptor",
        "estrogen receptor",
        "androgen receptor",
        "progesterone receptor",
        "vitamin d receptor",
        "chemokine receptor",
        "cxcr",
        "s1p receptor",
        "sphingosine receptor",
        "nmda receptor",
        "gaba receptor",
        "acetylcholine receptor",

        "ion channel",
        "channel blocker",
        "channel opener",
        "channel modulator",
        "voltage gated",
        "voltage activated",
        "calcium channel",
        "sodium channel",
        "potassium channel",
        "chloride channel",
        "anion channel",
        "herg",
        "enac",
        "ryanodine receptor",
        "l type calcium channel",
        "inwardly rectifying potassium channel",
        "calcium activated potassium channel",

        "acetylcholinesterase",
        "cholinesterase",
        "dopamine transporter",
        "serotonin transporter",
        "norepinephrine transporter",
        "gaba transporter",
        "synaptic vesicle transporter",
        "vesicular amine transporter",
        "sv2a",
        "neurotransmitter reuptake",
        "bcr abl",
        "abl kinase",
        "fusion protein inhibitor"

        "angiotensin-converting enzyme",
        "angiotensin converting enzyme",
        "ace inhibitor",
        "lipoxygenase",
        "cyclooxygenase",
        "phosphodiesterase",
        "pde inhibitor",
        "neprilysin",
        "lipase",
        "glucosidase",
        "decarboxylase",
        "monoamine oxidase",
        "thrombin",
        "farnesyltransferase",
        "kinase",
        "phosphatase",
        "ubiquitin ligase",
        "aurora kinase",
        "pdgfr",
        "pdk1",
        "ikk",

        "transporter",
        "solute carrier",
        "slc",
        "cotransporter",
        "sodium chloride cotransporter",
        "sodium potassium chloride cotransporter",
        "npc1l1",
        "amine transporter",
        "renal transporter",

        "insulin receptor",
        "hormone receptor",
        "glucose metabolism",
        "lipid metabolism",
        "fatty acid metabolism",
        "ppar alpha",
        "ppar gamma",
        "ppar delta",
        "metabolic regulator",

        "growth factor receptor",
        "egfr",
        "map kinase",
        "mtor",
        "nf kb",
        "signal transduction",
        "transcription factor",
        "cell signaling",
        "cytokine receptor",

        "tubulin",
        "microtubule",
        "cytoskeleton",
        "spindle assembly",
        "microtubule stabilizer",
        "microtubule inhibitor"
        "potassium transporting atpase",
        "npc1l1",
        "niemmann-pick c1",
        "niemann pick c1",
        "synaptic vesicle glycoprotein",
        "sv2a",
        "sv2a modulator",
        "laxative",
        "potassium-transporting atpase inhibitor",
        "potassium transporting atpase",
        "potassium atpase",
        "synaptic vesicle glycoprotein 2a",
        "cholesterol absorption inhibitor"



    ]):
        return "Affecting Human Physiology"

    # 12 Multi-target
    if any(k in text for k in [
        "multi target",
        "multiple",
        "polypharmacology",
        "pleiotropic"
    ]):
        return "Multi-target / Polypharmacology"

    return "Others"


# ======================================
# 6. Apply categorization
# ======================================
df["MoA_category"] = df["MoA_clean"].apply(categorize_moa)

# ======================================
# 7. Keep only required columns
# ======================================
df_final = df[["Name", "SMILES", "MoA_category"]]

# ======================================
# 8. Rename column
# ======================================
df_final = df_final.rename(columns={"MoA_category": "MoA"})

# ======================================
# 9. Check distribution
# ======================================
print(df["MoA_category"].value_counts())

# ======================================
# 10. Save final file
# ======================================
df_final.to_excel("final_moa_categorized.xlsx", index=False)

print(df[df["MoA_category"]=="Others"]["MoA"].unique())

# ======================================
# 10. Save categorized data into multiple sheets
# ======================================

with pd.ExcelWriter("final_moa_categorized.xlsx", engine="xlsxwriter") as writer:

    # Main categorized dataset
    df_final.to_excel(writer, sheet_name="All_Data_PA", index=False)

    # Human physiology targets
    human_targets = df[df["MoA_category"] == "Affecting Human Physiology"]
    human_targets.to_excel(writer, sheet_name="Human_Physiology_Targets", index=False)

    # Antiviral agents
    antiviral_targets = df[df["MoA_category"] == "Antiviral agents"]
    antiviral_targets.to_excel(writer, sheet_name="Antiviral_Targets", index=False)

    # Antifungal agents
    antifungal_targets = df[df["MoA_category"] == "Antifungal Sterol Biosynthesis Inhibitors"]
    antifungal_targets.to_excel(writer, sheet_name="Antifungal_Targets", index=False)

    # Others
    Others = df[df["MoA_category"] == "Others"]
    Others.to_excel(writer, sheet_name="Others", index=False)

    # ======================================
    # Direct Antibacterial (remove unwanted categories)
    # ======================================
    direct_antibacterial = df[~df["MoA_category"].isin([
        "Affecting Human Physiology",
        "Antiviral agents",
        "Antifungal Sterol Biosynthesis Inhibitors",
        "Others"
    ])]

    direct_antibacterial.to_excel(writer, sheet_name="Direct_Antibacterial", index=False)

print("Excel file created with multiple sheets including Direct_Antibacterial.")

import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

# Load dataset from the correct file and sheet
df = pd.read_excel("final_moa_categorized.xlsx", sheet_name="Direct_Antibacterial")

# Get list of all RDKit descriptors
descriptor_names = [desc[0] for desc in Descriptors.descList]

def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return [None]*len(descriptor_names)

    descriptor_values = []

    for name, function in Descriptors.descList:
        try:
            value = function(mol)
        except:
            value = None
        descriptor_values.append(value)

    return descriptor_values


# Apply descriptor calculation
descriptor_data = df["SMILES"].apply(calculate_descriptors)

# Convert to dataframe
X = pd.DataFrame(descriptor_data.tolist(), columns=descriptor_names)

# Target variable
y = df["MoA_category"]

# Combine if needed
final_df = pd.concat([X, y], axis=1)

# Save descriptor dataset
final_df.to_excel("MoA_rdkit_all_descriptors_calculated.xlsx", index=False)

print("Total descriptors extracted:", len(descriptor_names))
print("Dataset shape:", final_df.shape)


# Load Descriptor data

import pandas as pd

df = pd.read_excel("MoA_rdkit_all_descriptors_calculated.xlsx")

print("Dataset shape:", df.shape)

df.head()

# seperate descriptors and label(moa)

X = df.drop(columns=["MoA_category"])
y = df["MoA_category"]

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

# -------------------------------
#  FILTER RARE CLASSES HERE
# -------------------------------

df_model = X.copy()
df_model["MoA_category"] = y

counts = df_model["MoA_category"].value_counts()
valid_classes = counts[counts >= 3].index

df_filtered = df_model[df_model["MoA_category"].isin(valid_classes)].copy()

# Update X and y AFTER filtering
X = df_filtered.drop(columns=["MoA_category"])
y = df_filtered["MoA_category"]

print("After filtering:", X.shape, y.shape)

import numpy as np

# Replace infinity values
X = X.replace([np.inf, -np.inf], np.nan)

# Convert to float32 to catch values too large for this dtype and re-handle potential new NaNs
X = X.astype(np.float32)

# Replace any new inf values that might have been created due to float32 conversion overflow
X = X.replace([np.inf, -np.inf], np.nan)

# Fill missing values

X = X.fillna(X.median())

# Remove Extremely Large Descriptors
# Some descriptors can reach 10¹² or more, causing overflow.

X.describe().T.sort_values("max", ascending=False).head(10)

# A safe solution is to clip values.

X = X.clip(-1e6, 1e6)

# A safe solution is to clip values.

X = X.astype(np.float64)

# check missing, infinity and large values

import numpy as np

print("NaN values:", np.isnan(X).sum().sum())
print("Infinity values:", np.isinf(X).sum().sum())
print("Max value in dataset:", np.nanmax(X.values))

# Train-Test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# label encoding

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_encoded = le.fit_transform(y_train)

y_test_encoded = le.transform(y_test)

# Feature stability

from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0)

# Fit ONLY on training data
X_train_var = selector.fit_transform(X_train)

# Apply SAME transformation on test data
X_test_var = selector.transform(X_test)

# Convert to DataFrame (optional but good)
selected_columns = X_train.columns[selector.get_support()]

X_train_var = pd.DataFrame(X_train_var, columns=selected_columns)
X_test_var = pd.DataFrame(X_test_var, columns=selected_columns)

print("After removing constant features:", X_train_var.shape[1])

import numpy as np

# Step 1: Compute correlation ONLY on training data
corr_matrix = X_train_var.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]

print("Descriptors to drop:", len(to_drop))

# Step 2: Drop from TRAIN
X_train_corr = X_train_var.drop(columns=to_drop)

# Step 3: Drop SAME columns from TEST
X_test_corr = X_test_var.drop(columns=to_drop)

print("After correlation removal (train):", X_train_corr.shape)
print("After correlation removal (test):", X_test_corr.shape)

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=400,
    random_state=42
)

# ✅ Train ONLY on training data
rf.fit(X_train_corr, y_train)

# Get importance
importance = pd.Series(
    rf.feature_importances_,
    index=X_train_corr.columns
)

# Sort
importance = importance.sort_values(ascending=False)

print("Top features:\n", importance.head(30))

# Select top 60 features
top_features = importance.head(60).index

# Apply SAME features to train and test
X_train_sel = X_train_corr[top_features]
X_test_sel = X_test_corr[top_features]

print("Final train shape:", X_train_sel.shape)
print("Final test shape:", X_test_sel.shape)

# Normalization

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit ONLY on training
X_train_scaled = scaler.fit_transform(X_train_sel)

# Apply to test
X_test_scaled = scaler.transform(X_test_sel)

# # Find best K

# from sklearn.mixture import GaussianMixture
# import matplotlib.pyplot as plt

# bic_scores = []

# k_values = [5, 8, 10, 12, 15, 18, 20, 22]

# for k in k_values:

#     gmm = GaussianMixture(
#         n_components=k,
#         covariance_type='full',
#         random_state=42
#     )

#     gmm.fit(X_train_scaled)

#     bic = gmm.bic(X_train_scaled)

#     bic_scores.append(bic)

#     print(f"K = {k}, BIC = {bic}")

# # Silhouette Score for best K

# from sklearn.metrics import silhouette_score

# for k in k_values:

#     gmm = GaussianMixture(n_components=k, random_state=42)

#     labels = gmm.fit_predict(X_train_scaled)

#     score = silhouette_score(X_train_scaled, labels)

#     print(f"K = {k}, Silhouette Score = {score}")

# # Apply GMM

# from sklearn.mixture import GaussianMixture

# gmm = GaussianMixture(n_components=10, random_state=42)

# gmm.fit(X_train_scaled)

# train_gmm = gmm.predict_proba(X_train_scaled)
# test_gmm = gmm.predict_proba(X_test_scaled)

# # Combine
# X_train_final = np.hstack([X_train_scaled, train_gmm])
# X_test_final = np.hstack([X_test_scaled, test_gmm])

# # GMM probabilities for TRAIN
# gmm_train_probs = gmm.predict_proba(X_train_scaled)

# gmm_train_df = pd.DataFrame(
#     gmm_train_probs,
#     columns=[f"GMM_Cluster_{i}" for i in range(gmm_train_probs.shape[1])]
# )

# # Convert scaled train to DataFrame
# X_train_scaled_df = pd.DataFrame(
#     X_train_scaled,
#     columns=X_train_sel.columns
# )

# # Concatenate
# X_train_enhanced = pd.concat([X_train_scaled_df, gmm_train_df], axis=1)

# print("Train shape:", X_train_enhanced.shape)
# print(X_train_enhanced.columns[-10:])

# # GMM probabilities for TEST
# gmm_test_probs = gmm.predict_proba(X_test_scaled)

# gmm_test_df = pd.DataFrame(
#     gmm_test_probs,
#     columns=[f"GMM_Cluster_{i}" for i in range(gmm_test_probs.shape[1])]
# )

# # Convert scaled test to DataFrame
# X_test_scaled_df = pd.DataFrame(
#     X_test_scaled,
#     columns=X_train_sel.columns   # IMPORTANT: use train columns
# )

# # Concatenate
# X_test_enhanced = pd.concat([X_test_scaled_df, gmm_test_df], axis=1)

# print("Test shape:", X_test_enhanced.shape)
# print(X_test_enhanced.columns[-10:])

# cluster_labels = gmm.predict(X_train_scaled)

# df_analysis = pd.DataFrame({
#     "Cluster": cluster_labels,
#     "MoA": y_train.values
# })

# print(df_analysis.groupby("Cluster")["MoA"].value_counts())

# ---Function for evaluating metrices---

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    classification_report
)

def evaluate_model_full(model_name, y_test, y_pred, y_prob):

    print("\n==============================")
    print("MODEL:", model_name)
    print("==============================")

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    print("\nOverall Confusion Matrix:\n")
    print(cm)

    # Plot confusion matrix
    plt.figure(figsize=(8,6))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=le.classes_,
        yticklabels=le.classes_
    )

    plt.xlabel("Predicted MoA")
    plt.ylabel("Actual MoA")
    plt.title(model_name + " Confusion Matrix")

    plt.show()

    # ------------------------------
    # TP FP FN TN PER CLASS - CALCULATE HERE
    # ------------------------------

    TP = np.diag(cm)

    FP = np.sum(cm, axis=0) - TP

    FN = np.sum(cm, axis=1) - TP

    TN = np.sum(cm) - (TP + FP + FN)

    # --------------------------------
    # PER-MoA CONFUSION MATRICES
    # --------------------------------

    per_moa_cm_list = []

    for i, moa in enumerate(le.classes_):

        tp = TP[i]
        fp = FP[i]
        fn = FN[i]
        tn = TN[i]

        cm_moa = np.array([
            [tp, fn],
            [fp, tn]
        ])

        print("\nConfusion Matrix for MoA:", moa)
        print(cm_moa)

        plt.figure(figsize=(4,4))

        sns.heatmap(
            cm_moa,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=["Predicted "+moa, "Predicted Other"],
            yticklabels=["Actual "+moa, "Actual Other"]
        )

        plt.title(model_name + " - " + moa)

        plt.show()

        per_moa_cm_list.append({
            "MoA": moa,
            "TP": tp,
            "FN": fn,
            "FP": fp,
            "TN": tn
        })

    per_moa_cm_df = pd.DataFrame(per_moa_cm_list)

    per_moa_cm_df.to_excel(model_name+"_per_moa_confusion_matrix.xlsx", index=False)

    # Sensitivity (Recall)
    sensitivity = TP / (TP + FN)

    # Specificity
    specificity = TN / (TN + FP)

    per_class_df = pd.DataFrame({
        "MoA": le.classes_,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "Sensitivity": sensitivity,
        "Specificity": specificity
    })

    print("\nPer-MoA Metrics\n")
    print(per_class_df)

    # Save results
    per_class_df.to_excel(model_name+"_per_MoA_metrics.xlsx", index=False)

    # ------------------------------
    # OVERALL METRICS
    # ------------------------------

    overall_TP = TP.sum()

    overall_FP = FP.sum()

    overall_FN = FN.sum()

    overall_TN = TN.sum()

    overall_sensitivity = overall_TP / (overall_TP + overall_FN)

    overall_specificity = overall_TN / (overall_TN + overall_FP)

    overall_df = pd.DataFrame({
        "Metric": [
            "TP",
            "FP",
            "FN",
            "TN",
            "Sensitivity",
            "Specificity"
        ],
        "Value": [
            overall_TP,
            overall_FP,
            overall_FN,
            overall_TN,
            overall_sensitivity,
            overall_specificity
        ]
    })

    print("\nOverall Metrics\n")
    print(overall_df)

    overall_df.to_excel(model_name+"_overall_metrics.xlsx", index=False)

    # ------------------------------
    # Accuracy
    # ------------------------------

    acc = accuracy_score(y_test, y_pred)

    print("\nAccuracy:", acc)

    # ------------------------------
    # MCC
    # ------------------------------

    mcc = matthews_corrcoef(y_test, y_pred)

    print("MCC:", mcc)

    # ------------------------------
    # ROC AUC (Multiclass)
    # ------------------------------

    from sklearn.preprocessing import label_binarize

    y_test_bin = label_binarize(y_test, classes=np.unique(y_test))

    auc = roc_auc_score(y_test_bin, y_prob, multi_class="ovr")

    print("ROC-AUC:", auc)


# Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features='sqrt'
)

# ✅ Train on FINAL enhanced features (IMPORTANT)
rf_model.fit(X_train_scaled, y_train_encoded)

# ✅ Predictions
y_pred_rf = rf_model.predict(X_test_scaled)

# Accuracy
print("Random Forest Accuracy:",
      accuracy_score(y_test_encoded, y_pred_rf))

# Classification Report
print(classification_report(y_test_encoded, y_pred_rf))

# Probabilities (for ROC-AUC)
y_prob_rf = rf_model.predict_proba(X_test_scaled)

# Full evaluation
evaluate_model_full(
    "Random Forest",
    y_test_encoded,
    y_pred_rf,
    y_prob_rf
)


# XGBoost

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    eval_metric="mlogloss"
)

# ✅ Train on FINAL enhanced features
xgb.fit(X_train_scaled, y_train_encoded)

# ✅ Predictions
y_pred_xgb = xgb.predict(X_test_scaled)

# Accuracy
print("XGBoost Accuracy:",
      accuracy_score(y_test_encoded, y_pred_xgb))

# Classification Report
print(classification_report(y_test_encoded, y_pred_xgb))

# Probabilities (for ROC-AUC)
y_prob_xgb = xgb.predict_proba(X_test_scaled)

# Full evaluation
evaluate_model_full(
    "XGBoost",
    y_test_encoded,
    y_pred_xgb,
    y_prob_xgb
)


